# When Foundation Models Fall Short: From Off-the-Shelf AI to Problem-Specific Adaptation

<table width="100%" bgcolor="#4A148C"><tr><td>
<font color="white" size="6"><b>When Foundation Models Fall Short</b></font><br>
<font color="white" size="4">From Off-the-Shelf AI to Problem-Specific Adaptation</font><br>
<font color="white">AMLD 2027 · EPFL, 11 March 2027 &nbsp;·&nbsp; SDS 2027 · Zurich, 1 June 2027</font>
<br><br>
<font color="white"><b>The question:</b> what can modern AI solve out of the box, when do its limitations
start to matter, and what should we do next?</font>
<br><br>
<font color="white"><b>At the end of this notebook you will be able to:</b></font>
<ol>
<li><font color="white">Judge when a vision foundation model is good enough out of the box, and recognise
the signs that it is not.</font></li>
<li><font color="white">Adapt it with cheap, incomplete supervision (scribbles, missing labels,
pseudo-labels, explanations, prompts), and explain what each approach uses and assumes.</font></li>
<li><font color="white">Compare approaches on accuracy, annotation effort, reliability and compute, and
decide what to do next on a new problem, including your own images.</font></li>
</ol>
<font color="white">Levels: 🟢 beginner · 🟡 intermediate (the default path) · 🔴 advanced, optional.
&nbsp; ✍️ = you write a few lines &nbsp; 🖍️ = you draw &nbsp; 👥 = compare with your neighbour.</font>
<br><br>
<font color="white"><b>Run <code>setup_check.ipynb</code> first.</b> It was shared with you the week
before: it puts the models, images and precomputed files in your Google Drive, so nothing is downloaded
in the room. If you have not run it, do it now (about 5 minutes), then come back here.</font>
</td></tr></table>

**Objective:** load the cached models and data; see the question and outcomes.

**What you need:** basic ML (training, validation, objectives, evaluation), some Python, this notebook open
in Google Colab with **Runtime → Change runtime type → T4 GPU**. No computer-vision background is needed.
Every step also runs on a CPU, just slower, and each training cell offers a precomputed checkpoint.

**The plan.** Block 1 (sections 1–3): a foundation model segments pets almost perfectly, then meets a
specialised task and falls short. Block 2 (sections 4–10): the ways out, from scribbles to prompting the
model directly, all compared on the same hidden ground truth, then on your own images.

<table width="100%" bgcolor="#FFF8E1"><tr><td><font color="#5D4037">🧾 <b>Recipe and sources.</b>
The feature extractor is <b>DINOv3</b> ViT-S/16 (Meta, 2025, via <code>timm</code>) and the promptable
segmenter is <b>SAM 2.1</b> hiera-small (Meta, Apache-2.0). The scribble idea follows the MIT
"DINOv2 + scribbles" recipe: a tiny head on frozen patch features, trained with a loss that ignores
unlabelled pixels. Datasets and licences are listed in section 10 and in <code>data/ATTRIBUTION.md</code>.
</font></td></tr></table>

## 0. Setup & the question

The next cell detects where you are (Colab or not, GPU or not), fixes the random seed, creates the
`RESULTS` dictionary every section writes into, and defines `use()`, the helper that lets you write your
own code without ever blocking the notebook. Run it once.

In [ ]:
# @title Setup (run me first) { display-mode: "form" }
import ast
import inspect
import json
import os
import random
import sys
import time
from pathlib import Path

import numpy as np

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("NB_SMOKE") == "1"  # automated test: tiny data, CPU, seconds. Never set this by hand.
SEED = 42

# Where the cache lives. setup_check.ipynb filled MyDrive/fm-workshop-cache/ in Colab; the smoke run uses the
# small fixtures committed in data/smoke/; a local checkout uses data/cache/ (scripts/prepare_data.py).
if IN_COLAB and not SMOKE:
    CACHE_DIR = Path("/content/drive/MyDrive/fm-workshop-cache")
    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception as err:  # noqa: BLE001 - one plain line, then a plain message below
        print(f"(Drive mount error: {type(err).__name__}: {err})")
    if not CACHE_DIR.is_dir():
        print(
            "⚠️ Google Drive is not mounted or the cache folder is missing. Run this cell again and click "
            "'Allow' in the pop-up. If the folder is still missing, run setup_check.ipynb first (5 minutes)."
        )
    if (CACHE_DIR / "hf").is_dir():
        os.environ["HF_HUB_CACHE"] = str(CACHE_DIR / "hf")  # model weights, before torch/timm are imported
        os.environ["HF_HUB_OFFLINE"] = "1"  # nothing is downloaded in the room (N-030)
    DATA_DIR, PRECOMPUTED = CACHE_DIR / "data", CACHE_DIR / "precomputed"
    SPECIALISED_JSON = CACHE_DIR / "specialised.json"
else:
    CACHE_DIR = Path("data/smoke" if SMOKE else "data/cache")
    DATA_DIR = CACHE_DIR if SMOKE else CACHE_DIR / "workshop"
    PRECOMPUTED = CACHE_DIR / "precomputed"
    SPECIALISED_JSON = Path("data/specialised.json")

if IN_COLAB:
    try:
        import pytorch_grad_cam  # noqa: F401 - the only package Colab lacks (SPEC §5); installed if missing
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q grad-cam")  # noqa: F821 - %pip, inside a cell

import torch

if SMOKE:
    DEVICE = "cpu"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
    print(
        "ℹ️ No GPU in this session, so everything runs on the CPU: fine, but slower. For a GPU, use "
        "Runtime → Change runtime type → T4 GPU, then run this cell again. Every training cell also "
        "offers our precomputed checkpoint."
    )


def seed_everything(seed: int = SEED) -> None:
    """Fix every random source, so re-running a section from its first cell gives the same result (N-013)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()

RESULTS: dict = {"use_fallbacks": {}}  # every section writes its named numbers here (SPEC §9)


def _is_stub(fn) -> bool:
    """True when the function body is still the `...` placeholder (optionally after a docstring)."""
    try:
        tree = ast.parse(inspect.getsource(fn).strip() or "pass")
    except (OSError, TypeError, SyntaxError):
        return False
    node = tree.body[0]
    if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        return False
    def const(n, kind):
        is_const = isinstance(n, ast.Expr) and isinstance(n.value, ast.Constant)
        return is_const and isinstance(n.value.value, kind)

    body = [n for n in node.body if not const(n, str)]  # drop the docstring
    return all(const(n, type(...)) for n in body)


def _same(a, b) -> bool:
    """Compare two results of any common type (numbers, arrays, tensors, dicts, sequences)."""
    if isinstance(a, torch.Tensor) or isinstance(b, torch.Tensor):
        a = a.detach().cpu().numpy() if isinstance(a, torch.Tensor) else a
        b = b.detach().cpu().numpy() if isinstance(b, torch.Tensor) else b
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b, strict=True))
    try:
        return bool(np.allclose(np.asarray(a, dtype=float), np.asarray(b, dtype=float), atol=1e-5))
    except (TypeError, ValueError):
        return a == b


def use(fn, reference, check=None, name: str | None = None):
    """Return the function the notebook continues with: yours if it matches the reference, else the reference.

    `check(f)` runs `f` on a small play example and returns what to compare (any numbers, arrays or dicts).
    Nothing here is mandatory: an untouched stub (`...` body) silently selects the reference (N-029).
    """
    name = name or fn.__name__
    if _is_stub(fn):
        RESULTS["use_fallbacks"][name] = True
        print(f"ℹ️ {name}: using the reference solution. Fill the stub above to use your own version.")
        return reference
    if check is None:
        RESULTS["use_fallbacks"][name] = False
        print(f"✅ {name}: using your version (no automatic check for this one).")
        return fn
    try:
        yours, theirs = check(fn), check(reference)
    except Exception as err:  # noqa: BLE001 - the participant's code may fail in any way; keep going
        RESULTS["use_fallbacks"][name] = True
        print(f"❌ {name}: your version raised {type(err).__name__}: {err}. Continuing with the reference.")
        return reference
    if _same(yours, theirs):
        RESULTS["use_fallbacks"][name] = False
        print(f"✅ {name}: matches the reference. Continuing with your version.")
        return fn
    RESULTS["use_fallbacks"][name] = True
    print(f"❌ {name}: {yours} vs reference {theirs}. Continuing with the reference.")
    return reference


if not SPECIALISED_JSON.is_file():
    raise SystemExit(f"{SPECIALISED_JSON} is missing. Run setup_check.ipynb first, then run this cell again.")
SPECIALISED: dict = json.loads(SPECIALISED_JSON.read_text())  # the specialised task; never hard-coded
SPECIALISED["dir"] = DATA_DIR / "specialised"
PETS: dict = {"key": "pets", "name": "Oxford-IIIT Pet", "task": "pet vs background"}
PETS |= {"licence": "CC BY-SA 4.0", "dir": DATA_DIR / "pets"}

print(f"Colab: {IN_COLAB} · smoke mode: {SMOKE} · device: {DEVICE} · seed: {SEED}")
print(f"cache: {CACHE_DIR} · specialised task: {SPECIALISED['name']} ({SPECIALISED['task']})")

<table width="100%" bgcolor="#EDE7F6"><tr><td><font color="#311B92">📘 <b>Theory: how <code>use()</code>
works.</b> Several cells later ask you to fill a function whose body is <code>...</code>. The line after
it, <code>fn = use(my_fn, reference=_ref_fn, check=...)</code>, runs both versions on a small play
example and prints ✅ <i>matches</i> or ❌ <i>yours vs reference</i>. Either way the notebook continues:
with your version if it matches, with ours if the stub is untouched or wrong. Writing the code yourself is
where the learning happens; skipping it never breaks anything.</font></td></tr></table>

### The theme

This cell sets the colours used in every figure: cool violet means "it works", warm orange means "it
fails, we adapt", green is the hidden ground truth and teal marks scribbles. A colour means the same
thing in every plot, so you can read a figure without its legend.

In [ ]:
# @title Theme and image helpers { display-mode: "form" }
import matplotlib as mpl
import matplotlib.pyplot as plt
from PIL import Image

THEME = {
    "works": "#6A1B9A",  # violet: the foundation model does the job
    "fails": "#E65100",  # orange: it falls short / we adapt
    "truth": "#2E7D32",  # green: hidden ground truth, evaluation only
    "scribble": "#00838F",  # teal: strokes a person drew
    "background": "#9E9E9E",  # grey overlay
    "banner": "#4A148C",
}
mpl.rcParams.update(
    {
        "figure.dpi": 100,
        "figure.facecolor": "white",
        "axes.titlesize": 10,
        "axes.titlecolor": "#311B92",
        "axes.prop_cycle": mpl.cycler(color=[THEME[k] for k in ("works", "fails", "truth", "scribble")]),
    }
)


def hex_rgb(colour: str) -> np.ndarray:
    """'#RRGGBB' -> float array in [0, 1]."""
    return np.array([int(colour[i : i + 2], 16) for i in (1, 3, 5)]) / 255


def overlay(image: np.ndarray, mask: np.ndarray, colour: str = THEME["truth"], alpha: float = 0.55):
    """Blend a binary mask in one theme colour over an RGB uint8 image; returns an RGB uint8 image."""
    out = image.astype(float) / 255
    out[mask] = (1 - alpha) * out[mask] + alpha * hex_rgb(colour)
    return (out * 255).astype(np.uint8)


def read_png(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert("L"))


def load_item(dataset: dict, item: dict) -> dict:
    """Read one image with its full mask (evaluation only!), scribbles, visible labels and FOV."""
    root = dataset["dir"]
    out = {
        "name": item["name"],
        "split": item["split"],
        "image": np.array(Image.open(root / item["image"]).convert("RGB")),
        "mask": read_png(root / item["mask"]) > 127,
        "scribbles": read_png(root / item["scribbles"]),
        "visible": read_png(root / item["visible"]) > 127,
        "fov": read_png(root / item["fov"]) > 127 if "fov" in item else None,
    }
    out["hidden"] = out["mask"] & ~out["visible"]
    return out


def load_index(dataset: dict) -> dict:
    """Attach the dataset's index.json (facts + one entry per image) and return it."""
    index_file = dataset["dir"] / "index.json"
    if not index_file.is_file():
        raise SystemExit(f"{index_file} is missing. Run setup_check.ipynb first, then run this cell again.")
    dataset["index"] = json.loads(index_file.read_text())
    dataset["items"] = dataset["index"]["items"]
    dataset["fit"] = [it for it in dataset["items"] if it["split"] == "fit"]
    dataset["held"] = [it for it in dataset["items"] if it["split"] == "held"]
    return dataset["index"]

### The data

This cell reads the two datasets from the cache, the simple task (pets) and the specialised one, and
shows one image of each with its ground truth in green. The ground truth is for evaluation only: from
section 3 on, training sees only incomplete labels.

In [ ]:
for ds in (PETS, SPECIALISED):
    idx = load_index(ds)
    print(
        f"{ds['name']}: {len(ds['items'])} images ({len(ds['fit'])} fit, {len(ds['held'])} held-out) · "
        f"{idx['task']} · licence {idx['licence']}"
    )
PRECOMPUTED_HEADS = json.loads((PRECOMPUTED / "heads" / "index.json").read_text())["heads"]
print(f"precomputed: {len(PRECOMPUTED_HEADS)} head checkpoints in {PRECOMPUTED}")

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for k, ds in enumerate((PETS, SPECIALISED)):
    it = load_item(ds, ds["fit"][0])
    axes[2 * k].imshow(it["image"])
    axes[2 * k].set_title(f"{ds['name']}: {it['name']}")
    axes[2 * k + 1].imshow(overlay(it["image"], it["mask"]))
    axes[2 * k + 1].set_title(f"ground truth: {ds['index']['task']}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your prediction.</b> Look at the
two images above. Which task do you expect the foundation model to solve out of the box, and which one
will give it trouble? Write one reason, then keep it: section 2 will tell you whether you were right.
</font></td></tr></table>

<table width="100%" bgcolor="#F3E5F5"><tr><td><font color="#4A148C">❓ <b>Question.</b> Answer in your
head or in a note; a worked answer is folded below.</font></td></tr></table>

**Q1.** 🟢 What is different about the specialised task that could matter for a model trained on everyday
photos? Think about what the objects look like (size, shape, contrast), not about the domain name.

<details><summary>One possible answer</summary>

Everyday photos show compact objects with clear boundaries, exactly what pets are. The specialised task
has structures that are thin, branching and low-contrast, and they cover a small fraction of the image.
A model that learnt "objects" from the web has seen few of those, and its patch features (16 px squares)
are coarser than the structures themselves. Whether that matters is what the next two sections measure.

</details>

### The models

This cell loads the two foundation models from the cache and prints where each one comes from and under
which licence, then wraps them in two small helpers: `dino.features(image, long_side)` gives patch
features, `sam.probability(image, points, labels)` gives a foreground map from point prompts.

In [ ]:
# @title Model loaders { display-mode: "form" }
import timm
import torch.nn.functional as F

DINO_ID = "vit_small_patch16_dinov3.lvd1689m"
DINO_SOURCE = "timm/vit_small_patch16_dinov3.lvd1689m"
DINO_LICENCE = "DINOv3 License (Meta), see the notice below"
SAM_ID = "facebook/sam2.1-hiera-small"  # SPEC §5: SAM 2.1 hiera-small, Apache-2.0, no account needed
SAM_LICENCE = "Apache-2.0"


class Dino:
    """Frozen DINOv3 ViT-S/16: an image in, a grid of 384-dim patch features out."""

    def __init__(self, device: str = DEVICE):
        t = time.monotonic()
        self.device = device
        model = timm.create_model(DINO_ID, pretrained=True, num_classes=0, dynamic_img_size=True)
        self.model = model.eval().to(device)
        cfg = timm.data.resolve_data_config({}, model=self.model)
        self.mean = torch.tensor(cfg["mean"]).view(1, 3, 1, 1).to(device)
        self.std = torch.tensor(cfg["std"]).view(1, 3, 1, 1).to(device)
        self.n_prefix = self.model.num_prefix_tokens
        self.dim = self.model.num_features
        dt = time.monotonic() - t
        print(f"DINOv3 ViT-S/16 · source {DINO_SOURCE} · licence {DINO_LICENCE} · {dt:.0f} s")

    @torch.no_grad()
    def features(self, image: np.ndarray, long_side: int) -> torch.Tensor:
        """Patch features [D, h, w] for the image resized so its long side is `long_side` (multiple of 16)."""
        x = torch.from_numpy(np.ascontiguousarray(image)).permute(2, 0, 1).float().unsqueeze(0) / 255
        h, w = image.shape[:2]
        s = long_side / max(h, w)
        H, W = max(16, round(h * s / 16) * 16), max(16, round(w * s / 16) * 16)
        x = F.interpolate(x.to(self.device), size=(H, W), mode="bilinear", align_corners=False)
        x = (x - self.mean) / self.std
        f = self.model.forward_features(x)[0, self.n_prefix :]
        return f.reshape(H // 16, W // 16, -1).permute(2, 0, 1).contiguous().float()


class Sam:
    """SAM 2.1 (promptable segmentation): an image plus points or a box in, a foreground probability out."""

    def __init__(self, device: str = DEVICE):
        from transformers import Sam2Model, Sam2Processor

        t = time.monotonic()
        self.device = device
        self.proc = Sam2Processor.from_pretrained(SAM_ID)
        self.model = Sam2Model.from_pretrained(SAM_ID).eval().to(device)
        self._cache: tuple[np.ndarray, torch.Tensor] | None = None  # (the last image, its embeddings)
        print(f"SAM 2.1 hiera-small · source {SAM_ID} · licence {SAM_LICENCE} · {time.monotonic() - t:.0f} s")

    @torch.no_grad()
    def probability(
        self,
        image: np.ndarray,
        points: list[list[float]] | None = None,
        labels: list[int] | None = None,
        box: list[float] | None = None,
    ) -> np.ndarray:
        """Foreground probability (H x W float) for points (x, y; label 1 = fg, 0 = bg) and/or a box."""
        kw = {}
        if points:
            kw["input_points"], kw["input_labels"] = [[points]], [[labels]]
        if box is not None:
            kw["input_boxes"] = [[box]]
        inputs = self.proc(images=image, return_tensors="pt", **kw)
        if self._cache is None or self._cache[0] is not image:
            self._cache = (image, self.model.get_image_embeddings(inputs["pixel_values"].to(self.device)))
        out = self.model(
            image_embeddings=self._cache[1],
            input_points=inputs["input_points"].to(self.device) if points else None,
            input_labels=inputs["input_labels"].to(self.device) if points else None,
            input_boxes=inputs["input_boxes"].to(self.device) if box is not None else None,
            multimask_output=False,
        )
        masks = out.pred_masks.cpu()
        logits = self.proc.post_process_masks(masks, inputs["original_sizes"], binarize=False)[0]
        return torch.sigmoid(logits[0, 0]).numpy()

In [ ]:
dino = Dino()
sam = Sam()
RESULTS["models"] = {"dino": DINO_SOURCE, "sam": SAM_ID, "device": DEVICE}

<table width="100%" bgcolor="#FFF8E1"><tr><td><font color="#5D4037">🧾 <b>DINOv3 licence notice.</b>
DINOv3 is licensed under the <a href="https://huggingface.co/timm/vit_small_patch16_dinov3.lvd1689m">DINOv3
License</a>, Copyright © Meta Platforms, Inc. All Rights Reserved. It allows research and commercial use
under Meta's conditions and acceptable-use policy; keep this notice if you redistribute the weights or
anything derived from them. The <code>timm</code> re-host used here is open to everyone, no account needed.
SAM 2.1 is Apache-2.0. Both are re-checked about four weeks before each event, since Meta can change the
terms.</font></td></tr></table>

<table width="100%" bgcolor="#FDECEA"><tr><td><font color="#8B1A10">⚠️ <b>Pitfall: the first slow step.</b>
The first time a model sees a new image size, PyTorch compiles kernels for it: the first call of the next
section is slower than the ones after. On the CPU the first SAM call on a 1024 px image takes ~20 s.
</font></td></tr></table>

## 1. Out of the box: pets

**Objective:** see a foundation model segment pets almost perfectly, zero-shot.

Three tools, no training data of ours: **SAM** segments whatever you point at, **DINOv3** turns an image
into a grid of features that already "know" what belongs together, and a **linear head** on those
features learns a new class from a handful of masks in seconds. Here they meet the easy task. Keep the
numbers you get: section 2 runs exactly the same three cells on the specialised task.

<table width="100%" bgcolor="#EDE7F6"><tr><td><font color="#311B92">📘 <b>Theory: two kinds of
foundation model.</b> <b>SAM</b> (Segment Anything) is <i>promptable</i>: you give it an image and a
hint (a click, a box), and it returns the mask of the object the hint is on. Nothing is trained. It
learnt "what an object is" from a billion masks of everyday photos. <b>DINOv3</b> is a <i>feature
extractor</i>: it cuts the image into 16 × 16 px patches and returns a 384-number description of each
patch, learnt without any labels. Two patches with similar descriptions look alike to the model. On
their own the features segment nothing; a tiny classifier on top of them does, and that is the
<b>linear head</b> we train below.</font></td></tr></table>

This helper cell defines the two pieces this section measures with: how to turn a person's hint into a
SAM prompt, and `show_masks()`, the figure used for every comparison in the notebook (prediction in
violet, ground truth in green, prompts in teal).

In [ ]:
# @title Prompts and figure helpers { display-mode: "form" }
from safetensors import safe_open
from safetensors.torch import load_file

_FEATURE_FILES: dict = {}


def click_point(item: dict) -> list[float]:
    """One (x, y) where a person would click: a pixel of the first foreground scribble stroke."""
    ys, xs = np.nonzero(item["scribbles"] == 1)
    rng = np.random.default_rng([SEED, len(ys)])
    k = int(rng.integers(len(ys)))
    return [float(xs[k]), float(ys[k])]


def box_prompt(item: dict) -> list[float]:
    """The box a person would draw around the object: the full mask's bounding box, as x0, y0, x1, y1."""
    ys, xs = np.nonzero(item["mask"])
    return [float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())]


def show_masks(panels: list[tuple[str, np.ndarray]], prompts: dict | None = None, width: float = 3.2):
    """One row of RGB panels (title, image). `prompts` = {"point": [x, y], "box": [x0, y0, x1, y1]}."""
    fig, axes = plt.subplots(1, len(panels), figsize=(width * len(panels), width))
    axes = np.atleast_1d(axes)
    for ax, (title, img) in zip(axes, panels, strict=True):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    if prompts:
        for ax in axes:
            if prompts.get("point") is not None:
                x, y = prompts["point"]
                ax.plot(x, y, marker="*", markersize=14, color=THEME["scribble"], markeredgecolor="white")
            if prompts.get("box") is not None:
                x0, y0, x1, y1 = prompts["box"]
                rect = mpl.patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, lw=2)
                rect.set_edgecolor(THEME["scribble"])
                ax.add_patch(rect)
    plt.tight_layout()
    plt.show()


def features_for(dataset: dict, item: dict, long_side: int) -> torch.Tensor:
    """DINOv3 patch features [384, h, w]: from the precomputed file when it exists, else computed now."""
    prefix = "pets" if dataset is PETS else "specialised"
    path = PRECOMPUTED / "features" / f"{prefix}_{long_side}.safetensors"
    if path.is_file():
        if path not in _FEATURE_FILES:
            _FEATURE_FILES[path] = safe_open(path, framework="pt")
        store = _FEATURE_FILES[path]
        if item["name"] in store.keys():
            return store.get_tensor(item["name"]).to(DEVICE).float()
    return dino.features(item["image"], long_side)


def pca_rgb(feat: torch.Tensor, shape: tuple[int, int]) -> np.ndarray:
    """Project the 384-dim patch features onto their 3 main directions and show them as red, green, blue."""
    d, h, w = feat.shape
    x = feat.reshape(d, h * w).T.cpu().numpy().astype(np.float64)
    x = x - x.mean(0)
    _, _, vt = np.linalg.svd(x, full_matrices=False)
    rgb = x @ vt[:3].T
    rgb = (rgb - rgb.min(0)) / (rgb.max(0) - rgb.min(0) + 1e-9)
    img = torch.from_numpy(rgb.reshape(h, w, 3)).permute(2, 0, 1)[None].float()
    img = F.interpolate(img, size=shape, mode="bilinear", align_corners=False)[0].permute(1, 2, 0)
    return (img.numpy() * 255).astype(np.uint8)

This cell loads the held-out pets, the evaluation set every approach in this notebook is scored on, and
fixes the resolution DINOv3 works at. On a CPU, 12 of them keep each cell under a minute.

In [ ]:
PETS["feat_side"] = 256 if SMOKE else PETS["index"]["feature_long_side"]
PETS["eval"] = PETS["held"] if (SMOKE or DEVICE == "cuda") else PETS["held"][:12]
PETS_EVAL = [load_item(PETS, it) for it in PETS["eval"]]
print(f"evaluation set: {len(PETS_EVAL)} held-out pets · DINOv3 features at {PETS['feat_side']} px long side")

### 1.1 SAM: one click, one box

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your prediction.</b> The next
cell gives SAM a single click on a pet, then a box around it. Before you run it: will one click be
enough for a clean mask of the whole animal, and what could go wrong (fur vs floor, the tail, a second
animal)? Say it to your neighbour or write it down.</font></td></tr></table>

<table width="100%" bgcolor="#F3E5F5"><tr><td><font color="#4A148C">❓ <b>Question.</b> Run the next
cell and look at the three panels.</font></td></tr></table>

**Q2.** 🟡 Where do the one-click mask and the box mask differ from the ground truth, if at all? Is the
difference at the object's boundary, or is a whole part missing or added? Try the same cell on another
pet by changing the index.

```python
demo = PETS_EVAL[0]   # try 1, 2, ...
```

This cell runs SAM twice on one held-out pet, once with a single click and once with a box, and shows
both masks next to the ground truth, so you can see what a promptable model gives for free.

In [ ]:
# Q2 — SAM with one click and with one box on a held-out pet
demo = PETS_EVAL[0]  # try 1, 2, ...
point, box = click_point(demo), box_prompt(demo)
t0 = time.monotonic()
sam_point = sam.probability(demo["image"], points=[point], labels=[1]) >= 0.5
sam_box = sam.probability(demo["image"], box=box) >= 0.5
print(f"{demo['name']}: two SAM calls in {time.monotonic() - t0:.1f} s on {DEVICE}")
show_masks(
    [
        ("SAM, one click", overlay(demo["image"], sam_point, THEME["works"])),
        ("SAM, one box", overlay(demo["image"], sam_box, THEME["works"])),
        ("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"])),
    ],
    prompts={"point": point, "box": box},
)

### 1.2 One number: IoU

Looking at masks is the honest way to judge a segmentation, and a single number lets us compare
approaches on a hundred images. We use one metric for the whole workshop: **IoU** (intersection over
union), the overlap between the prediction and the ground truth divided by the area covered by either.
1.0 is a perfect match, 0.0 means no overlap. Section 2 adds one extra metric for thin structures.

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your turn (optional, 2 min).</b>
Fill <code>my_iou</code>: two boolean arrays in, one float out. Pixels where both are True are the
intersection; pixels where at least one is True are the union. If the union is empty, return 1.0 (two
empty masks agree). The cell below checks your version on a 4 × 4 play example and continues with
ours if you skip it.</font></td></tr></table>

In [ ]:
# ✍️ YOUR CODE
def my_iou(pred: np.ndarray, truth: np.ndarray) -> float:
    """Intersection over union of two boolean masks of the same shape; 1.0 when both are empty."""
    ...


def _ref_iou(pred: np.ndarray, truth: np.ndarray) -> float:
    """Reference: |pred ∧ truth| / |pred ∨ truth|."""
    inter, union = float((pred & truth).sum()), float((pred | truth).sum())
    return inter / union if union else 1.0


_play_pred = np.zeros((4, 4), bool)
_play_pred[1:3, 1:4] = True  # 6 pixels
_play_truth = np.zeros((4, 4), bool)
_play_truth[0:3, 1:3] = True  # 6 pixels, 4 shared -> IoU = 4 / 8
iou = use(my_iou, reference=_ref_iou, check=lambda f: round(float(f(_play_pred, _play_truth)), 4), name="iou")

This cell scores SAM's one-click and one-box masks on every evaluation pet with the IoU you just
defined, and keeps the mean as this task's zero-shot number: section 2 compares it with the
specialised task.

In [ ]:
t0 = time.monotonic()
zero_shot = {"point": [], "box": []}
for it in PETS_EVAL:
    pt = click_point(it)
    zero_shot["point"].append(iou(sam.probability(it["image"], points=[pt], labels=[1]) >= 0.5, it["mask"]))
    zero_shot["box"].append(iou(sam.probability(it["image"], box=box_prompt(it)) >= 0.5, it["mask"]))
zero_shot = {k: float(np.mean(v)) for k, v in zero_shot.items()}
RESULTS["pets_zero_shot_iou"] = zero_shot["box"]  # the headline: SAM's best single prompt, one box
RESULTS["pets_zero_shot_point_iou"] = zero_shot["point"]
RESULTS["pets_zero_shot_box_iou"] = zero_shot["box"]
RESULTS["pets_zero_shot_seconds"] = round(time.monotonic() - t0, 1)
print(
    f"SAM zero-shot on {len(PETS_EVAL)} held-out pets · mean IoU one click {zero_shot['point']:.3f} · "
    f"one box {zero_shot['box']:.3f} · {RESULTS['pets_zero_shot_seconds']} s"
)

### 1.3 What DINOv3 sees

This cell takes the 384-number description DINOv3 gives each 16 × 16 px patch, keeps the three
directions that vary most (a PCA) and shows them as red, green and blue, so you can see what the model
groups together before any training.

In [ ]:
fig_items = [load_item(PETS, it) for it in PETS["fit"][:3]]
panels = []
for it in fig_items:
    feat = features_for(PETS, it, PETS["feat_side"])
    title = f"DINOv3 PCA · {feat.shape[2]}×{feat.shape[1]} patches"
    panels += [(it["name"], it["image"]), (title, pca_rgb(feat, it["image"].shape[:2]))]
show_masks(panels, width=2.6)

**Q3.** 🟡 In the PCA images, which pixels share a colour: the whole animal, parts of it (head, fur,
legs), or the floor and the background too? Does the colour follow the object's outline, and how fine is
the grid behind it?

<details><summary>What to look for</summary>

The colours are not classes: they are the three directions along which the features vary most across
this image, so the same colour means "the model describes these patches alike". Typically the animal
separates from the background in one direction and its parts (face vs body) in another, and the blocky
edges show the 16 px grid: the features know <i>what</i> is where, at patch resolution, not <i>exactly
where</i> the boundary is. Whether that is enough depends on the size of what you want to segment.

</details>

### 1.4 A linear head on five pets

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your prediction.</b> The next
cells train a classifier with 385 parameters on the patch features of just <b>five</b> pets with full
masks, then score it on the held-out ones. Will its IoU land above, near or below SAM's one-click number
from 1.2? Write down a number before you run.</font></td></tr></table>

This helper cell defines the head used for every training step in the notebook: a 1 × 1 convolution on
standardised patch features, trained with a balanced cross-entropy that ignores pixels labelled −1, plus
`load_head()` for our precomputed checkpoints.

In [ ]:
# @title Linear head: train, predict, load a checkpoint { display-mode: "form" }
def train_head(
    feats: list[torch.Tensor], labels: list[np.ndarray], steps: int, lr: float = 0.05, balanced: bool = True
) -> tuple[dict, float]:
    """Linear head (384 -> 1) on standardised patch features. Labels per pixel: 1 fg, 0 bg, -1 ignored.

    Returns the head's tensors (weight, bias, feat_mean, feat_std) and the training seconds.
    """
    torch.manual_seed(SEED)
    allf = torch.cat([f.flatten(1) for f in feats], 1)
    mu, sd = allf.mean(1), allf.std(1) + 1e-6
    feats = [(f - mu.view(-1, 1, 1)) / sd.view(-1, 1, 1) for f in feats]
    labs = [torch.from_numpy(np.asarray(lab, dtype=np.int8)).to(DEVICE) for lab in labels]
    n_pos = sum(int((lab == 1).sum()) for lab in labs)
    n_neg = sum(int((lab == 0).sum()) for lab in labs)
    if balanced:
        w_pos, w_neg = 0.5 / max(n_pos, 1), 0.5 / max(n_neg, 1)
    else:
        w_pos = w_neg = 1.0 / max(n_pos + n_neg, 1)
    head = torch.nn.Conv2d(feats[0].shape[0], 1, 1).to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=lr)
    t0 = time.monotonic()
    for _ in range(steps):
        opt.zero_grad()
        loss = 0.0
        for f, lab in zip(feats, labs, strict=True):
            logit = F.interpolate(head(f[None]), size=lab.shape, mode="bilinear", align_corners=False)[0, 0]
            w = torch.where(lab == 1, w_pos, torch.where(lab == 0, w_neg, 0.0))
            bce = F.binary_cross_entropy_with_logits(logit, (lab == 1).float(), reduction="none")
            loss = loss + (bce * w).sum()
        loss.backward()
        opt.step()
    tensors = {
        "weight": head.weight.detach().cpu(),
        "bias": head.bias.detach().cpu(),
        "feat_mean": mu.cpu(),
        "feat_std": sd.cpu(),
    }
    return tensors, time.monotonic() - t0


@torch.no_grad()
def predict_head(tensors: dict, feat: torch.Tensor, shape: tuple[int, int]) -> np.ndarray:
    """Apply a head to one feature grid; returns the boolean foreground mask at `shape` (logit > 0)."""
    dev = feat.device
    f = (feat - tensors["feat_mean"].to(dev).view(-1, 1, 1)) / tensors["feat_std"].to(dev).view(-1, 1, 1)
    logit = F.conv2d(f[None], tensors["weight"].to(dev), tensors["bias"].to(dev))
    logit = F.interpolate(logit, size=shape, mode="bilinear", align_corners=False)[0, 0]
    return (logit > 0).cpu().numpy()


def load_head(dataset: dict, regime: str, long_side: int | None = None) -> tuple[dict, dict]:
    """Our precomputed checkpoint for (dataset, regime): its tensors and its index entry."""
    long_side = long_side or dataset["feat_side"]
    key = "pets" if dataset is PETS else SPECIALISED["key"]
    wanted = (key, regime, long_side)
    heads = PRECOMPUTED_HEADS
    entry = next((h for h in heads if (h["dataset"], h["regime"], h["long_side"]) == wanted), None)
    if entry is None:
        raise SystemExit(f"no precomputed head for {wanted}. Run setup_check.ipynb first.")
    return load_file(PRECOMPUTED / entry["file"]), entry

This cell trains the head on the full masks of five fit pets (sections 1 and 2 are the only places
training sees full masks) and prints the seconds it took. Not done in a minute, or on a CPU and in a
hurry? Set `TRAIN_YOURSELF = False` and run it again: it loads our checkpoint instead.

In [ ]:
TRAIN_YOURSELF = True  # False -> load our precomputed head "full5" instead of training
N_TRAIN_PETS, HEAD_STEPS = 5, 30 if SMOKE else 200
if not isinstance(TRAIN_YOURSELF, bool):
    raise SystemExit("TRAIN_YOURSELF must be True or False.")

train_items = [load_item(PETS, it) for it in PETS["fit"][:N_TRAIN_PETS]]
train_feats = [features_for(PETS, it, PETS["feat_side"]) for it in train_items]
if TRAIN_YOURSELF:
    train_labels = [it["mask"].astype(np.int8) for it in train_items]  # full masks: allowed in section 1 only
    head_full5, head_sec = train_head(train_feats, train_labels, HEAD_STEPS)
    head_src = f"trained here on {len(train_items)} pets, {HEAD_STEPS} steps"
else:
    head_full5, entry = load_head(PETS, "full5")
    head_sec = entry["train_sec"]
    head_src = f"our checkpoint {entry['file']} ({len(entry['train_names'])} pets)"
print(f"linear head · {head_src} · {head_sec:.1f} s on {DEVICE}")
if head_sec > 60:
    print("ℹ️ Slow session? Set TRAIN_YOURSELF = False above to load our checkpoint next time.")

This cell scores the head on the same evaluation pets as SAM, keeps the mean IoU, and shows three held-out
pets with the head's mask next to the ground truth.

In [ ]:
t0 = time.monotonic()
head_preds = [
    predict_head(head_full5, features_for(PETS, it, PETS["feat_side"]), it["mask"].shape) for it in PETS_EVAL
]
head_ious = [iou(pred, it["mask"]) for pred, it in zip(head_preds, PETS_EVAL, strict=True)]
RESULTS["pets_head_iou"] = float(np.mean(head_ious))
RESULTS["pets_head"] = {
    "iou": RESULTS["pets_head_iou"],
    "n_train": len(train_items),
    "train_sec": round(float(head_sec), 2),
    "infer_sec": round(time.monotonic() - t0, 1),
}
print(
    f"linear head on {len(PETS_EVAL)} held-out pets · mean IoU {RESULTS['pets_head_iou']:.3f} "
    f"(SAM one click: {RESULTS['pets_zero_shot_point_iou']:.3f}, "
    f"one box: {RESULTS['pets_zero_shot_box_iou']:.3f})"
)
panels = []
for it, pred, v in list(zip(PETS_EVAL, head_preds, head_ious, strict=True))[:3]:
    panels += [
        (f"head · IoU {v:.2f}", overlay(it["image"], pred, THEME["works"])),
        ("ground truth", overlay(it["image"], it["mask"], THEME["truth"])),
    ]
show_masks(panels, width=2.6)

<table width="100%" bgcolor="#E8EAF6"><tr><td><font color="#1A237E">🔀 <b>Across approaches.</b> Two ways
to the same mask, scored on the same hidden ground truth: SAM needs <i>one hint per image</i> and no
training; the head needs <i>five full masks once</i> and then runs on every image with no hint. Keep both
numbers: the comparison table in section 8 starts with them, and section 2 reruns both on the
specialised task.</font></td></tr></table>

**Q4.** 🟡 Compare the two numbers you got with the prediction you wrote down. Which approach would you
pick for 10 000 pet photos, and which for 10 photos? What does each one ask of a person?

<details><summary>One possible answer</summary>

For 10 photos, a click each is nothing, and SAM's masks come with sharp boundaries. For 10 000, a person
cannot click on each one, while the head, once trained on a few masks, runs with no hint at all. Its
boundaries follow the 16 px patch grid, so it loses a little IoU at the edges, which matters more as the
objects get smaller. Hardware or seed may change the numbers; the trade-off stays.

</details>

<table width="100%" bgcolor="#F3E5F5"><tr><td><font color="#4A148C">👥 <b>Optional, 3 min.</b> Compare
the demo pet with your neighbour's: did one click fail on either of yours? If it did, was the click on
an ambiguous spot (fur against a similar floor, a collar, a second animal)?</font></td></tr></table>

<table width="100%" bgcolor="#FDECEA"><tr><td><font color="#8B1A10">⚠️ <b>Pitfall: a clean mask is not
a correct one.</b> SAM always returns a confident-looking mask, whatever you click on. On pets that is
usually the right object. Keep that habit of looking at the mask next to the ground truth: in the next
section it is the only thing that shows the model has quietly segmented the wrong thing.
</font></td></tr></table>

## 2. Where it falls short

**Objective:** run the same tools on the specialised task and probe SAM with escalating prompts.

Same three tools, same cells, new task. Section 1 gave you three numbers on pets: SAM with one click,
SAM with one box, and a linear head on five full masks. Here they meet the specialised images, where the
objects are thin, branching and low-contrast. Then we push SAM with more and more prompts, and give
DINOv3 twice the resolution, to see which of the two limits the result.

<table width="100%" bgcolor="#EDE7F6"><tr><td><font color="#311B92">📘 <b>Theory: why thin structures
are hard.</b> DINOv3 describes the image in 16 × 16 px patches. A vessel 5 px wide covers a third of a
patch, so its description is mostly "background with a dark line in it", and the head can only say
"vessel" or "not" for the whole square. SAM sees finer detail, but it was trained on a billion masks of
<i>objects</i>: compact things with a clear inside and outside. A vessel tree has
no inside, and one click on it is ambiguous (this vessel? the whole tree? the eye?).
</font></td></tr></table>

This helper cell defines what this section measures with: the evaluation region (the fundus, not the
black border around it), skeletons for the thin-structure metric, clicks spread along a person's
scribbles, and a figure that marks every click.

In [ ]:
# @title Thin-structure helpers: region, skeleton, clicks, figure { display-mode: "form" }
from skimage.morphology import skeletonize


def region(item: dict) -> np.ndarray:
    """Where we score: the field of view if the dataset has one, else the whole image."""
    return item["fov"] if item.get("fov") is not None else np.ones_like(item["mask"])


def skeleton(mask: np.ndarray) -> np.ndarray:
    """The one-pixel-wide centre line of a boolean mask."""
    return skeletonize(np.asarray(mask, bool))


def scribble_points(item: dict, n: int) -> list[list[float]]:
    """n (x, y) clicks along the foreground scribbles: the section-1 click first, then spread out as far as
    possible from each other (farthest-point sampling), the way a person would click along the vessels."""
    ys, xs = np.nonzero(item["scribbles"] == 1)
    pts = np.stack([xs, ys], 1).astype(float)
    chosen = [np.array(click_point(item))]
    dist = np.linalg.norm(pts - chosen[0], axis=1)
    while len(chosen) < min(n, len(pts)):
        k = int(dist.argmax())
        chosen.append(pts[k])
        dist = np.minimum(dist, np.linalg.norm(pts - pts[k], axis=1))
    return [[float(x), float(y)] for x, y in chosen]


def score(pred: np.ndarray, item: dict) -> dict:
    """IoU and clDice of a boolean prediction, both inside the evaluation region."""
    r = region(item)
    return {"iou": iou(pred & r, item["mask"] & r), "cldice": cldice(pred & r, item["mask"] & r)}


def mean_scores(scores: list[dict]) -> dict:
    return {k: float(np.mean([s[k] for s in scores])) for k in scores[0]}


def show_clicks(panels: list[tuple[str, np.ndarray, list | None]], width: float = 2.6):
    """One row of panels (title, RGB image, clicks or None); clicks are drawn as teal stars."""
    fig, axes = plt.subplots(1, len(panels), figsize=(width * len(panels), width))
    for ax, (title, img, pts) in zip(np.atleast_1d(axes), panels, strict=True):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
        for x, y in pts or []:
            ax.plot(x, y, marker="*", markersize=10, color=THEME["scribble"], markeredgecolor="white")
    plt.tight_layout()
    plt.show()

This cell loads the held-out specialised images, the evaluation set for the rest of the notebook, and
prints how much of the fundus the structures cover: keep that number, it comes back as a baseline.

In [ ]:
SPECIALISED["feat_side"] = 256 if SMOKE else SPECIALISED["feature_long_side"]
SPECIALISED["eval"] = SPECIALISED["held"] if (SMOKE or DEVICE == "cuda") else SPECIALISED["held"][:12]
SPEC_EVAL = [load_item(SPECIALISED, it) for it in SPECIALISED["eval"]]
fg_share = float(np.mean([it["mask"][region(it)].mean() for it in SPEC_EVAL]))
print(
    f"evaluation set: {len(SPEC_EVAL)} held-out images · {SPECIALISED['task']} · the foreground covers "
    f"{fg_share:.0%} of the evaluation region · DINOv3 features at {SPECIALISED['feat_side']} px long side"
)

### 2.1 SAM: one click, one box, again

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your prediction.</b> The next
cell is the section-1 cell on a specialised image: one click on a vessel (a pixel of a person's
scribble), then a box around all of them. Before you run it: what will SAM segment from one click on a
thin line? And what is "the object" inside a box that covers the whole image?</font></td></tr></table>

<table width="100%" bgcolor="#F3E5F5"><tr><td><font color="#4A148C">❓ <b>Question.</b> Run the next
cell and compare the panels with the pets in section 1.</font></td></tr></table>

**Q5.** 🟡 What did SAM segment from the click, and from the box: a vessel, part of the tree, a blob, the
whole fundus? Try two other images by changing the index (on a CPU session there are 12).

```python
demo = SPEC_EVAL[0]   # try 1, 2, ...
```

This cell runs SAM twice on one held-out specialised image, once with the click and once with the box,
and shows both masks in orange next to the ground truth.

In [ ]:
# Q5 — SAM with one click and with one box on a held-out specialised image
demo = SPEC_EVAL[0]  # try 1, 2, ...
point, box = click_point(demo), box_prompt(demo)
t0 = time.monotonic()
sam_point = sam.probability(demo["image"], points=[point], labels=[1]) >= 0.5
sam_box = sam.probability(demo["image"], box=box) >= 0.5
print(f"{demo['name']}: two SAM calls in {time.monotonic() - t0:.1f} s on {DEVICE}")
show_masks(
    [
        ("SAM, one click", overlay(demo["image"], sam_point, THEME["fails"])),
        ("SAM, one box", overlay(demo["image"], sam_box, THEME["fails"])),
        ("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"])),
    ],
    prompts={"point": point, "box": box},
)

### 2.2 A second number for thin structures: clDice

<table width="100%" bgcolor="#EDE7F6"><tr><td><font color="#311B92">📘 <b>Theory: clDice.</b> IoU counts
pixels, and on thin structures pixels mislead. Predict every vessel twice as wide as it is and IoU drops
to about 0.5, although nothing was missed; miss all the capillaries and IoU barely moves, because they
hold few pixels. <b>clDice</b> (Shit et al., CVPR 2021) compares <i>centre lines</i> instead. It takes
the skeleton of each mask and asks two questions: which share of the predicted centre line lies inside
the true vessels (<i>topology precision</i>), and which share of the true centre line the prediction
covers (<i>topology sensitivity</i>). clDice is their harmonic mean: 1.0 when the tree is traced
exactly, low when branches are missing or invented. We report it next to IoU for the specialised task.
</font></td></tr></table>

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your turn (optional, 3 min).</b>
Fill <code>my_cldice</code>: two boolean masks in, one float out. <code>skeleton(mask)</code> gives the
centre line. Topology precision = pixels of <code>skeleton(pred)</code> inside <code>truth</code>, divided
by the pixels of <code>skeleton(pred)</code>; topology sensitivity is the same with the roles swapped.
Return 0.0 when either skeleton is empty. The cell checks your version on two play masks, a good trace
and a blob that covers everything, and continues with ours if you skip it.</font></td></tr></table>

In [ ]:
# ✍️ YOUR CODE
def my_cldice(pred: np.ndarray, truth: np.ndarray) -> float:
    """clDice of two boolean masks: 2·tprec·tsens / (tprec + tsens); 0.0 when either skeleton is empty."""
    ...


def _ref_cldice(pred: np.ndarray, truth: np.ndarray) -> float:
    """Reference: harmonic mean of topology precision and topology sensitivity."""
    sk_pred, sk_truth = skeleton(pred), skeleton(truth)
    if not sk_pred.any() or not sk_truth.any():
        return 0.0
    tprec = float((sk_pred & truth).sum() / sk_pred.sum())
    tsens = float((sk_truth & pred).sum() / sk_truth.sum())
    return 2 * tprec * tsens / (tprec + tsens) if tprec + tsens else 0.0


_play_truth = np.zeros((12, 12), bool)
_play_truth[2:4, :] = True  # a horizontal vessel, 2 px wide
_play_truth[:, 9:11] = True  # crossed by a vertical one
_play_trace = np.zeros((12, 12), bool)
_play_trace[1:5, :] = True  # the horizontal vessel, drawn too wide; the vertical one missed
_play_blob = np.ones((12, 12), bool)  # "everything is vessel"
cldice = use(
    my_cldice,
    reference=_ref_cldice,
    check=lambda f: [round(float(f(p, _play_truth)), 4) for p in (_play_trace, _play_blob)],
    name="cldice",
)
print(
    f"play example · IoU / clDice of the wide trace: {iou(_play_trace, _play_truth):.2f} / "
    f"{cldice(_play_trace, _play_truth):.2f} · of the blob: {iou(_play_blob, _play_truth):.2f} / "
    f"{cldice(_play_blob, _play_truth):.2f}"
)

### 2.3 Escalating prompts: 1 → 5 → 20 clicks, and a box

If one click is ambiguous, maybe more clicks tell SAM what we mean. The next cell gives it 1, 5 and 20
clicks spread along the scribbles a person drew, then the box, on every evaluation image. It also scores
the laziest possible answer, a mask that says "everything is foreground": a prediction no better than
that line has learnt nothing about the task.

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your prediction.</b> Write down
what you expect from 1 to 20 clicks: will IoU rise steadily, jump at some point, or stay flat? Will the
box beat 20 clicks? And where will the grey "everything" line sit?</font></td></tr></table>

This cell runs the prompt ladder on every evaluation image (one SAM image encoding per image, then four
cheap prompt decodings) and keeps the mean IoU, clDice and the share of the fundus SAM marks per setting.

In [ ]:
LADDER = {"1 point": 1, "5 points": 5, "20 points": 20, "box": None}  # setting -> number of clicks
t0 = time.monotonic()
ladder_scores = {name: [] for name in LADDER}
ladder_cover = {name: [] for name in LADDER}
ladder_demo = {}  # the demo image's masks and clicks, for the figure below
sam_box_masks = []  # every image's box mask, for the gallery in 2.6
everything = []
for it in SPEC_EVAL:
    r = region(it)
    for name, n in LADDER.items():
        pts = scribble_points(it, n) if n else None
        if n:
            pred = sam.probability(it["image"], points=pts, labels=[1] * len(pts)) >= 0.5
        else:
            pred = sam.probability(it["image"], box=box_prompt(it)) >= 0.5
        ladder_scores[name].append(score(pred, it))
        ladder_cover[name].append(float(pred[r].mean()))
        if it is demo:
            ladder_demo[name] = (pred, pts)
    sam_box_masks.append(pred)  # the last setting is the box
    everything.append(score(r.copy(), it))
ladder_mean = {name: mean_scores(s) for name, s in ladder_scores.items()}
everything_mean = mean_scores(everything)
spec_sec = round(time.monotonic() - t0, 1)

RESULTS["sam_prompt_ladder"] = {name: v["iou"] for name, v in ladder_mean.items()}
RESULTS["sam_prompt_ladder_cldice"] = {name: v["cldice"] for name, v in ladder_mean.items()}
RESULTS["specialised_zero_shot_iou"] = ladder_mean["box"]["iou"]  # same headline as pets: one box
RESULTS["specialised_zero_shot_point_iou"] = ladder_mean["1 point"]["iou"]
RESULTS["specialised_zero_shot_box_iou"] = ladder_mean["box"]["iou"]
RESULTS["specialised_everything_iou"] = everything_mean["iou"]
RESULTS["specialised_zero_shot_seconds"] = spec_sec

print(f"SAM prompt ladder on {len(SPEC_EVAL)} held-out images · {spec_sec} s on {DEVICE}")
print(f"{'setting':<12}{'IoU':>7}{'clDice':>8}{'marked':>8}")
for name, v in ladder_mean.items():
    print(f"{name:<12}{v['iou']:>7.3f}{v['cldice']:>8.3f}{np.mean(ladder_cover[name]):>8.0%}")
print(f"{'everything':<12}{everything_mean['iou']:>7.3f}{everything_mean['cldice']:>8.3f}{1:>8.0%}")

fig, ax = plt.subplots(figsize=(6.4, 3))
names = list(ladder_mean)
ax.bar(names, [ladder_mean[k]["iou"] for k in names], color=THEME["fails"], label="specialised, SAM")
ax.axhline(everything_mean["iou"], color=THEME["background"], ls="--", label='"everything is foreground"')
ax.axhline(RESULTS["pets_zero_shot_iou"], color=THEME["works"], ls=":", label="pets, SAM one box (section 1)")
ax.set_ylabel("mean IoU")
ax.set_ylim(0, 1)
ax.set_title("More prompts, same model")
ax.legend(fontsize=8, frameon=False, loc="upper left")
plt.tight_layout()
plt.show()

show_clicks(
    [(f"SAM, {k}", overlay(demo["image"], m, THEME["fails"]), p) for k, (m, p) in ladder_demo.items()]
    + [("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"]), None)]
)

**Q6.** 🟡 Compare the ladder with your prediction. Did more clicks help, and did the masks get closer to
the vessels or just bigger (look at the "marked" column and the grey line)? What does that tell you about
what SAM thinks an object is?

<details><summary>What to look for</summary>

Read the IoU next to the grey line, not on its own. A mask that marks most of the fundus scores exactly
the "everything" baseline, whatever the prompt. If more clicks push the "marked" share up while IoU stays
near the line, SAM is answering "the object that contains all these points", and for a vessel tree that
object is the eye. If clicks do help on your images, look at where: usually along the thick vessels near
the clicks, rarely the capillaries in between. SAM has no notion of "all the thin dark lines"; a prompt
can only say <i>where</i>, not <i>what kind of thing</i>. Section 7 comes back to prompting with
background clicks too.

</details>

<table width="100%" bgcolor="#FDECEA"><tr><td><font color="#8B1A10">⚠️ <b>Pitfall: over-trusting SAM's
confident masks.</b> SAM returns a crisp mask for every prompt, and its own confidence score is high for
a mask of the whole eye, because the whole eye <i>is</i> a good object. Nothing in the output says "this
is the wrong kind of thing". Always score against a baseline that knows nothing (the grey line), and
look at the mask.</font></td></tr></table>

### 2.4 What DINOv3 sees here

This cell shows the same 3-colour PCA of the DINOv3 patch features as section 1.3, now on specialised
images, so you can see whether the features separate the vessels from the background before any training.

In [ ]:
fig_items = [load_item(SPECIALISED, it) for it in SPECIALISED["fit"][:3]]
panels = []
for it in fig_items:
    feat = features_for(SPECIALISED, it, SPECIALISED["feat_side"])
    title = f"DINOv3 PCA · {feat.shape[2]}×{feat.shape[1]} patches"
    panels += [(it["name"], it["image"]), (title, pca_rgb(feat, it["image"].shape[:2]))]
show_masks(panels, width=2.6)

**Q7.** 🟡 Do the PCA colours follow the vessels, or something else (the bright optic disc, the dark
border, the overall brightness)? Compare with the pets, where one direction separated animal from floor.

<details><summary>What to look for</summary>

The three main directions of variation describe what varies most across the image, and on a fundus that
is rarely the vessels: it is the optic disc, the illumination falling off towards the edge, the border.
Thick vessels may show as a faint trace of blocks; thin ones vanish inside the 16 px squares. That does
not mean the features hold no vessel information (a trained head can still find a direction for it), but
it is no longer the first thing the model "sees".

</details>

### 2.5 The linear head, and the resolution it looks at

The head from section 1.4, trained the same way on the full masks of five specialised images. This is
the last time training sees a full mask: from section 3 on, only incomplete labels. We train it twice,
on DINOv3 features computed at two resolutions. At a 512 px long side, each 16 px patch covers 32 × 32
pixels of the 1024 px crop; at 1024 px, 16 × 16. (DINOv2 papers use 518 px: 37 patches of 14 px. DINOv3's
patches are 16 px, so 512 is the same idea.) Double the side and you get four times the patches, and
roughly four to sixteen times the compute.

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your prediction.</b> With
<i>perfect</i> labels, will the head beat SAM here? And will doubling the resolution help IoU, clDice,
both or neither? Write two guesses.</font></td></tr></table>

This cell trains one head per resolution on the five fit images and prints the seconds it took. Not done
in a minute, or on a CPU and in a hurry? Set `TRAIN_YOURSELF = False` and run it again: it loads our
checkpoints (trained the same way on all fit images).

In [ ]:
TRAIN_YOURSELF = True  # False -> load our precomputed "full" heads instead of training
N_TRAIN_SPEC, HEAD_STEPS = 5, 30 if SMOKE else 200
SIDES = SPECIALISED["feature_long_sides"]  # [512, 1024]: the two resolutions we compare
SIDES = [s // 4 for s in SIDES] if SMOKE else SIDES  # smoke thumbnails are 256 px
if not isinstance(TRAIN_YOURSELF, bool):
    raise SystemExit("TRAIN_YOURSELF must be True or False.")

spec_train = [load_item(SPECIALISED, it) for it in SPECIALISED["fit"][:N_TRAIN_SPEC]]
spec_heads = {}  # long side -> (head tensors, training seconds)
for side in SIDES:
    if TRAIN_YOURSELF:
        feats = [features_for(SPECIALISED, it, side) for it in spec_train]
        labels = [np.where(region(it), it["mask"], -1).astype(np.int8) for it in spec_train]  # border ignored
        spec_heads[side] = train_head(feats, labels, HEAD_STEPS)
        src = f"trained here on {len(spec_train)} images, {HEAD_STEPS} steps"
    else:
        tensors, entry = load_head(SPECIALISED, "full", side)
        spec_heads[side] = (tensors, entry["train_sec"])
        src = f"our checkpoint {entry['file']} ({len(entry['train_names'])} images)"
    print(f"linear head @ {side} px · {src} · {spec_heads[side][1]:.1f} s on {DEVICE}")
if sum(sec for _, sec in spec_heads.values()) > 60:
    print("ℹ️ Slow session? Set TRAIN_YOURSELF = False above to load our checkpoints next time.")

This cell scores both heads on the evaluation images with IoU and clDice, and zooms into the centre of the
demo image so you can compare the two resolutions on the thinnest vessels.

In [ ]:
t0 = time.monotonic()
head_preds, res_scores = {}, {}
for side, (tensors, _) in spec_heads.items():
    head_preds[side] = [
        predict_head(tensors, features_for(SPECIALISED, it, side), it["mask"].shape) for it in SPEC_EVAL
    ]
    res_scores[side] = mean_scores([score(p, it) for p, it in zip(head_preds[side], SPEC_EVAL, strict=True)])
lo, hi = min(SIDES), max(SIDES)
RESULTS["specialised_head_iou"] = res_scores[hi]["iou"]
RESULTS["specialised_head"] = {
    "iou": res_scores[hi]["iou"],
    "cldice": res_scores[hi]["cldice"],
    "long_side": hi,
    "n_train": len(spec_train),
    "train_sec": round(float(spec_heads[hi][1]), 2),
    "infer_sec": round(time.monotonic() - t0, 1),
}
RESULTS["resolution_effect"] = {f"{side} px": v for side, v in res_scores.items()}
for side, v in res_scores.items():
    print(f"linear head @ {side:>4} px · mean IoU {v['iou']:.3f} · clDice {v['cldice']:.3f}")
print(f"SAM one box · mean IoU {ladder_mean['box']['iou']:.3f} · clDice {ladder_mean['box']['cldice']:.3f}")

k = next(i for i, it in enumerate(SPEC_EVAL) if it is demo)
h, w = demo["mask"].shape
zoom = (slice(h // 4, 3 * h // 4), slice(w // 4, 3 * w // 4))  # the central quarter
show_masks(
    [
        (
            f"head @ {s} px · IoU {score(head_preds[s][k], demo)['iou']:.2f}",
            overlay(demo["image"], head_preds[s][k], THEME["fails"])[zoom],
        )
        for s in SIDES
    ]
    +[("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"])[zoom])],
    width=3.0,
)

<table width="100%" bgcolor="#FDECEA"><tr><td><font color="#8B1A10">⚠️ <b>Pitfall: resolution is not
free.</b> Self-attention compares every patch with every other, so doubling the side multiplies the
patches by four and the attention cost by up to sixteen. At 1024 px, one specialised image is 4096 patches.
On the full 3504 px fundus it would be 48 000: that is why we work on crops and why the features are
precomputed. When the structure is thinner than a patch, check the resolution before anything else.
</font></td></tr></table>

### 2.6 The failure gallery

Numbers average the failures away; the gallery shows them. The next cell sorts the evaluation images
from worst to best (by the head's IoU at the higher resolution) and shows three at a time: the image, SAM
with a box, the head, and the ground truth. The last row puts the same models on the whole fundus, shrunk
to 1024 px, where the vessels are only 1–3 px wide.

<table width="100%" bgcolor="#F3E5F5"><tr><td><font color="#4A148C">❓ <b>Question.</b> Run the next
cell, then page through the gallery.</font></td></tr></table>

**Q8.** 🟡 For the worst images, is the failure the same kind for SAM and for the head? Look for: missed
thin vessels, vessels drawn too wide, the optic disc or the border marked as vessel, whole regions marked.
Page with the start index:

```python
GALLERY_START = 0   # 0 = the three worst; try 3, 6, ...
```

This cell draws the failure gallery from the predictions you already have (new model calls only for the
last row) and prints each image's IoU and clDice in the titles.

In [ ]:
# Q8 — the failure gallery, worst images first
GALLERY_START = 0  # 0 = the three worst; try 3, 6, ...
if not isinstance(GALLERY_START, int) or not 0 <= GALLERY_START < len(SPEC_EVAL):
    raise SystemExit(f"GALLERY_START must be a whole number from 0 to {len(SPEC_EVAL) - 1}.")
gallery = [score(p, it) for p, it in zip(head_preds[hi], SPEC_EVAL, strict=True)]
order = np.argsort([s["iou"] for s in gallery])
for k in order[GALLERY_START : GALLERY_START + 3]:
    it, sam_s, head_s = SPEC_EVAL[k], ladder_scores["box"][k], gallery[k]
    show_masks(
        [
            (it["name"], it["image"]),
            (
                f"SAM box · IoU {sam_s['iou']:.2f} · clDice {sam_s['cldice']:.2f}",
                overlay(it["image"], sam_box_masks[k], THEME["fails"]),
            ),
            (
                f"head @ {hi} px · IoU {head_s['iou']:.2f} · clDice {head_s['cldice']:.2f}",
                overlay(it["image"], head_preds[hi][k], THEME["fails"]),
            ),
            ("ground truth", overlay(it["image"], it["mask"], THEME["truth"])),
        ],
        width=2.8,
    )

# the whole fundus of the demo image, shrunk so its long side is 1024 px (smoke: 256 px)
meta = next(m for m in SPECIALISED["eval"] if m["name"] == demo["name"])
root = SPECIALISED["dir"]
full = {
    "image": np.array(Image.open(root / meta["full"]["image"]).convert("RGB")),
    "mask": read_png(root / meta["full"]["mask"]) > 127,
    "fov": read_png(root / meta["full"]["fov"]) > 127,
}
full_sam = sam.probability(full["image"], box=box_prompt(full)) >= 0.5
full_feat = dino.features(full["image"], max(full["image"].shape[:2]))
full_head = predict_head(spec_heads[hi][0], full_feat, full["mask"].shape)
full_masks = {"SAM box": full_sam, f"head @ {hi} px": full_head}
full_s = {name: score(m, full) for name, m in full_masks.items()}
show_masks(
    [(f"whole fundus · {meta['name']}", full["image"])]
    + [
        (f"{name} · IoU {full_s[name]['iou']:.2f} · clDice {full_s[name]['cldice']:.2f}",
         overlay(full["image"], m, THEME["fails"]))
        for name, m in full_masks.items()
    ]
    + [("ground truth", overlay(full["image"], full["mask"], THEME["truth"]))],
    width=2.8,
)
RESULTS["specialised_full_view"] = full_s

<table width="100%" bgcolor="#F3E5F5"><tr><td><font color="#4A148C">👥 <b>Pairs, 5 min: whose image
fails worst?</b> Each of you picks a different <code>GALLERY_START</code> or <code>demo</code> index and
finds the worst mask you can. Show it to your neighbour and name the failure in three words ("misses thin
vessels", "marks the disc", "the whole eye"). Is it the same failure for SAM and for the head?
</font></td></tr></table>

<details><summary>What people usually find</summary>

The two tools fail differently. SAM tends to fail <i>globally</i>: it marks a whole region, or one thick
vessel, because it looks for an object. The head fails <i>locally</i>: it finds the thick vessels, draws
them as wide as a patch, and loses the capillaries, because it can only label 16 px squares. On the
whole fundus both get worse, since every vessel is now thinner than a patch.

</details>

### 2.7 Pets vs the specialised task

This cell puts the section-1 numbers next to this section's, the same three tools on both tasks, so the
gap is one picture.

In [ ]:
rows = {
    "SAM, one click": (RESULTS["pets_zero_shot_point_iou"], RESULTS["specialised_zero_shot_point_iou"]),
    "SAM, one box": (RESULTS["pets_zero_shot_box_iou"], RESULTS["specialised_zero_shot_box_iou"]),
    "linear head": (RESULTS["pets_head_iou"], RESULTS["specialised_head_iou"]),
}
print(f"{'tool':<16}{'pets IoU':>10}{'specialised IoU':>17}")
for name, (p, s) in rows.items():
    print(f"{name:<16}{p:>10.3f}{s:>17.3f}")
print(f"{'everything':<16}{'':>10}{RESULTS['specialised_everything_iou']:>17.3f}")

x = np.arange(len(rows))
fig, ax = plt.subplots(figsize=(6.4, 3))
ax.bar(x - 0.2, [v[0] for v in rows.values()], 0.4, color=THEME["works"], label=PETS["name"])
ax.bar(x + 0.2, [v[1] for v in rows.values()], 0.4, color=THEME["fails"], label=SPECIALISED["name"])
ax.axhline(RESULTS["specialised_everything_iou"], color=THEME["background"], ls="--",
           label='specialised, "everything is foreground"')
ax.set_xticks(x, list(rows))
ax.set_ylabel("mean IoU on held-out images")
ax.set_ylim(0, 1)
ax.set_title("Same tools, two tasks")
ax.legend(fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

<table width="100%" bgcolor="#E8EAF6"><tr><td><font color="#1A237E">🔀 <b>Across approaches.</b> Two
different limits showed up. SAM's limit is its idea of an object: more prompts make the mask bigger, not
thinner. The head's limit is the patch: even with perfect labels it can only answer per 16 px square, and
resolution is one lever (look at how far it moved IoU and clDice, and whether they agreed). From here on
the question changes from "does it work?" to "what information can we add, cheaply, to make it work?".
Perfect masks cost 3–5 hours per fundus, so the next sections only use cheap, incomplete ones.
</font></td></tr></table>

**Q9.** 🟡 You now have a small checklist for a new problem. Which two or three signs from this section
would tell you, on your own data, that a foundation model is not good enough out of the box?

<details><summary>One possible answer</summary>

(1) Its score sits near a baseline that knows nothing ("everything is foreground"), so compare with one.
(2) More or better prompts change the <i>size</i> of the mask but not its <i>shape</i>. (3) The objects
are smaller than a feature patch, and a higher resolution moves the numbers: a sign the representation,
not the labels, is the bottleneck. (4) A pixel metric and a structure metric (IoU vs clDice) disagree.
And always: the mask looks crisp and confident, so look at it next to the truth.

</details>

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Try it on your image (optional,
2 min).</b> Got a photo of something thin or specialised (a leaf's veins, a crack, a cable, a
micrograph)? Upload it with the folder icon on the left, put its path in <code>MY_IMAGE</code> and one or
more <code>(x, y)</code> clicks on the thing in <code>MY_CLICKS</code>, then run the cell. Left empty, it
runs on the demo image. Section 9 does this properly, with scribbles and the whole ladder.
</font></td></tr></table>

This cell runs SAM on your image (or the demo image) with your clicks and shows the mask, the quickest way
to see whether your own problem looks more like the pets or like this section.

In [ ]:
MY_IMAGE = ""  # e.g. "/content/my_photo.jpg"; empty = the demo image
MY_CLICKS = []  # e.g. [(120, 80), (300, 210)]; empty = the image centre
if not isinstance(MY_IMAGE, str) or not isinstance(MY_CLICKS, list):
    raise SystemExit('MY_IMAGE must be a path in quotes (or "") and MY_CLICKS a list of (x, y) pairs.')
if MY_IMAGE and not Path(MY_IMAGE).is_file():
    raise SystemExit(f"{MY_IMAGE} not found. Upload it with the folder icon on the left and copy its path.")
my_img = np.array(Image.open(MY_IMAGE).convert("RGB")) if MY_IMAGE else demo["image"]
my_h, my_w = my_img.shape[:2]
my_clicks = [[float(x), float(y)] for x, y in MY_CLICKS] or [[my_w / 2, my_h / 2]]
if not all(0 <= x < my_w and 0 <= y < my_h for x, y in my_clicks):
    raise SystemExit(f"every click must lie inside the image: 0 ≤ x < {my_w}, 0 ≤ y < {my_h}.")
my_mask = sam.probability(my_img, points=my_clicks, labels=[1] * len(my_clicks)) >= 0.5
show_clicks(
    [
        (MY_IMAGE or f"demo · {demo['name']}", my_img, my_clicks),
        (
            f"SAM, {len(my_clicks)} click(s) · marks {my_mask.mean():.0%}",
            overlay(my_img, my_mask, THEME["fails"]),
            my_clicks,
        ),
    ],
    width=3.2,
)

## 3. Hide the labels: the naive baseline

**Objective:** train a head with "missing = background" and watch it fail differently.

A perfect vessel map of one fundus takes an expert 3 to 5 hours, so real label files are rarely complete.
Picture an annotator who traced the big vessels, ran out of time halfway through the tree, and saved the
file. Nothing in that file says "I stopped here": every pixel they did not paint is stored as background,
exactly like the pixels that really are background. In this section we make such a file ourselves, by
hiding about half of the vessels from the ground truth, train the section-2 head on it the obvious way,
and score it on the vessels we hid. From here on the full masks are used for scoring only.

<table width="100%" bgcolor="#EDE7F6"><tr><td><font color="#311B92">📘 <b>Theory: incomplete
supervision.</b> The rest of the notebook trains on cheap labels made from the full ground truth, which we
keep hidden for evaluation. <b>Missing labels</b> (this section and section 5): a share of the objects is
fully labelled, the rest is silently left out. <b>Scribbles</b> (section 4): a few strokes on some
objects and some background, everything else unlabelled. Both leave pixels nobody looked at. The question
of this block is what a model should believe about those pixels. The obvious answer, and the one you get
by passing the label file straight into a loss, is "background".</font></td></tr></table>

This helper cell defines how we hide labels, the same way `scripts/prepare_data.py` built the cached
files: cut the ground-truth skeleton into segments at its junctions (connected objects when the structures
are not thin), give every foreground pixel to its nearest segment, then hide a random share of the
segments.

In [ ]:
# @title Missing labels: cut into segments, hide a share { display-mode: "form" }
import zlib

from scipy import ndimage as ndi


def label_units(mask: np.ndarray, thin: bool) -> np.ndarray:
    """Label map of the units we hide (0 = background): skeleton segments for thin structures or a single
    object, connected components otherwise. Every foreground pixel gets the label of its nearest segment."""
    lab, n = ndi.label(mask)
    if not thin and n >= 2:
        return lab
    sk = skeleton(mask)
    neighbours = ndi.convolve(sk.astype(np.uint8), np.ones((3, 3), np.uint8), mode="constant") - sk
    seg, _ = ndi.label(sk & ~(sk & (neighbours > 2)), structure=np.ones((3, 3)))  # cut at junctions
    if seg.max() == 0:
        return np.where(mask, 1, 0)
    _, (iy, ix) = ndi.distance_transform_edt(seg == 0, return_indices=True)
    return np.where(mask, seg[iy, ix], 0)


def hide_units(mask: np.ndarray, fraction: float, rng: np.random.Generator, thin: bool) -> np.ndarray:
    """The 'missing labels' file: the ground truth with ~fraction of its units removed. Returns the visible
    mask; the hidden foreground is mask & ~visible."""
    lab = label_units(mask, thin)
    ids = np.arange(1, int(lab.max()) + 1)
    rng.shuffle(ids)
    k = int(round(len(ids) * fraction))
    hidden_ids = ids[: max(1, k)] if len(ids) >= 2 else ids[:0]
    return mask & ~np.isin(lab, hidden_ids)


def hide_rng(name: str) -> np.random.Generator:
    """One fixed random generator per image, so the same segments are hidden on every run (N-013)."""
    return np.random.default_rng([SEED, zlib.crc32(f"{name}/hide".encode())])


def part_recall(preds: list[np.ndarray], items: list[dict], part: str) -> float:
    """Mean over images of the share of item[part] ("mask", "visible", "hidden"), inside the evaluation
    region, that the prediction marks. Uses `hidden_recall` from 3.2."""
    pairs = zip(preds, items, strict=True)
    return float(np.mean([hidden_recall(p, it[part] & region(it)) for p, it in pairs]))


def two_colour(image: np.ndarray, good: np.ndarray, bad: np.ndarray) -> np.ndarray:
    """Overlay `good` in violet ("works") and `bad` in orange ("fails") on one image."""
    return overlay(overlay(image, good, THEME["works"]), bad, THEME["fails"])

### 3.1 Make a half-finished label file

This cell takes the five specialised training images of section 2.5, hides about half of each image's
vessel segments (the share comes from the dataset config), and shows one of them: the full truth we keep
for scoring, the labels training may use, and what the label file now wrongly calls background.

In [ ]:
MISSING = SPECIALISED["missing_labels"]  # {"hidden_fraction": 0.5, "unit": ...}: never hard-coded here
miss_train = [load_item(SPECIALISED, it) for it in SPECIALISED["fit"][:N_TRAIN_SPEC]]
same_as_cache = []
for it in miss_train:
    rng = hide_rng(it["name"])
    visible = hide_units(it["mask"], MISSING["hidden_fraction"], rng, SPECIALISED["thin_structures"])
    same_as_cache.append(float((visible == it["visible"]).mean()))
    it["visible"], it["hidden"] = visible, it["mask"] & ~visible  # from now on: what training may see
hidden_share = float(np.mean([it["hidden"].sum() / max(it["mask"].sum(), 1) for it in miss_train]))
print(
    f"{len(miss_train)} training images · {hidden_share:.0%} of the vessel pixels hidden "
    f"(asked for {MISSING['hidden_fraction']:.0%} of the segments) · pixels identical to the cached labels "
    f"our checkpoints used: {np.mean(same_as_cache):.2%}"
)

ex = miss_train[0]
show_masks(
    [
        (ex["name"], ex["image"]),
        ("ground truth · evaluation only", overlay(ex["image"], ex["mask"], THEME["truth"])),
        ("labels training sees: vessel", overlay(ex["image"], ex["visible"], THEME["scribble"])),
        ('hidden: the file says "background"', overlay(ex["image"], ex["hidden"], THEME["fails"])),
    ],
    width=2.8,
)

**Q10.** 🟡 Look at the third panel on its own. If a colleague handed you that label file, could you tell
that half of it is missing? What would you need to know to tell?

<details><summary>What to look for</summary>

Usually not. The visible vessels are clean and correct, they simply stop, and a vessel tree that stops
looks plausible. The file has two values, vessel and background, and no third value for "nobody looked
here". To tell, you would need to know how the file was made (who annotated, for how long, what they were
told to skip), which is exactly the information that gets lost. That is why we hid the labels on purpose:
here we know which pixels are missing, so we can measure what the model does with them.

</details>

### 3.2 A metric for the missing vessels: recall

<table width="100%" bgcolor="#EDE7F6"><tr><td><font color="#311B92">📘 <b>Theory: recall and
precision.</b> <b>Recall</b> is the share of the true vessel pixels that the prediction marks: it asks
"of what is there, how much did you find?". <b>Precision</b> is the share of the marked pixels that are
true vessels: "of what you marked, how much is right?". IoU mixes the two into one number. Here we want
them apart, because the question is specific: of the vessels we hid, which the label file calls
background, how many does the trained head still find? That is recall on the hidden foreground.
</font></td></tr></table>

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your turn (optional, 1 min).</b>
Fill <code>my_hidden_recall</code>: a boolean prediction and a boolean mask of hidden pixels in, one
float out, the share of the hidden pixels the prediction marks. Return 0.0 when nothing is hidden. The
cell checks your version on two play masks and continues with ours if you skip it.</font></td></tr></table>

In [ ]:
# ✍️ YOUR CODE
def my_hidden_recall(pred: np.ndarray, hidden: np.ndarray) -> float:
    """Share of the `hidden` pixels that `pred` marks as foreground; 0.0 when `hidden` is empty."""
    ...


def _ref_hidden_recall(pred: np.ndarray, hidden: np.ndarray) -> float:
    """Reference: the prediction's mean over the hidden pixels."""
    hidden = np.asarray(hidden, bool)
    return float(np.asarray(pred, bool)[hidden].mean()) if hidden.any() else 0.0


_play_hidden = np.zeros((8, 8), bool)
_play_hidden[2, :] = True  # a hidden vessel, 8 px long
_play_pred = np.zeros((8, 8), bool)
_play_pred[1:4, :2] = True  # the prediction finds its first 2 px, plus some background
hidden_recall = use(
    my_hidden_recall,
    reference=_ref_hidden_recall,
    check=lambda f: [round(float(f(_play_pred, h)), 4) for h in (_play_hidden, np.zeros((8, 8), bool))],
    name="hidden_recall",
)
print(f"play example · recall on the hidden vessel: {hidden_recall(_play_pred, _play_hidden):.2f}")

### 3.3 The naive baseline: missing = background

The naive recipe is what you get by passing the label file straight to the textbook loss: every pixel
not painted as vessel is a background target, and plain binary cross-entropy counts every pixel equally.
Two things change from section 2.5, the labels and the loss weighting; 3.4 separates them. Everything else
is the same: the same five images, the same DINOv3 features, the same linear head, the same steps.

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your prediction.</b> The head
was told that half of the vessels in these five images are background. On those hidden vessels, will its
recall be near the section-2 head's, about half of it, or close to zero? And on new images, will it mark
more pixels than the section-2 head, or fewer? Write two guesses.</font></td></tr></table>

This cell trains the naive head on the incomplete labels only (the fundus border is ignored, as in 2.5)
and prints the seconds it took. Not done in a minute, or on a CPU and in a hurry? Set
`TRAIN_YOURSELF = False` and run it again: it loads our checkpoint, trained the same way on all fit images.

In [ ]:
TRAIN_YOURSELF = True  # False -> load our precomputed "hidden_bg" head instead of training
if not isinstance(TRAIN_YOURSELF, bool):
    raise SystemExit("TRAIN_YOURSELF must be True or False.")

side = SPECIALISED["feat_side"]
miss_feats = [features_for(SPECIALISED, it, side) for it in miss_train]
# missing = background: the label file as it is. The hidden ground truth never enters training (L-005).
naive_labels = [np.where(region(it), it["visible"], -1).astype(np.int8) for it in miss_train]
if TRAIN_YOURSELF:
    naive_head, naive_sec = train_head(miss_feats, naive_labels, HEAD_STEPS, balanced=False)
    src = f"trained here on {len(miss_train)} images, {HEAD_STEPS} steps, plain cross-entropy"
else:
    naive_head, entry = load_head(SPECIALISED, "hidden_bg", side)
    naive_sec = entry["train_sec"]
    src = f"our checkpoint {entry['file']} ({len(entry['train_names'])} images)"
print(f"naive head @ {side} px · {src} · {naive_sec:.1f} s on {DEVICE}")
if naive_sec > 60:
    print("ℹ️ Slow session? Set TRAIN_YOURSELF = False above to load our checkpoint next time.")

This cell scores the naive head and the section-2 head (full masks) on three groups of vessel pixels: the
ones we hid in the training images, the ones training saw, and every vessel of the held-out images. It
prints recall per group plus IoU, clDice and precision on the held-out images, and plots the recalls.

In [ ]:
full_head = spec_heads[side][0]  # section 2.5: same features, trained on full masks
HEADS = {"full masks (section 2)": full_head, "naive: missing = background": naive_head}
t0 = time.monotonic()
naive_report = {}
for name, tensors in HEADS.items():
    fit_preds = [
        predict_head(tensors, f, it["mask"].shape) for f, it in zip(miss_feats, miss_train, strict=True)
    ]
    held_preds = [
        predict_head(tensors, features_for(SPECIALISED, it, side), it["mask"].shape) for it in SPEC_EVAL
    ]
    held = mean_scores([score(p, it) for p, it in zip(held_preds, SPEC_EVAL, strict=True)])
    # precision = the share of the marked pixels that are true vessels
    held_pairs = zip(held_preds, SPEC_EVAL, strict=True)
    precision = [hidden_recall(it["mask"], p & region(it)) for p, it in held_pairs]
    naive_report[name] = {
        "recall_hidden_fit": part_recall(fit_preds, miss_train, "hidden"),
        "recall_visible_fit": part_recall(fit_preds, miss_train, "visible"),
        "recall_held": part_recall(held_preds, SPEC_EVAL, "mask"),
        "precision_held": float(np.mean(precision)),
        "iou_held": held["iou"],
        "cldice_held": held["cldice"],
        "fit_preds": fit_preds,
        "held_preds": held_preds,
    }
infer_sec = round(time.monotonic() - t0, 1)

naive = naive_report["naive: missing = background"]
RESULTS["naive_recall_hidden_fg"] = naive["recall_hidden_fit"]  # recall on the vessels hidden in training
RESULTS["naive_head"] = {k: v for k, v in naive.items() if not k.endswith("_preds")} | {
    "long_side": side,
    "n_train": len(miss_train),
    "hidden_share": hidden_share,
    "train_sec": round(float(naive_sec), 2),
    "infer_sec": infer_sec,
}

GROUPS = {
    "recall_hidden_fit": "hidden vessels\n(told: background)",
    "recall_visible_fit": "visible vessels\n(told: vessel)",
    "recall_held": "held-out images\n(all vessels)",
}
print(f"{'head':<30}{'recall: hidden':>15}{'visible':>9}{'held-out':>10}   held-out IoU / clDice / precision")
for name, r in naive_report.items():
    print(
        f"{name:<30}{r['recall_hidden_fit']:>15.3f}{r['recall_visible_fit']:>9.3f}{r['recall_held']:>10.3f}"
        f"   {r['iou_held']:.3f} / {r['cldice_held']:.3f} / {r['precision_held']:.3f}"
    )

x = np.arange(len(GROUPS))
fig, ax = plt.subplots(figsize=(6.4, 3))
for shift, name, colour in zip((-0.2, 0.2), HEADS, (THEME["works"], THEME["fails"]), strict=True):
    ax.bar(x + shift, [naive_report[name][g] for g in GROUPS], 0.4, color=colour, label=name)
ax.set_xticks(x, list(GROUPS.values()), fontsize=8)
ax.set_ylabel("recall (share of vessel pixels found)")
ax.set_ylim(0, 1)
ax.set_title("Which vessels does each head still find?")
ax.legend(fontsize=8, frameon=False, loc="upper right")
plt.tight_layout()
plt.show()

<table width="100%" bgcolor="#F3E5F5"><tr><td><font color="#4A148C">❓ <b>Question.</b> Run the next
cell: it shows where the naive head looks for vessels, first on a training image, then on the held-out
demo image of section 2.</font></td></tr></table>

**Q11.** 🟡 On the training image, which hidden vessels did the naive head miss (orange), and which did it
still find (violet)? On the held-out image, compare it with the section-2 head: does it fail the same way
(too wide, the disc, the whole eye) or differently? Try another training image:

```python
FAIL_IMAGE = 0   # try 1, 2, ...
```

This cell draws the failure figure from the predictions you already have: no new training, no model calls.

In [ ]:
# Q11 — where the naive head fails: a training image, then the held-out demo image
FAIL_IMAGE = 0  # try 1, 2, ...
if not isinstance(FAIL_IMAGE, int) or not 0 <= FAIL_IMAGE < len(miss_train):
    raise SystemExit(f"FAIL_IMAGE must be a whole number from 0 to {len(miss_train) - 1}.")
it, pred = miss_train[FAIL_IMAGE], naive["fit_preds"][FAIL_IMAGE]
found, missed = it["hidden"] & pred, it["hidden"] & ~pred
show_masks(
    [
        ("labels training saw", overlay(it["image"], it["visible"], THEME["scribble"])),
        ("naive head", overlay(it["image"], pred, THEME["fails"])),
        (
            f"hidden vessels · found {hidden_recall(pred, it['hidden']):.0%}",
            two_colour(it["image"], found, missed),
        ),
    ],
    width=3.0,
)
k = next(i for i, e in enumerate(SPEC_EVAL) if e is demo)
show_masks(
    [
        (
            f"{name} · IoU {score(r['held_preds'][k], demo)['iou']:.2f}",
            overlay(demo["image"], r["held_preds"][k], THEME["fails"]),
        )
        for name, r in naive_report.items()
    ]
    + [("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"]))],
    width=3.0,
)

<details><summary>What to look for</summary>

The naive head did what the labels told it. Half of the vessels it saw were labelled background, so a
vessel-looking patch became, for this head, more likely background than not, and plain cross-entropy
turns "less than 50 %" into "background". It keeps the clearest, thickest vessels and drops the rest: on
the hidden segments, on new images, and often even on the visible segments it was shown. Compare the
precision column: what it does mark is mostly right. The section-2 head failed by being too generous
(vessels drawn as wide as a patch); this one fails by being too stingy. Now look at the held-out IoU of
the two heads: it can be close, because IoU rewards the precision the naive head gained as much as it
punishes the recall it lost. One number would have hidden the failure. Same features, same head: the only
thing that changed is what we told it about the pixels nobody labelled.

</details>

<table width="100%" bgcolor="#FDECEA"><tr><td><font color="#8B1A10">⚠️ <b>Pitfall: a precise model looks
trustworthy.</b> Every vessel the naive head marks is a real vessel, so a quick look at a few predictions,
or a precision score, says "works". The failure is in what is <i>not</i> marked, and you can only measure
that against labels that are complete. On your own data, keep a small, fully labelled evaluation set, even
if training uses cheap labels.</font></td></tr></table>

### 3.4 🔴 Optional: is it the labels, or the loss?

Plain cross-entropy is one of the two changes from section 2.5. The heads of sections 1 and 2 used a
<i>balanced</i> loss, which gives the vessel pixels as much total weight as the background pixels.

**Q12.** 🔴 Train the same naive labels with the balanced loss of section 2.5 and compare its recall on
the hidden vessels with the plain head's. Does reweighting fix "missing = background"? What did it fix?

```python
head, sec = train_head(miss_feats, naive_labels, HEAD_STEPS, balanced=True)
```

This cell trains that balanced variant on the same incomplete labels (seconds) and prints its recall on
the three groups next to the plain head's.

In [ ]:
# Q12 — the same "missing = background" labels with the balanced loss of section 2.5
balanced_head, balanced_sec = train_head(miss_feats, naive_labels, HEAD_STEPS, balanced=True)
bal_fit = [
    predict_head(balanced_head, f, it["mask"].shape) for f, it in zip(miss_feats, miss_train, strict=True)
]
bal_hidden, bal_visible = (part_recall(bal_fit, miss_train, part) for part in ("hidden", "visible"))
RESULTS["naive_balanced_recall_hidden_fg"] = bal_hidden
print(f"balanced head · trained in {balanced_sec:.1f} s on {DEVICE}")
print(f"{'loss':<12}{'recall: hidden':>15}{'visible':>9}")
print(f"{'plain':<12}{naive['recall_hidden_fit']:>15.3f}{naive['recall_visible_fit']:>9.3f}")
print(f"{'balanced':<12}{bal_hidden:>15.3f}{bal_visible:>9.3f}")

<details><summary>What to look for</summary>

Reweighting usually recovers a lot of the hidden vessels. It does not fix the labels, though: it fixes
the <i>proportion</i>. Missing = background told the head that vessels are about half as common as they
are, and the balanced loss stops caring about proportions, so the hidden vessels come back as a side
effect. It works here for two reasons that need not hold on your data: we hid segments <i>at random</i>,
so the hidden ones look like the visible ones, and a linear head is too simple to memorise which segment
was hidden. If annotators systematically skip the thin capillaries, the head learns "thin = background",
and no class weight undoes that. Section 5 fixes the cause instead: tell the loss which pixels are
unknown.

</details>

<table width="100%" bgcolor="#E8EAF6"><tr><td><font color="#1A237E">🔀 <b>Across approaches.</b> Block 1
in one line each. Pets: the foundation model works out of the box. Specialised task: SAM looks for objects
and finds the eye; the head with perfect masks is limited by its 16 px patches. Incomplete labels, used
naively: the head believes the label file and learns that vessels are background. Three different
failures, and none of them is fixed by "a bigger model". Keep this section's recall on the hidden vessels:
it is the baseline every approach after the break has to beat.</font></td></tr></table>

### ☕ Before the break

<table width="100%" bgcolor="#F3E5F5"><tr><td><font color="#4A148C">❓ <b>Question for the coffee
break.</b> No code, just think about it over coffee.</font></td></tr></table>

**Q13.** 🟡 Perfect labels cost hours per image, and the cheap ones lie: the label file says "background"
wherever nobody looked. Without buying more labels, what could you tell the training about a pixel nobody
labelled? Think of at least two options before section 4.

<details><summary>A hint, for after the break</summary>

Three families of answers come back in block 2. (1) Say "unknown" instead of "background", and let the
loss skip those pixels (sections 4 and 5). (2) Guess the missing labels with another model, for example
SAM prompted from the labels you have, and train on the guesses (section 6). (3) Skip training and give
the cheap labels to the foundation model directly as prompts (section 7).

</details>

## 4. The adaptation ladder

**Objective:** zero-shot → one-shot → scribbles: see how predictions change at each rung.

Welcome back. Before the break the head believed a label file that said "background" wherever nobody had
looked. Block 2 is about the ways out, and the first one is a ladder: give the foundation model a little
more task-specific information at each rung and look at what changes. Rung 0 is section 2, no training,
just a prompt. Rung 1 is **one-shot**: a single fully annotated image. Rung 2 is **scribbles**: a few
quick strokes on several images, and a loss that skips every pixel nobody labelled. Each rung costs a
different kind of effort, and each one fails in its own way. Higher on the ladder is not automatically
better, so we look at the masks, not only at the numbers.

<table width="100%" bgcolor="#EDE7F6"><tr><td><font color="#311B92">📘 <b>Theory: what each rung tells
the model.</b> <b>Zero-shot</b> (section 2): SAM gets a box and nothing about the task; it returns its
best guess of "the object". <b>One-shot</b>: one image with a complete mask. With frozen DINOv3 features
the simplest use of it is a <i>prototype</i>: the mean feature of the vessel patches and the mean feature
of the background patches; every patch of a new image goes to the nearer of the two. The same map can
also pick clicks for SAM, the idea behind PerSAM (Zhang et al., "Personalize Segment Anything Model with
One Shot", ICLR 2024). <b>Scribbles</b>: strokes on some vessels and some background in a few images,
everything else <i>unlabelled</i>. The section-2 head is trained with a <i>partial</i> cross-entropy
that sums only over the labelled pixels (Tang et al., ECCV 2018), the recipe the MIT "DINOv2 + scribbles"
work used on frozen features, here on DINOv3. One complete mask costs hours; a scribble costs seconds.
</font></td></tr></table>

This helper cell defines the tools of the ladder: one colour per approach (kept for the rest of the
notebook), prototypes from patch features, clicks picked from a similarity map, an error map that shows
what each prediction found, missed and invented, and a small report of the numbers we compare.

In [ ]:
# @title Ladder helpers: approach colours, prototypes, clicks, error maps { display-mode: "form" }
APPROACH_COLOURS = {  # one colour per approach (L-003); later sections add theirs and keep these
    "zero_shot": "#E65100",
    "one_shot": "#C2185B",
    "one_shot_sam": "#F06292",
    "scribbles": "#3949AB",
}
APPROACH_NAMES = {
    "zero_shot": "zero-shot: SAM, one box",
    "one_shot": "one-shot: nearest prototype",
    "one_shot_sam": "one-shot: prototype → SAM clicks",
    "scribbles": "scribbles: partial cross-entropy",
}


def patch_share(mask: np.ndarray, grid: tuple[int, int]) -> torch.Tensor:
    """Share of each feature patch covered by a boolean pixel mask: [h, w] floats in [0, 1]."""
    m = torch.from_numpy(np.asarray(mask, np.float32))[None, None]
    return F.interpolate(m, size=grid, mode="area")[0, 0].to(DEVICE)


def prototypes(feat: torch.Tensor, w_fg: torch.Tensor, w_bg: torch.Tensor) -> tuple[torch.Tensor, ...]:
    """Foreground and background prototypes: the weighted means of the unit-length patch features."""
    f = F.normalize(feat.flatten(1), dim=0)  # [D, h*w]
    return tuple(F.normalize(f @ w.flatten().to(f), dim=0) for w in (w_fg, w_bg))


@torch.no_grad()
def prototype_score(feat: torch.Tensor, protos: tuple[torch.Tensor, torch.Tensor]) -> torch.Tensor:
    """Per patch: cosine similarity to the foreground prototype minus to the background one ([h, w])."""
    return torch.einsum("dhw,d->hw", F.normalize(feat, dim=0), protos[0] - protos[1])


def score_to_mask(s: torch.Tensor, shape: tuple[int, int]) -> np.ndarray:
    """Upsample a patch score map to the image and mark the pixels nearer the foreground prototype."""
    up = F.interpolate(s[None, None], size=shape, mode="bilinear", align_corners=False)[0, 0]
    return (up > 0).cpu().numpy()


def heat_rgb(s: torch.Tensor, shape: tuple[int, int]) -> np.ndarray:
    """A score map as an RGB image: purple = like the foreground prototype, orange = like the background."""
    up = F.interpolate(s[None, None], size=shape, mode="bilinear", align_corners=False)[0, 0].cpu().numpy()
    up = 0.5 + 0.5 * up / (np.abs(up).max() + 1e-9)
    return (mpl.colormaps["PuOr"](up)[..., :3] * 255).astype(np.uint8)


def top_points(s: torch.Tensor, keep: torch.Tensor, k: int, shape: tuple[int, int], largest: bool = True):
    """k (x, y) clicks at the centres of the patches that score highest (or lowest) inside `keep`, at least
    3 patches apart: how a similarity map becomes SAM prompts (the PerSAM idea)."""
    s = (s if largest else -s).clone()
    s[keep < 0.5] = -float("inf")
    (h, w), (H, W) = s.shape, shape
    pts = []
    for _ in range(k):
        idx = int(s.argmax())
        i, j = divmod(idx, w)
        if not torch.isfinite(s[i, j]):
            break
        pts.append([(j + 0.5) * W / w, (i + 0.5) * H / h])
        s[max(0, i - 2) : i + 3, max(0, j - 2) : j + 3] = -float("inf")
    return pts


def error_map(image: np.ndarray, pred: np.ndarray, item: dict) -> np.ndarray:
    """Inside the evaluation region: vessels found in violet, false alarms in orange, missed vessels in
    green (the ground truth the prediction did not cover)."""
    r = region(item)
    truth, p = item["mask"] & r, pred & r
    out = overlay(image, truth & ~p, THEME["truth"])
    out = overlay(out, p & truth, THEME["works"])
    return overlay(out, p & ~truth, THEME["fails"])


def rung_report(preds: list[np.ndarray]) -> dict:
    """Mean IoU, clDice, recall, precision and marked share of the evaluation region, over SPEC_EVAL."""
    out = mean_scores([score(p, it) for p, it in zip(preds, SPEC_EVAL, strict=True)])
    out["recall"] = part_recall(preds, SPEC_EVAL, "mask")
    pairs = zip(preds, SPEC_EVAL, strict=True)
    out["precision"] = float(np.mean([hidden_recall(it["mask"], p & region(it)) for p, it in pairs]))
    out["marked"] = float(np.mean([p[region(it)].mean() for p, it in zip(preds, SPEC_EVAL, strict=True)]))
    return out


def effort_minutes(n_strokes: int = 0, n_full_masks: int = 0) -> float:
    """Annotation minutes under the dataset's stated assumptions (seconds per stroke, hours per full mask)."""
    cost = SPECIALISED["annotation_effort"]
    return n_strokes * cost["seconds_per_stroke"] / 60 + n_full_masks * 60 * float(
        np.mean(cost["hours_per_full_mask"])
    )

### 4.1 Rung 0: zero-shot, from section 2

Nothing new to run for rung 0: SAM with one box on every evaluation image, the section-2 headline. This
cell picks up those masks and computes the numbers every rung of this section is compared on: IoU and
clDice as before, plus the recall and precision of section 3, and the share of the fundus each mask marks.

In [ ]:
k_demo = next(i for i, it in enumerate(SPEC_EVAL) if it is demo)  # the section-2 demo image
ladder_preds = {"zero_shot": sam_box_masks}  # section 2.3: SAM, one box, per evaluation image
ladder_report = {"zero_shot": rung_report(sam_box_masks)}
v = ladder_report["zero_shot"]
print(
    f"zero-shot (SAM, one box) on {len(SPEC_EVAL)} held-out images · IoU {v['iou']:.3f} · clDice "
    f"{v['cldice']:.3f} · recall {v['recall']:.3f} · precision {v['precision']:.3f} · marks {v['marked']:.0%}"
)

### 4.2 Rung 1: one annotated image

Someone spends an afternoon and traces every vessel of **one** training image. That complete mask is all
this rung may use (L-005): the held-out ground truth stays for scoring. From it we build the two
prototypes, "what a vessel patch looks like" and "what a background patch looks like", and label every
patch of a held-out image by the nearer one. No training: one mean per class.

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your prediction.</b> One
complete example against SAM's box from rung 0. Will the prototype mark more of the fundus than SAM, or
less? Will it find the thin vessels, or only the thick ones? Write one guess for each.
</font></td></tr></table>

<table width="100%" bgcolor="#F3E5F5"><tr><td><font color="#4A148C">❓ <b>Question.</b> Run the next
cell, then change which training image is the annotated one.</font></td></tr></table>

**Q14.** 🟡 Look at the similarity map: which parts of the held-out image look most like the annotated
vessels (purple), and which like the background (orange)? Then pick another annotated image: does the
held-out IoU move much? What does that say about relying on a single example?

```python
ONE_SHOT_IMAGE = 0   # try 1, 2, ...
```

This cell builds the two prototypes from the annotated image, scores every evaluation image against them
(one DINOv3 feature grid each, seconds), and shows the annotated image, the similarity map on the demo
image, and the resulting mask.

In [ ]:
# Q14 — one-shot: the prototypes of one annotated image, applied to every held-out image
ONE_SHOT_IMAGE = 0  # which training image is the annotated one; try 1, 2, ...
if not isinstance(ONE_SHOT_IMAGE, int) or not 0 <= ONE_SHOT_IMAGE < len(SPECIALISED["fit"]):
    raise SystemExit(f"ONE_SHOT_IMAGE must be a whole number from 0 to {len(SPECIALISED['fit']) - 1}.")
side = SPECIALISED["feat_side"]
shot = load_item(SPECIALISED, SPECIALISED["fit"][ONE_SHOT_IMAGE])
shot_feat = features_for(SPECIALISED, shot, side)
grid, r = tuple(shot_feat.shape[1:]), region(shot)
# the rung's supervision: one complete mask of one *training* image (L-005)
protos = prototypes(shot_feat, patch_share(shot["mask"] & r, grid), patch_share(~shot["mask"] & r, grid))
t0 = time.monotonic()
proto_scores = [prototype_score(features_for(SPECIALISED, it, side), protos) for it in SPEC_EVAL]
ladder_preds["one_shot"] = [
    score_to_mask(s, it["mask"].shape) for s, it in zip(proto_scores, SPEC_EVAL, strict=True)
]
ladder_report["one_shot"] = rung_report(ladder_preds["one_shot"])
proto_sec = round(time.monotonic() - t0, 1)
v = ladder_report["one_shot"]
print(
    f"one-shot prototype from {shot['name']} · IoU {v['iou']:.3f} · clDice {v['cldice']:.3f} · recall "
    f"{v['recall']:.3f} · precision {v['precision']:.3f} · marks {v['marked']:.0%} · "
    f"{proto_sec} s on {DEVICE}"
)
print(f"for scale: the section-2 head on five complete masks · IoU {RESULTS['specialised_head_iou']:.3f}")
show_masks(
    [
        (f"the one annotated image · {shot['name']}", overlay(shot["image"], shot["mask"], THEME["truth"])),
        ("similarity: vessel − background", heat_rgb(proto_scores[k_demo], demo["mask"].shape)),
        (
            f"one-shot prototype · IoU {score(ladder_preds['one_shot'][k_demo], demo)['iou']:.2f}",
            overlay(demo["image"], ladder_preds["one_shot"][k_demo], THEME["fails"]),
        ),
        ("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"])),
    ],
    width=2.8,
)

<details><summary>What to look for</summary>

The prototype answers "which patches look like the annotated vessel patches", one 16 px square at a time.
Expect the purple to follow the thick vessels and the dark, textured parts of the fundus, and the mask to
be drawn as wide as a patch: the prototype has the same resolution limit as the section-2 head. Compare
the two IoUs printed above: one example and a mean, against five complete masks and a trained head. More
labels do not automatically give a better mask, and a simpler rule can fail less. How much the IoU moves
when you swap the annotated image tells you how much this rung depends on <i>which</i> example you
happened to annotate: with one image, a lucky or unlucky choice is the whole training set.

</details>

The same similarity map can do something else: point SAM at the right places. This cell takes, on every
evaluation image, the five patches most like a vessel as positive clicks and the five most like background
as negative clicks, and asks SAM for a mask (one SAM image encoding per image: seconds, also on a CPU).

In [ ]:
N_PROTO_CLICKS = 5  # positive clicks, and as many negative ones, per image
t0 = time.monotonic()
ladder_preds["one_shot_sam"], proto_clicks = [], []
for it, s in zip(SPEC_EVAL, proto_scores, strict=True):
    keep = patch_share(region(it), tuple(s.shape))
    pos = top_points(s, keep, N_PROTO_CLICKS, it["mask"].shape, largest=True)
    neg = top_points(s, keep, N_PROTO_CLICKS, it["mask"].shape, largest=False)
    prob = sam.probability(it["image"], points=pos + neg, labels=[1] * len(pos) + [0] * len(neg))
    ladder_preds["one_shot_sam"].append(prob >= 0.5)
    proto_clicks.append((pos, neg))
ladder_report["one_shot_sam"] = rung_report(ladder_preds["one_shot_sam"])
proto_sam_sec = round(time.monotonic() - t0, 1)
v = ladder_report["one_shot_sam"]
print(
    f"one-shot prototype → SAM clicks · IoU {v['iou']:.3f} · clDice {v['cldice']:.3f} · recall "
    f"{v['recall']:.3f} · precision {v['precision']:.3f} · marks {v['marked']:.0%} · {proto_sam_sec} s"
)
pos, neg = proto_clicks[k_demo]
demo_sam = overlay(demo["image"], ladder_preds["one_shot_sam"][k_demo], THEME["fails"])
show_clicks(
    [
        ("clicks chosen as vessel", demo["image"], pos),
        ("clicks chosen as background", demo["image"], neg),
        ("SAM from those clicks", demo_sam, pos),
        ("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"]), None),
    ],
    width=2.8,
)

**Q15.** 🟡 Same annotated image, same similarity map, two ways to use it. Where do the prototype mask and
the SAM mask differ: the width of the vessels, the regions they mark, the thin branches? Did the clicks
land on vessels?

<details><summary>What to look for</summary>

The prototype labels <i>every</i> patch, so it covers the whole tree at patch resolution. SAM only sees
ten clicks, and it does what section 2 showed it does with clicks: it looks for the object that contains
the positive ones and avoids the negative ones. Its edges are sharper, but it can lose whole branches no
click touched, or grow into a region around a thick vessel. If a "vessel" click landed on the optic disc
or a dark spot, the mask follows it: SAM trusts its prompts. Two uses of the same information can fail in
opposite directions.

</details>

### 4.3 Rung 2: scribbles

The annotator from section 3 again, with a different instruction: "don't trace everything; draw a few
strokes along some vessels and a few on clear background, in each of five images, then stop". Every
other pixel is unlabelled, and this time the file says so (value 0), so the loss can skip it. This cell
shows the strokes on one training image: vessel strokes in teal, background strokes in grey, thickened
for the figure only.

In [ ]:
scrib_train = [load_item(SPECIALISED, it) for it in SPECIALISED["fit"][:N_TRAIN_SPEC]]
labelled = float(np.mean([(it["scribbles"] > 0)[region(it)].mean() for it in scrib_train]))
scrib_meta = SPECIALISED["fit"][:N_TRAIN_SPEC]
n_strokes = sum(m.get("n_fg_strokes", 0) + m.get("n_bg_strokes", 0) for m in scrib_meta)
print(
    f"{len(scrib_train)} training images · {n_strokes} strokes · {labelled:.2%} of the fundus pixels "
    f"labelled · ≈ {effort_minutes(n_strokes=n_strokes):.1f} min at "
    f"{SPECIALISED['annotation_effort']['seconds_per_stroke']} s per stroke"
)
ex = scrib_train[0]
thick = {v: ndi.binary_dilation(ex["scribbles"] == v, iterations=2) for v in (1, 2)}
strokes = overlay(overlay(ex["image"], thick[2], THEME["background"], 0.9), thick[1], THEME["scribble"], 0.9)
show_masks(
    [
        (ex["name"], ex["image"]),
        ("scribbles: vessel · background", strokes),
        ("ground truth · evaluation only", overlay(ex["image"], ex["mask"], THEME["truth"])),
    ],
    width=3.0,
)

<table width="100%" bgcolor="#EDE7F6"><tr><td><font color="#311B92">📘 <b>Theory: partial
cross-entropy.</b> The loss of section 2 sums a cross-entropy term over every pixel. The partial version
sums it only over the pixels a person labelled: a vessel stroke says "1", a background stroke says "0",
and an unlabelled pixel contributes nothing at all, neither "vessel" nor "background". In code that is a
target of <code>-1</code>, which <code>train_head</code> already skips (the border of the fundus has been
skipped that way since section 2). The balanced weighting is computed over the labelled pixels only.
Compare with section 3: there the unpainted pixels were targets of <code>0</code>.</font></td></tr></table>

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your turn (optional, 3 min):
choose which pixels to ignore.</b> Fill <code>my_scribble_targets</code>. It turns a scribble map
(0 unlabelled, 1 vessel stroke, 2 background stroke) into training targets: 1, 0, or −1 for "ignore".
Pixels outside <code>region</code> are always −1. With <code>bg_beyond_px=None</code>, every unlabelled
pixel is ignored. With a number, you decide that unlabelled pixels farther than that many pixels from any
vessel stroke count as background (<code>ndi.distance_transform_edt(scribbles != 1)</code> gives each
pixel's distance to the nearest vessel stroke). The cell checks your version on a 6 × 6 play map with both
settings and continues with ours if you skip it. You will use the second setting in 4.5.
</font></td></tr></table>

In [ ]:
# ✍️ YOUR CODE
def my_scribble_targets(scribbles: np.ndarray, region: np.ndarray, bg_beyond_px: float | None = None):
    """Targets (int8) from a scribble map: 1 vessel stroke, 0 background stroke, -1 ignored. Outside
    `region`: -1. If bg_beyond_px is a number, unlabelled pixels farther than bg_beyond_px from every vessel
    stroke become 0 (background); the others stay -1."""
    ...


def _ref_scribble_targets(scribbles: np.ndarray, region: np.ndarray, bg_beyond_px: float | None = None):
    """Reference: strokes as given, unlabelled ignored (or background when far from every vessel stroke)."""
    targets = np.full(scribbles.shape, -1, np.int8)
    if bg_beyond_px is not None:
        targets[ndi.distance_transform_edt(scribbles != 1) > bg_beyond_px] = 0
    targets[scribbles == 2] = 0
    targets[scribbles == 1] = 1
    targets[~np.asarray(region, bool)] = -1
    return targets


_play_scrib = np.zeros((6, 6), np.uint8)
_play_scrib[1, :4] = 1  # a vessel stroke
_play_scrib[4, 1:3] = 2  # a background stroke
_play_scrib[5, 5] = 2  # a background stroke outside the region
_play_region = np.ones((6, 6), bool)
_play_region[:, 5] = False  # the last column is outside the evaluation region
scribble_targets = use(
    my_scribble_targets,
    reference=_ref_scribble_targets,
    check=lambda f: [np.asarray(f(_play_scrib, _play_region, d)).astype(int) for d in (None, 2)],
    name="scribble_targets",
)
print("play example · targets with bg_beyond_px=2 (−1 = ignored):")
print(scribble_targets(_play_scrib, _play_region, 2))

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your prediction.</b> The head
will see labels on fewer than two pixels in a hundred, and no vessel will ever be called "background".
Against the one complete mask of rung 1, will it find more of the held-out vessels, or fewer? Will it
mark more of the fundus, or less? Write two guesses.</font></td></tr></table>

This cell trains the section-2 head on the scribbles of the five training images, every unlabelled pixel
ignored, and prints the seconds it took. Not done in a minute, or on a CPU and in a hurry? Set
`TRAIN_YOURSELF = False` and run it again: it loads our checkpoint, trained the same way on all fit images.

In [ ]:
TRAIN_YOURSELF = True  # False -> load our precomputed "scribbles" head instead of training
if not isinstance(TRAIN_YOURSELF, bool):
    raise SystemExit("TRAIN_YOURSELF must be True or False.")

scrib_feats = [features_for(SPECIALISED, it, side) for it in scrib_train]
# the rung's supervision: strokes only; unlabelled pixels and the border are ignored (-1), never "background"
scrib_labels = [scribble_targets(it["scribbles"], region(it)) for it in scrib_train]
if TRAIN_YOURSELF:
    scrib_head, scrib_sec = train_head(scrib_feats, scrib_labels, HEAD_STEPS)
    src = f"trained here on {len(scrib_train)} images, {HEAD_STEPS} steps, partial cross-entropy"
else:
    scrib_head, entry = load_head(SPECIALISED, "scribbles", side)
    scrib_sec = entry["train_sec"]
    src = f"our checkpoint {entry['file']} ({len(entry['train_names'])} images)"
print(f"scribble head @ {side} px · {src} · {scrib_sec:.1f} s on {DEVICE}")
if scrib_sec > 60:
    print("ℹ️ Slow session? Set TRAIN_YOURSELF = False above to load our checkpoint next time.")

This cell scores the scribble head on the evaluation images with the same five numbers as the other rungs.

In [ ]:
t0 = time.monotonic()
ladder_preds["scribbles"] = [
    predict_head(scrib_head, features_for(SPECIALISED, it, side), it["mask"].shape) for it in SPEC_EVAL
]
ladder_report["scribbles"] = rung_report(ladder_preds["scribbles"])
scrib_infer = round(time.monotonic() - t0, 1)
v = ladder_report["scribbles"]
print(
    f"scribble head · IoU {v['iou']:.3f} · clDice {v['cldice']:.3f} · recall {v['recall']:.3f} · precision "
    f"{v['precision']:.3f} · marks {v['marked']:.0%} · {scrib_infer} s on {DEVICE}"
)

### 4.4 The ladder, side by side

This cell puts the rungs next to each other: a table with the five numbers and the annotation each rung
needed, a bar chart (one colour per approach, kept for the rest of the notebook), and the demo image's
error maps: **violet** = vessels found, **orange** = false alarms, **green** = vessels missed.

In [ ]:
effort = {
    "zero_shot": effort_minutes(n_strokes=len(SPEC_EVAL)),  # one box per image you segment
    "one_shot": effort_minutes(n_full_masks=1),
    "one_shot_sam": effort_minutes(n_full_masks=1),
    "scribbles": effort_minutes(n_strokes=n_strokes),
}
effort_text = {
    "zero_shot": f"1 box per image ({len(SPEC_EVAL)} here)",
    "one_shot": "1 complete mask",
    "one_shot_sam": "1 complete mask",
    "scribbles": f"{n_strokes} strokes, {len(scrib_train)} images",
}
RESULTS["ladder"] = {k: ladder_report[k]["iou"] for k in ("zero_shot", "one_shot", "scribbles")}  # SPEC §9
RESULTS["ladder_detail"] = ladder_report
RESULTS["ladder_effort_min"] = {k: round(v, 2) for k, v in effort.items()}
RESULTS["scribbles_head"] = ladder_report["scribbles"] | {
    "long_side": side,
    "n_train": len(scrib_train),
    "n_strokes": n_strokes,
    "labelled_fraction": labelled,
    "train_sec": round(float(scrib_sec), 2),
    "infer_sec": scrib_infer,
}
RESULTS["one_shot"] = ladder_report["one_shot"] | {"image": shot["name"], "infer_sec": proto_sec}

COLUMNS = f"{'IoU':>6}{'clDice':>8}{'recall':>8}{'precis.':>8}{'marked':>8}"
print(f"{'approach':<36}{COLUMNS}   annotation (≈ minutes)")
for k, v in ladder_report.items():
    print(
        f"{APPROACH_NAMES[k]:<36}{v['iou']:>6.3f}{v['cldice']:>8.3f}{v['recall']:>8.3f}{v['precision']:>8.3f}"
        f"{v['marked']:>8.0%}   {effort_text[k]} (≈ {effort[k]:.1f})"
    )
print(f"{'everything is foreground':<36}{RESULTS['specialised_everything_iou']:>6.3f}")

metrics = ["iou", "cldice", "recall", "precision"]
x = np.arange(len(metrics))
width = 0.8 / len(ladder_report)
fig, ax = plt.subplots(figsize=(7.2, 3.2))
for n, (k, v) in enumerate(ladder_report.items()):
    shift = (n - (len(ladder_report) - 1) / 2) * width
    ax.bar(x + shift, [v[m] for m in metrics], width, color=APPROACH_COLOURS[k], label=APPROACH_NAMES[k])
ax.axhline(
    RESULTS["specialised_everything_iou"], color=THEME["background"], ls="--", label='IoU of "everything"'
)
ax.set_xticks(x, ["IoU", "clDice", "recall", "precision"])
ax.set_ylim(0, 1)
ax.set_title("The adaptation ladder: different information, different failures")
ax.legend(fontsize=7, frameon=False, loc="upper left", ncol=2)
plt.tight_layout()
plt.show()

show_masks(
    [
        (
            f"{APPROACH_NAMES[k].split(':')[0]} · IoU {score(ladder_preds[k][k_demo], demo)['iou']:.2f}",
            error_map(demo["image"], ladder_preds[k][k_demo], demo),
        )
        for k in ladder_preds
    ]
    + [("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"]))],
    width=2.6,
)

**Q16.** 🟡 Read the error maps rung by rung. Which rung misses the most vessels (green), which invents the
most (orange), and is that the same rung that has the lowest IoU? If you had to pick one rung for a
screening tool that must not miss a vessel, and one for a tool whose every mark must be right, would you
pick the same?

<details><summary>What to look for</summary>

The rungs do not line up on a single "quality" axis. SAM's box marks a region or the whole eye: high
recall, many false alarms, an IoU near the grey line. The prototype labels by patch, so it draws vessels
as wide as a square and confuses dark, textured background with vessel. SAM from prototype clicks is
sharper but only covers what its clicks touched. The scribble head saw the fewest labelled pixels, yet
from several images, and it was never told a vessel was background, so its errors are not the section-3
errors: look at its recall next to the naive head's. Which rung is "best" depends on whether missing or
inventing a vessel costs you more, and on the minutes in the last column. On your run the order may differ
from your neighbour's: that is the point of reading the maps, not only the IoU.

</details>

### 4.5 Your choice: which unlabelled pixels to ignore

Ignoring every unlabelled pixel is the safe choice, and also a wasteful one: more than 98 % of the fundus
says nothing. A tempting shortcut is "far from every vessel stroke, it's surely background". That is the
second setting of your `scribble_targets`.

<table width="100%" bgcolor="#FDECEA"><tr><td><font color="#8B1A10">⚠️ <b>Pitfall: "surely background"
is a guess.</b> The strokes cover only some of the vessel segments. A vessel nobody scribbled can be
far from every stroke, and the shortcut then labels it background: the section-3 mistake, back through
the side door. The next cell measures how much of the true vessel area your rule mislabels, on the
training images, using the hidden ground truth for that diagnosis only, never for training.
</font></td></tr></table>

<table width="100%" bgcolor="#F3E5F5"><tr><td><font color="#4A148C">❓ <b>Question.</b> Run the next cell,
then try a few distances.</font></td></tr></table>

**Q17.** 🟡 Set a distance and train a second scribble head with it. How many true vessel pixels did your
rule call background? Did recall, precision and IoU on the held-out images move, and in which direction?
Is there a distance that helps without hurting?

```python
BG_BEYOND_PX = 40   # try 20, 80, None (= ignore every unlabelled pixel)
```

This cell turns the scribbles into targets with your distance, trains the same head on them (seconds),
and prints its numbers next to the scribble head of 4.3, which ignored every unlabelled pixel.

In [ ]:
# Q17 — scribbles plus "far from every vessel stroke = background"
BG_BEYOND_PX = 40  # pixels, on the 1024 px crops; try 20, 80, None (= ignore every unlabelled pixel)
if BG_BEYOND_PX is not None and (not isinstance(BG_BEYOND_PX, (int, float)) or BG_BEYOND_PX <= 0):
    raise SystemExit("BG_BEYOND_PX must be a positive number of pixels, or None.")
bg_px = BG_BEYOND_PX / 4 if (SMOKE and BG_BEYOND_PX) else BG_BEYOND_PX  # smoke thumbnails are 4× smaller
far_labels = [scribble_targets(it["scribbles"], region(it), bg_px) for it in scrib_train]
# diagnosis only: how much of the true vessel area the rule called background (never used for training)
far_pairs = list(zip(far_labels, scrib_train, strict=True))
wrong = float(np.mean([((t == 0) & it["mask"]).sum() / max(it["mask"].sum(), 1) for t, it in far_pairs]))
far_labelled = float(np.mean([(t >= 0)[region(it)].mean() for t, it in far_pairs]))
far_head, far_sec = train_head(scrib_feats, far_labels, HEAD_STEPS)
far_preds = [
    predict_head(far_head, features_for(SPECIALISED, it, side), it["mask"].shape) for it in SPEC_EVAL
]
far_report = rung_report(far_preds)
RESULTS["scribbles_bg_beyond"] = far_report | {"bg_beyond_px": BG_BEYOND_PX, "vessel_called_bg": wrong}
print(
    f"rule: unlabelled pixels > {BG_BEYOND_PX} px from every vessel stroke = background · labelled "
    f"{far_labelled:.0%} of the fundus · {wrong:.1%} of the true vessel pixels called background · "
    f"trained in {far_sec:.1f} s"
)
print(f"{'scribble head':<28}{COLUMNS}")
for name, v in {"ignore every unlabelled": ladder_report["scribbles"], "your distance": far_report}.items():
    nums = f"{v['iou']:>6.3f}{v['cldice']:>8.3f}{v['recall']:>8.3f}{v['precision']:>8.3f}"
    print(f"{name:<28}{nums}{v['marked']:>8.0%}")

<details><summary>What to look for</summary>

A large distance adds a lot of background that really is background, and a little that is not; a small
distance adds more of both. More "background" targets push the head towards marking less, so watch recall
fall, the direction of section 3, and check whether precision rose enough to pay for it. Whether IoU
improves depends on which effect wins on your images, and the line "vessel pixels called background"
tells you the price. The honest lesson is not a
number: every rule that fills in unlabelled pixels is an assumption about where the annotator did not
look, and it is only as good as that assumption. Section 5 makes "unknown" explicit in the loss; section 6
fills unlabelled pixels with a second model's guesses instead of a distance.

</details>

<table width="100%" bgcolor="#E8EAF6"><tr><td><font color="#1A237E">🔀 <b>Across approaches.</b> Three
rungs, three kinds of information. Zero-shot: a prompt, so SAM's idea of an object decides. One-shot: one
complete example, so <i>which</i> example you annotated decides, and the patch resolution limits the
mask. Scribbles: many cheap, partial hints from several images, with "unknown" kept unknown. The rungs
differ more in <i>how</i> they fail than in a single number, and the cheapest rung in minutes is not the
worst. Keep the colours: these approaches come back in the comparison table of section 8.
</font></td></tr></table>

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Try it on your image (optional,
2 min).</b> The one-shot idea works from clicks too: a few clicks on the thing you want and a few on
what you do not want give two prototypes, and every similar patch of the image is marked. Upload a photo
with the folder icon on the left, put its path in <code>MY_IMAGE</code>, and some <code>(x, y)</code>
clicks in <code>MY_FG_CLICKS</code> and <code>MY_BG_CLICKS</code>. Left empty, it runs on the demo image
with clicks along its scribbles. Section 9 runs the whole ladder on your own scribbles.
</font></td></tr></table>

This cell computes DINOv3 features of your image (or the demo image), builds the two prototypes from the
patches under your clicks, and shows the similarity map and the mask: one model pass, no training.

In [ ]:
MY_IMAGE = ""  # e.g. "/content/my_photo.jpg"; empty = the demo image
MY_FG_CLICKS = []  # e.g. [(120, 80), (300, 210)] on the thing you want
MY_BG_CLICKS = []  # e.g. [(20, 20)] on what you do not want; empty = every other patch
if not isinstance(MY_IMAGE, str) or not isinstance(MY_FG_CLICKS, list) or not isinstance(MY_BG_CLICKS, list):
    raise SystemExit('MY_IMAGE must be a path in quotes (or "") and the clicks lists of (x, y) pairs.')
if MY_IMAGE and not Path(MY_IMAGE).is_file():
    raise SystemExit(f"{MY_IMAGE} not found. Upload it with the folder icon on the left and copy its path.")
if MY_IMAGE and not MY_FG_CLICKS:
    raise SystemExit("Add at least one (x, y) click on the thing you want to MY_FG_CLICKS.")
my_img = np.array(Image.open(MY_IMAGE).convert("RGB")) if MY_IMAGE else demo["image"]
my_h, my_w = my_img.shape[:2]
if MY_IMAGE:
    my_fg = [[float(x), float(y)] for x, y in MY_FG_CLICKS]
    my_bg = [[float(x), float(y)] for x, y in MY_BG_CLICKS]
else:
    ys, xs = np.nonzero(demo["scribbles"] == 2)
    picks = np.linspace(0, len(ys) - 1, 3).astype(int) if len(ys) else []
    my_fg, my_bg = scribble_points(demo, 3), [[float(xs[k]), float(ys[k])] for k in picks]
if not all(0 <= x < my_w and 0 <= y < my_h for x, y in my_fg + my_bg):
    raise SystemExit(f"every click must lie inside the image: 0 ≤ x < {my_w}, 0 ≤ y < {my_h}.")
my_feat = dino.features(my_img, min(side, max(my_h, my_w)))
gh, gw = my_feat.shape[1:]
w_fg, w_bg = torch.zeros(gh, gw, device=DEVICE), torch.zeros(gh, gw, device=DEVICE)
for weights, pts in ((w_fg, my_fg), (w_bg, my_bg)):
    for x, y in pts:
        weights[min(int(y * gh / my_h), gh - 1), min(int(x * gw / my_w), gw - 1)] = 1.0
if not my_bg:
    w_bg = 1.0 - w_fg  # no background clicks: every other patch is the background prototype
my_score = prototype_score(my_feat, prototypes(my_feat, w_fg, w_bg))
my_mask = score_to_mask(my_score, (my_h, my_w))
show_clicks(
    [
        (MY_IMAGE or f"demo · {demo['name']} · clicks on the thing", my_img, my_fg),
        ("clicks on what you do not want", my_img, my_bg),
        ("similarity: thing − not", heat_rgb(my_score, (my_h, my_w)), None),
        (f"prototype mask · marks {my_mask.mean():.0%}", overlay(my_img, my_mask, THEME["fails"]), my_fg),
    ],
    width=2.8,
)

## 5. Path 1 · Learn from less

**Objective:** treat missing labels as unknown; write the masked loss, then 🔴 a PU-inspired loss.

Back to the half-finished label file of section 3. The naive head failed because we passed the file to
the loss as it was, and the file calls every unpainted pixel "background". Path 1 keeps the same file,
the same features and the same head, and changes only what the loss is told about the unpainted pixels.
Two answers. **Masked:** "unknown", so the loss skips them; the only background it hears about is a few
strokes someone drew on clear background. **🔴 PU:** "unlabelled, and about this share of them is
vessel", so the loss uses every pixel but corrects for the vessels hiding among them.

<table width="100%" bgcolor="#EDE7F6"><tr><td><font color="#311B92">📘 <b>Theory: learning from positives
and unknowns.</b> A <b>masked loss</b> is any loss with a mask that removes pixels from it: an ignored
pixel adds nothing to the loss and sends no gradient. The partial cross-entropy of section 4 is one;
here we write it ourselves. Ignoring pixels is safe only if some known background is left, otherwise
"everything is vessel" scores a perfect zero. <b>Positive-unlabelled (PU) learning</b> starts from the
opposite end: some positives are labelled, everything else is unlabelled, and no negative is known. If
the labelled positives are a random sample of all positives and you know (or guess) the share of
positives, the <i>prior</i>, you can still estimate the loss you would get with complete labels (Elkan
&amp; Noto, KDD 2008; Kiryo et al., "Positive-Unlabeled Learning with Non-Negative Risk Estimator",
NeurIPS 2017, <a href="https://arxiv.org/abs/1703.00593">arXiv:1703.00593</a>). We use their
non-negative version on a single pixel classifier, which is why we call it PU-<i>inspired</i>.
</font></td></tr></table>

This helper cell defines a training loop that takes the loss as an argument (the same linear head,
features and steps as every head so far), a report with the numbers of section 3 plus the held-out ones
of section 4, and one colour for each head of this path.

In [ ]:
# @title Path 1 helpers: train with any loss, score a head { display-mode: "form" }
APPROACH_COLOURS |= {
    "naive": "#BF360C",
    "naive_balanced": "#FF8A65",
    "masked": "#F9A825",
    "pu": "#6D4C41",
}
APPROACH_NAMES |= {
    "naive": "naive: missing = background",
    "naive_balanced": "naive, balanced loss (3.4)",
    "masked": "masked: missing = unknown",
    "pu": "PU: missing = unlabelled",
}


def train_head_with(
    feats: list[torch.Tensor], labels: list[np.ndarray], loss_fn, steps: int, lr: float = 0.05
):
    """The linear head of `train_head`, trained with `loss_fn(logits, labels, ignore_mask)` on every pixel of
    every training image at once. Labels per pixel: 1 vessel, 0 background (or unlabelled), -1 ignored.
    Returns the head's tensors and the training seconds."""
    torch.manual_seed(SEED)
    allf = torch.cat([f.flatten(1) for f in feats], 1)
    mu, sd = allf.mean(1), allf.std(1) + 1e-6
    feats = [(f - mu.view(-1, 1, 1)) / sd.view(-1, 1, 1) for f in feats]
    lab = torch.cat([torch.from_numpy(np.asarray(x, dtype=np.int8)).flatten() for x in labels]).to(DEVICE)
    target, ignore = (lab == 1).float(), lab < 0
    shapes = [np.asarray(x).shape for x in labels]
    head = torch.nn.Conv2d(feats[0].shape[0], 1, 1).to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=lr)
    t0 = time.monotonic()
    for _ in range(steps):
        opt.zero_grad()
        logits = torch.cat(
            [
                F.interpolate(head(f[None]), size=s, mode="bilinear", align_corners=False).flatten()
                for f, s in zip(feats, shapes, strict=True)
            ]
        )
        loss_fn(logits, target, ignore).backward()
        opt.step()
    tensors = {
        "weight": head.weight.detach().cpu(),
        "bias": head.bias.detach().cpu(),
        "feat_mean": mu.cpu(),
        "feat_std": sd.cpu(),
    }
    return tensors, time.monotonic() - t0


def path_report(tensors: dict) -> dict:
    """Recall on the hidden and visible vessels of the training images (section 3) and the held-out
    numbers of section 4 (IoU, clDice, recall, precision, marked share), plus the predictions."""
    fit_preds = [
        predict_head(tensors, f, it["mask"].shape) for f, it in zip(miss_feats, miss_train, strict=True)
    ]
    held_preds = [
        predict_head(tensors, features_for(SPECIALISED, it, side), it["mask"].shape) for it in SPEC_EVAL
    ]
    out = {
        "recall_hidden_fit": part_recall(fit_preds, miss_train, "hidden"),
        "recall_visible_fit": part_recall(fit_preds, miss_train, "visible"),
    }
    return out | rung_report(held_preds) | {"fit_preds": fit_preds, "held_preds": held_preds}


def print_reports(reports: dict) -> None:
    """One line per head: recall on the hidden / visible training vessels, then the held-out numbers."""
    print(f"{'head':<34}{'recall: hidden':>15}{'visible':>9}   held-out IoU  recall  precision  marked")
    for name, r in reports.items():
        print(
            f"{name:<34}{r['recall_hidden_fit']:>15.3f}{r['recall_visible_fit']:>9.3f}   {r['iou']:>12.3f}"
            f"{r['recall']:>8.3f}{r['precision']:>11.3f}{r['marked']:>8.0%}"
        )


def public(r: dict) -> dict:
    """A report without its prediction arrays, for RESULTS."""
    return {k: v for k, v in r.items() if not k.endswith("_preds")}

### 5.1 Three ways to read the same label file

This cell builds what each recipe tells the loss for the five training images of section 3: the same
visible vessels every time, and three different answers for the rest. The masked recipe also gets the
3–5 background strokes of section 4 on the same images (seconds of work per image), because it needs
some known background. No ground truth goes in: only the visible vessels, the strokes and the fundus
border. The figure shows one image, with the background strokes drawn thicker so you can see them.

In [ ]:
bg_strokes = [(it["scribbles"] == 2) & region(it) & ~it["visible"] for it in miss_train]
# masked: vessel where painted, background only on the strokes, everything else unknown (-1)
masked_labels = [
    np.where(it["visible"] & region(it), 1, np.where(bg, 0, -1)).astype(np.int8)
    for it, bg in zip(miss_train, bg_strokes, strict=True)
]
# PU: vessel where painted, "unlabelled" (0) everywhere else inside the fundus
pu_labels = [np.where(region(it), it["visible"], -1).astype(np.int8) for it in miss_train]
known = {
    "naive": np.mean([(lab >= 0).mean() for lab in naive_labels]),
    "masked": np.mean([(lab >= 0).mean() for lab in masked_labels]),
}
n_bg_strokes = sum(m.get("n_bg_strokes", 0) for m in SPECIALISED["fit"][: len(miss_train)])
print(
    f"pixels the loss hears about · naive {known['naive']:.1%} (all called background unless painted) · "
    f"masked {known['masked']:.1%} ({n_bg_strokes} background strokes over {len(miss_train)} images) · "
    "PU: all of them, as vessel or unlabelled"
)

ex, k_ex = miss_train[0], 0
grey = ndi.binary_dilation(bg_strokes[k_ex], iterations=4)  # thicker, for display only
known_bg = naive_labels[k_ex] == 0
show_masks(
    [
        (ex["name"], ex["image"]),
        (
            "naive: vessel / background",
            overlay(overlay(ex["image"], known_bg, THEME["background"]), ex["visible"], THEME["scribble"]),
        ),
        (
            "masked: vessel / strokes / unknown",
            overlay(overlay(ex["image"], grey, THEME["background"]), ex["visible"], THEME["scribble"]),
        ),
        ("PU: vessel / unlabelled", overlay(ex["image"], ex["visible"], THEME["scribble"])),
    ],
    width=2.8,
)

Teal = painted vessel, grey = a background target, no colour = what the recipe leaves open.

**Q18.** 🟡 The masked recipe tells the loss about only a small share of the pixels, and almost all of
them are vessel. Which pixels is the head free to call vessel without any penalty? What would the loss do
if you also removed the background strokes?

<details><summary>What to look for</summary>

Every pixel without a colour in the third panel: the hidden vessels, which is the point, but also all the
background between the strokes, including the background right next to every vessel. The strokes sit on
clear background away from the vessels, so they say little about where a vessel ends. Without them the
loss only ever hears "vessel", and a head that calls every pixel vessel scores a perfect zero: the
trivial solution. A masked loss is only as good as the known background left in it.

</details>

### 5.2 Write the masked loss

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your turn (optional, 4 min).</b>
Fill <code>masked_loss</code>. In: <code>logits</code> (float tensor, any shape), <code>labels</code>
(same shape, 1.0 = vessel, 0.0 = background) and <code>ignore_mask</code> (same shape, bool, True =
unknown). Out: one scalar tensor. It is the balanced loss of every head so far, with one change: ignored
pixels count for nothing. Take the binary cross-entropy of each pixel
(<code>F.binary_cross_entropy_with_logits(..., reduction="none")</code>), average it over the known
vessel pixels and over the known background pixels separately, and return the mean of the two averages.
The cell checks your version on a 2 × 3 play example and continues with ours if you skip it.
</font></td></tr></table>

In [ ]:
# ✍️ YOUR CODE
def masked_loss(logits: torch.Tensor, labels: torch.Tensor, ignore_mask: torch.Tensor) -> torch.Tensor:
    """Balanced binary cross-entropy over the pixels where `ignore_mask` is False: the mean loss on the
    known vessel pixels and the mean loss on the known background pixels, averaged."""
    ...


def _ref_masked_loss(logits: torch.Tensor, labels: torch.Tensor, ignore_mask: torch.Tensor) -> torch.Tensor:
    """Reference: per-pixel BCE, then a mean per known class (an empty class counts as 0)."""
    bce = F.binary_cross_entropy_with_logits(logits, labels.float(), reduction="none")
    pos, neg = ~ignore_mask & (labels == 1), ~ignore_mask & (labels == 0)
    return 0.5 * bce[pos].sum() / pos.sum().clamp(min=1) + 0.5 * bce[neg].sum() / neg.sum().clamp(min=1)


_play_logits = torch.tensor([[2.0, -1.0, 0.5], [-2.0, 3.0, 0.0]])
_play_labels = torch.tensor([[1.0, 0.0, 1.0], [0.0, 0.0, 1.0]])
_play_ignore = torch.tensor([[False, False, True], [False, True, False]])  # two unknown pixels
masked_loss_fn = use(
    masked_loss,
    reference=_ref_masked_loss,
    check=lambda f: [
        round(float(f(_play_logits, _play_labels, ig)), 4)
        for ig in (_play_ignore, torch.zeros(2, 3, dtype=bool))
    ],
    name="masked_loss",
)
_with, _without = (
    float(masked_loss_fn(_play_logits, _play_labels, ig))
    for ig in (_play_ignore, torch.zeros_like(_play_ignore))
)
print(f"play example · loss {_with:.4f} with two pixels unknown, {_without:.4f} with none")

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your prediction.</b> Same
label file as the naive head, but the unpainted pixels are now unknown. On the hidden vessels of the
training images, will its recall be closer to the naive head's or to the section-2 head's (full masks)?
And on new images, will it mark more of the fundus than the naive head, or less? Write two guesses.
</font></td></tr></table>

This cell trains the head with your masked loss (or ours) on the masked labels of 5.1 and prints the
seconds it took. Not done in a minute, or on a CPU and in a hurry? Set `TRAIN_YOURSELF = False` and run it
again: it loads our "masked" checkpoint, which was trained with a stricter rule (the text it prints says
how).

In [ ]:
TRAIN_YOURSELF = True  # False -> load our precomputed "masked" head instead of training
if not isinstance(TRAIN_YOURSELF, bool):
    raise SystemExit("TRAIN_YOURSELF must be True or False.")

# the supervision of this path: painted vessels + background strokes; the hidden truth never enters (L-005)
if TRAIN_YOURSELF:
    masked_head, masked_sec = train_head_with(miss_feats, masked_labels, masked_loss_fn, HEAD_STEPS)
    src = f"trained here on {len(miss_train)} images, {HEAD_STEPS} steps, masked loss"
else:
    masked_head, entry = load_head(SPECIALISED, "masked", side)
    masked_sec = entry["train_sec"]
    src = (
        f"our checkpoint {entry['file']} ({len(entry['train_names'])} images). ⚠️ It was told exactly which "
        "pixels next to the hidden vessels are unknown and that the rest is background: a rule that needs "
        "the hidden truth, so read its numbers as an upper bound for this path"
    )
print(f"masked head @ {side} px · {src} · {masked_sec:.1f} s on {DEVICE}")
if masked_sec > 60:
    print("ℹ️ Slow session? Set TRAIN_YOURSELF = False above to load our checkpoint next time.")

This cell scores the masked head next to the heads of sections 2 and 3, on the same groups as in 3.3
(hidden and visible vessels of the training images) and on the held-out images, then plots the
recalls and precision.

In [ ]:
t0 = time.monotonic()
path_reports = {APPROACH_NAMES["masked"]: path_report(masked_head)}
masked_infer = round(time.monotonic() - t0, 1)
for key, tensors in (("naive", naive_head), ("naive_balanced", balanced_head)):
    path_reports[APPROACH_NAMES[key]] = path_report(tensors)
path_reports["full masks (section 2)"] = path_report(full_head)
masked = path_reports[APPROACH_NAMES["masked"]]

RESULTS["masked_recall_hidden_fg"] = masked["recall_hidden_fit"]  # same definition as section 3 (SPEC §9)
RESULTS["masked_head"] = public(masked) | {
    "long_side": side,
    "n_train": len(miss_train),
    "known_share": float(known["masked"]),
    "n_bg_strokes": int(n_bg_strokes),
    "trained_here": TRAIN_YOURSELF,
    "train_sec": round(float(masked_sec), 2),
    "infer_sec": masked_infer,
}
print_reports(path_reports)
print(f"section 3 recorded for the naive head · hidden-vessel recall {RESULTS['naive_recall_hidden_fg']:.3f}")

PATH_COLOURS = {
    APPROACH_NAMES["naive"]: APPROACH_COLOURS["naive"],
    APPROACH_NAMES["naive_balanced"]: APPROACH_COLOURS["naive_balanced"],
    APPROACH_NAMES["masked"]: APPROACH_COLOURS["masked"],
    "full masks (section 2)": THEME["works"],
}
BARS = {
    "recall_hidden_fit": "hidden vessels\n(training images)",
    "recall": "recall\n(held-out)",
    "precision": "precision\n(held-out)",
}


def path_chart(names: list[str]) -> None:
    """Grouped bars: recall on the hidden training vessels, held-out recall and precision, per head."""
    x, width = np.arange(len(BARS)), 0.8 / len(names)
    fig, ax = plt.subplots(figsize=(7.2, 3))
    for j, name in enumerate(names):
        shift = (j - (len(names) - 1) / 2) * width
        ax.bar(x + shift, [path_reports[name][b] for b in BARS], width, color=PATH_COLOURS[name], label=name)
    ax.set_xticks(x, list(BARS.values()), fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_title("Path 1: what each head finds, and how much of what it marks is right")
    ax.legend(fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.show()


path_chart(list(PATH_COLOURS))

<table width="100%" bgcolor="#F3E5F5"><tr><td><font color="#4A148C">❓ <b>Question.</b> Run the next
cell: it shows the hidden vessels each head found on a training image, then the error maps of section 4
on the held-out demo image (violet found, orange invented, green missed).</font></td></tr></table>

**Q19.** 🟡 Which hidden vessels does the masked head recover that the naive head missed? On the
held-out image, what did each head pay for what it found: compare the orange. Try another training image:

```python
LOOK_IMAGE = 0   # try 1, 2, ...
```

This cell draws both figures from the predictions you already have: no new training, no model calls.

In [ ]:
# Q19 — hidden vessels found on a training image, then error maps on the held-out demo image
LOOK_IMAGE = 0  # try 1, 2, ...
if not isinstance(LOOK_IMAGE, int) or not 0 <= LOOK_IMAGE < len(miss_train):
    raise SystemExit(f"LOOK_IMAGE must be a whole number from 0 to {len(miss_train) - 1}.")
it = miss_train[LOOK_IMAGE]
LOOK = [APPROACH_NAMES["naive"], APPROACH_NAMES["masked"]]
look_preds = {name: path_reports[name]["fit_preds"][LOOK_IMAGE] for name in LOOK}
show_masks(
    [
        (
            f"{name.split(':')[0]} · hidden found {hidden_recall(pred, it['hidden']):.0%}",
            two_colour(it["image"], it["hidden"] & pred, it["hidden"] & ~pred),
        )
        for name, pred in look_preds.items()
    ],
    width=3.4,
)
k_demo = next(i for i, e in enumerate(SPEC_EVAL) if e is demo)
show_masks(
    [
        (
            f"{name.split(':')[0]} · IoU {score(path_reports[name]['held_preds'][k_demo], demo)['iou']:.2f}",
            error_map(demo["image"], path_reports[name]["held_preds"][k_demo], demo),
        )
        for name in LOOK
    ],
    width=3.4,
)

<details><summary>What to look for</summary>

The masked head was never told that a hidden vessel is background, so it treats a vessel-looking patch as
a vessel wherever it is: it finds most of the hidden vessels, the ones the naive head dropped. The price
shows on the held-out image. The only background the loss heard about is a handful of strokes far from
any vessel, so nothing told it where a vessel ends: the area around each vessel, and sometimes darker
background, turns orange. Compare the "marked" column: the naive head is too stingy, the masked head is
often too generous, and IoU alone may rank them either way. Both failures come from the same place, what
the loss was told about the unpainted pixels, in opposite directions.

</details>

<table width="100%" bgcolor="#FDECEA"><tr><td><font color="#8B1A10">⚠️ <b>Pitfall: "ignore" is not
free.</b> Every pixel you ignore is a pixel the model may call anything. A masked loss with almost no
known background drifts towards "everything is foreground", and the training loss still looks perfect.
Watch the share of the image the model marks, and give the loss some background close to the objects,
not only easy background far away.</font></td></tr></table>

### 5.3 🔴 Optional: a PU-inspired loss

The masked head needed background strokes; the PU head needs none. It reads every unpainted pixel as
<i>unlabelled</i>: a mix of background and the vessels nobody painted. If you know roughly what share of
the vessels was painted, you know roughly how much vessel hides in that mix, and you can subtract it.

<table width="100%" bgcolor="#EDE7F6"><tr><td><font color="#311B92">📘 <b>Theory: the PU trick in three
lines.</b> Let the <b>prior</b> π be the share of vessel pixels in the fundus. Painted share ÷ share of
the vessels that was painted gives an estimate; we read the second number from the annotator ("I did
about half", here the dataset config). (1) Loss on the true vessels: the painted ones, target "vessel".
(2) Every pixel, target "background", mixes π × (vessels) + (1 − π) × (background); so the loss on the
true background is (every pixel as background − π × painted vessels as background) ÷ (1 − π). (3)
Average (1) and (2), the balanced loss again. That estimate can dip below zero when the head overfits the
few painted vessels; the non-negative trick of Kiryo et al. clips it at zero. It assumes the painted
vessels are a random sample of all vessels, which is true here because we hid segments at random.
</font></td></tr></table>

**Q20.** 🔴 The prior rests on one number from the annotator, `LABELLED_SHARE`. Read `pu_loss` in the next
cell, run it, then pretend the annotator was wrong: train again with a smaller and a larger share. Which
way does the head move on recall and on the marked share?

```python
LABELLED_SHARE = 0.3   # try 0.3, 0.5, 0.8
```

This cell defines the PU loss, estimates the prior from the painted vessels and `LABELLED_SHARE`, trains
the head with it on the PU labels of 5.1 (seconds) and prints its numbers next to the other heads.

In [ ]:
# Q20 — a PU-inspired loss: painted vessels + unlabelled pixels + one guess about the annotator
LABELLED_SHARE = 1 - MISSING["hidden_fraction"]  # the annotator's "I did about half"; try 0.3, 0.8
if not isinstance(LABELLED_SHARE, (int, float)) or not 0 < LABELLED_SHARE <= 1:
    raise SystemExit("LABELLED_SHARE must be a number above 0 and at most 1, e.g. 0.5.")


def pu_loss(
    logits: torch.Tensor, labels: torch.Tensor, ignore_mask: torch.Tensor, prior: float
) -> torch.Tensor:
    """Balanced non-negative PU loss: labels 1 = painted vessel, 0 = unlabelled; `prior` = vessel share."""
    z, y = logits[~ignore_mask], labels[~ignore_mask]
    painted = z[y == 1]
    vessel_loss = F.softplus(-painted).mean()  # (1) painted vessels, target "vessel"
    all_as_bg = F.softplus(z).mean()  # every pixel, target "background"
    painted_as_bg = F.softplus(painted).mean()  # painted vessels, target "background"
    bg_loss = (all_as_bg - prior * painted_as_bg) / (1 - prior)  # (2) the true background, estimated
    return 0.5 * vessel_loss + 0.5 * bg_loss.clamp(min=0)  # (3) balanced, clipped at zero


painted_share = sum(int((lab == 1).sum()) for lab in pu_labels) / sum(
    int((lab >= 0).sum()) for lab in pu_labels
)
prior = painted_share / LABELLED_SHARE
if not 0 < prior < 1:
    raise SystemExit(
        f"painted share {painted_share:.3f} ÷ LABELLED_SHARE {LABELLED_SHARE} gives a prior of "
        f"{prior:.2f}, which must lie between 0 and 1: choose a larger LABELLED_SHARE."
    )
pu_head, pu_sec = train_head_with(
    miss_feats, pu_labels, lambda z, y, ig: pu_loss(z, y, ig, prior), HEAD_STEPS
)
t0 = time.monotonic()
path_reports[APPROACH_NAMES["pu"]] = pu = path_report(pu_head)
PATH_COLOURS[APPROACH_NAMES["pu"]] = APPROACH_COLOURS["pu"]
RESULTS["pu_recall_hidden_fg"] = pu["recall_hidden_fit"]
RESULTS["pu_head"] = public(pu) | {
    "long_side": side,
    "n_train": len(miss_train),
    "labelled_share": float(LABELLED_SHARE),
    "prior": float(prior),
    "train_sec": round(float(pu_sec), 2),
    "infer_sec": round(time.monotonic() - t0, 1),
}
print(
    f"PU head · painted {painted_share:.1%} of the fundus ÷ labelled share {LABELLED_SHARE:.2f} → prior "
    f"{prior:.1%} vessel · trained in {pu_sec:.1f} s on {DEVICE}"
)
print_reports(
    {k: path_reports[k] for k in (APPROACH_NAMES["naive"], APPROACH_NAMES["masked"], APPROACH_NAMES["pu"])}
)
path_chart([APPROACH_NAMES["naive"], APPROACH_NAMES["masked"], APPROACH_NAMES["pu"]])

<details><summary>What to look for</summary>

The PU head hears about every pixel, like the naive head, but it no longer takes "unpainted" for
"background": it expects a share π of vessel in the mix and stops pushing once the mix looks like that.
Where it lands between the naive and the masked head depends on the run, so read your numbers. Compare
it with the balanced naive head of 3.4 as well: on labels hidden at random the two can land close,
because there the damage of "missing = background" is mostly a wrong proportion, and both correct it,
one by reweighting, the other by subtracting the vessels it expects in the mix. The
prior is the lever. A smaller `LABELLED_SHARE` claims more hidden vessel, a larger prior, and the head
marks more; at 1.0 it claims nothing is hidden and moves back towards the naive head. A wrong guess moves
the head in a predictable direction rather than breaking it, which is useful: if you have a small fully
labelled check set, you can pick the share that matches it.

</details>

<table width="100%" bgcolor="#E8EAF6"><tr><td><font color="#1A237E">🔀 <b>Across approaches.</b> Path 1
keeps the label file, the features and the head of section 3 and changes one thing, what the loss is told
about unpainted pixels. Naive: "background", so the head learns that vessels are background. Masked:
"unknown", so the hidden vessels come back, and the missing background lets it over-mark. PU: "a known
mix", the same pixels with a correction that rests on one guess. None of them adds information about the
hidden vessels; they only stop the loss from telling lies, or tell smaller ones. The labels are not cheap
either: half a vessel map is still hours per image. Section 6 tries to recover the missing labels with a
second model instead.</font></td></tr></table>

This cell writes the annotation effort of the three Path 1 heads for the comparison table of section 8,
with the assumptions of section 4: the painted half of each vessel map, plus seconds per background stroke.

In [ ]:
painted_masks = len(miss_train) * LABELLED_SHARE
RESULTS["path1_effort_min"] = {
    "naive": round(effort_minutes(n_full_masks=painted_masks), 2),
    "masked": round(effort_minutes(n_strokes=n_bg_strokes, n_full_masks=painted_masks), 2),
    "pu": round(effort_minutes(n_full_masks=painted_masks), 2),
}
print(
    " · ".join(
        f"{APPROACH_NAMES[k].split(':')[0]} {v:.0f} min" for k, v in RESULTS["path1_effort_min"].items()
    )
)

**Q21.** 🟡 Both fixes rest on one assumption: the hidden vessels look like the painted ones, because we
hid segments at random. Name a real annotation habit that breaks it. What would the masked head and the
PU head learn then?

<details><summary>A possible answer</summary>

Annotators skip what is hard or tedious: the thinnest capillaries, the low-contrast periphery, the
vessels inside a bleeding. Then the painted vessels are thick and clear, and the unpainted ones are thin
and faint. The masked head never hears "background" on the thin vessels, but it never hears "vessel"
either, so it learns "vessel = thick and clear" and still misses most of them. The PU head is worse off:
its correction assumes the hidden vessels look like the painted ones, so it pushes the thin ones towards
background. The PU literature calls this a violation of the "selected completely at random" assumption.
The practical fix is the one of the ⚠️ in section 3: keep a small fully labelled set and check the recall
on the kind of structure your annotators skip.

</details>

## 6. Path 2 · Recover supervision

**Objective:** turn SAM pseudo-labels and Grad-CAM maps into training signal.

Path 1 kept the label file and changed the loss. Path 2 does the opposite: it keeps the loss and fills in
the labels. A second model looks at every pixel nobody painted and makes a guess, and the head trains on
those guesses as if a person had drawn them. That is a **pseudo-label**. Think of handing your half-done
colouring-in to a fast, confident colleague: the page comes back full, and every mistake they made with
confidence is now in your training set. We start from the scribbles of section 4 (a few strokes per
image, minutes of work), so this path costs no more annotation than rung 2 of the ladder. Two colleagues
get the job: **SAM**, prompted with clicks taken from the strokes, and **Grad-CAM**, the heat map of a
small classifier that only learnt whether a tile contains a vessel.

<table width="100%" bgcolor="#EDE7F6"><tr><td><font color="#311B92">📘 <b>Theory: pseudo-labels.</b>
Pseudo-labelling (Lee, "Pseudo-Label: The Simple and Efficient Semi-Supervised Learning Method", ICML
2013 workshop) trains a model on another model's predictions for unlabelled data. It works when the
guesses are more often right than wrong <i>and</i> their mistakes are not systematic: a model trained on
its teacher's errors learns to repeat them, which the literature calls <i>confirmation bias</i>. The usual
guard is a confidence threshold: keep only guesses the teacher is sure of and leave the rest unlabelled
(FixMatch, Sohn et al., NeurIPS 2020, <a href="https://arxiv.org/abs/2001.07685">arXiv:2001.07685</a>).
Here the teacher is a different model, not the head itself, and the person's strokes always overrule it.
</font></td></tr></table>

This helper cell adds a colour for each head of this path and defines how to spread SAM clicks along the
strokes of one kind, vessel or background.

In [ ]:
# @title Path 2 helpers: colours, clicks along the strokes { display-mode: "form" }
APPROACH_COLOURS |= {
    "sam_pseudo": "#0277BD",
    "gradcam_pseudo": "#827717",
}
APPROACH_NAMES |= {
    "sam_pseudo": "pseudo-labels: SAM from stroke clicks",
    "gradcam_pseudo": "pseudo-labels: Grad-CAM of a tile classifier",
}


def prob_rgb(prob: np.ndarray) -> np.ndarray:
    """A probability map as RGB on a fixed scale: purple = 1, white = 0.5 (unsure), orange = 0."""
    return (mpl.colormaps["PuOr"](np.clip(prob, 0, 1))[..., :3] * 255).astype(np.uint8)


def stroke_points(item: dict, value: int, n: int) -> list[list[float]]:
    """n (x, y) clicks spread along the strokes of one kind (1 vessel, 2 background) by farthest-point
    sampling from the first stroke pixel: no randomness, no ground truth."""
    ys, xs = np.nonzero(item["scribbles"] == value)
    if len(ys) == 0:
        return []
    pts = np.stack([xs, ys], 1).astype(float)
    chosen = [0]
    dist = np.linalg.norm(pts - pts[0], axis=1)
    while len(chosen) < min(n, len(pts)):
        k = int(dist.argmax())
        chosen.append(k)
        dist = np.minimum(dist, np.linalg.norm(pts - pts[k], axis=1))
    return [[float(x), float(y)] for x, y in pts[chosen]]

### 6.1 SAM fills in the gaps

The scribbles of section 4 again: a few vessel strokes and a few background strokes on each of the same
five training images. We turn them into SAM prompts, twenty clicks spread along the vessel strokes as
"vessel" and twenty along the background strokes as "not vessel", and keep SAM's answer for every pixel.
Only the strokes go in (L-005). The ground truth appears in the figure for you to judge the guesses,
never in training.

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your prediction.</b> Section 2
showed what SAM does with clicks on a vessel tree. With forty clicks per image, half of them saying "not
here", will its mask follow the vessels, or cover a region around them? Write one guess.
</font></td></tr></table>

This cell asks SAM for a probability map on each training image from the stroke clicks (one SAM image
encoding per image: seconds on a GPU, up to half a minute per image on a CPU) and shows one image. Short
on time? Set `RUN_SAM = False` to load our maps, made with the same recipe and clicks drawn at random.

In [ ]:
RUN_SAM = True  # False -> load our precomputed SAM maps for the training images
N_PSEUDO_CLICKS = 20  # clicks per kind (vessel / background) and image, taken from the strokes
if not isinstance(RUN_SAM, bool):
    raise SystemExit("RUN_SAM must be True or False.")

t0 = time.monotonic()
pseudo_probs, pseudo_clicks = [], []  # per training image: SAM's probability map, (vessel, background) clicks
if RUN_SAM:
    for it in scrib_train:
        pos, neg = stroke_points(it, 1, N_PSEUDO_CLICKS), stroke_points(it, 2, N_PSEUDO_CLICKS)
        prob = sam.probability(it["image"], points=pos + neg, labels=[1] * len(pos) + [0] * len(neg))
        pseudo_probs.append(prob)
        pseudo_clicks.append((pos, neg))
    src = f"SAM 2.1 run here, {N_PSEUDO_CLICKS} + {N_PSEUDO_CLICKS} clicks per image"
else:
    ours = PRECOMPUTED / "pseudo_labels" / "specialised"
    if not (ours / "prompts.json").is_file():
        raise SystemExit(f"{ours} is missing. Run setup_check.ipynb first, or set RUN_SAM = True.")
    prompts = json.loads((ours / "prompts.json").read_text())["prompts"]
    for it in scrib_train:
        pseudo_probs.append(read_png(ours / f"{it['name']}.png") / 255)
        p = prompts[it["name"]]
        clicks = [
            [xy for xy, lab in zip(p["points_xy"], p["labels"], strict=True) if lab == k] for k in (1, 0)
        ]
        pseudo_clicks.append(tuple(clicks))
    src = "our precomputed maps (same recipe, clicks drawn at random from the strokes)"
pseudo_sec = round(time.monotonic() - t0, 1)

# diagnosis only: SAM's guesses at the usual 0.5 cut, scored against the hidden truth (never for training)
sam_quality = mean_scores([score(p >= 0.5, it) for p, it in zip(pseudo_probs, scrib_train, strict=True)])
sam_marked = float(
    np.mean([(p >= 0.5)[region(it)].mean() for p, it in zip(pseudo_probs, scrib_train, strict=True)])
)
# how sure SAM is, inside the fundus (no truth needed): the spread of its probabilities
spread = np.quantile(
    np.concatenate([p[region(it)] for p, it in zip(pseudo_probs, scrib_train, strict=True)]),
    [0.05, 0.5, 0.95],
)
print(
    f"SAM pseudo-labels on {len(scrib_train)} training images · {src} · {pseudo_sec} s on {DEVICE}\n"
    f"SAM's probability inside the fundus · 5 % of the pixels below {spread[0]:.2f} · median "
    f"{spread[1]:.2f} · 95 % below {spread[2]:.2f}\n"
    f"diagnosis (truth used to score only): at 0.5 they mark {sam_marked:.0%} of the fundus · IoU "
    f"{sam_quality['iou']:.3f} · clDice {sam_quality['cldice']:.3f}"
)
ex, (pos, neg) = scrib_train[0], pseudo_clicks[0]
show_clicks(
    [
        (f"{ex['name']} · vessel clicks", ex["image"], pos),
        ("background clicks", ex["image"], neg),
        ("SAM probability: purple 1, white 0.5, orange 0", prob_rgb(pseudo_probs[0]), None),
        ("pseudo-label at 0.5 vs truth", error_map(ex["image"], pseudo_probs[0] >= 0.5, ex), None),
    ],
    width=2.8,
)

In the last panel, violet = SAM's guess is a real vessel, orange = SAM invented a vessel, green = a vessel
SAM left out. Every orange and green pixel becomes a wrong training label if we take the guesses as they
are.

**Q22.** 🟡 Where does SAM's guess go wrong: around the thick vessels, along the thin ones, in the
background between the strokes? Now look at the probability panel and the printed spread: is SAM sure
of itself (deep purple, deep orange) or unsure (pale)? What does that mean for a cut at exactly 0.5?

<details><summary>What to look for</summary>

SAM looks for <i>objects</i> that contain the positive clicks and avoid the negative ones. Forty clicks
scattered over a vessel tree do not describe one object, so SAM settles for the object that contains
most of them, often the whole fundus, and draws its edge with care while the inside stays a region
(orange), not a tree. Thin branches between the clicks can drop out (green). The probability panel tells
you how much SAM believes it. If most of it is pale, the probabilities sit just around 0.5, and the
exact cut decides almost everything: a little below and the whole fundus is "vessel", a little above
and nothing is. The binary mask hides that doubt: it looks just as crisp as a confident one. Whatever you
see on your run, the threshold is a choice you make, not a property of SAM.

</details>

### 6.2 From a probability map to training targets

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your turn (optional, 3 min).</b>
Fill <code>my_pseudo_targets</code>. In: SAM's <code>prob</code> map (floats in [0, 1]), the
<code>scribbles</code> map (0 unlabelled, 1 vessel stroke, 2 background stroke), the boolean
<code>region</code>, a <code>threshold</code> and an unsure <code>band</code>. Out: int8 targets, 1 vessel,
0 background, −1 ignored. A pixel with <code>prob ≥ threshold + band</code> is vessel, one with
<code>prob &lt; threshold − band</code> is background, and the doubtful ones in between are ignored. The
person's strokes always win over SAM, and pixels outside <code>region</code> are always −1. The cell
checks your version on a 3 × 4 play map, without and with a band, and continues with ours if you skip it.
</font></td></tr></table>

In [ ]:
# ✍️ YOUR CODE
def my_pseudo_targets(
    prob: np.ndarray, scribbles: np.ndarray, region: np.ndarray, threshold: float = 0.5, band: float = 0.0
) -> np.ndarray:
    """Targets (int8) from a pseudo-label probability map: 1 where prob >= threshold + band, 0 where
    prob < threshold - band, -1 in between. Strokes overrule the map (1 vessel stroke -> 1, 2 background
    stroke -> 0); outside `region` -> -1."""
    ...


def _ref_pseudo_targets(
    prob: np.ndarray, scribbles: np.ndarray, region: np.ndarray, threshold: float = 0.5, band: float = 0.0
) -> np.ndarray:
    """Reference: sure guesses become labels, doubtful ones are ignored, strokes and region come last."""
    targets = np.full(prob.shape, -1, np.int8)
    targets[prob >= threshold + band] = 1
    targets[prob < threshold - band] = 0
    targets[scribbles == 1] = 1
    targets[scribbles == 2] = 0
    targets[~np.asarray(region, bool)] = -1
    return targets


_play_prob = np.array([[0.95, 0.62, 0.45, 0.10], [0.55, 0.30, 0.71, 0.05], [0.20, 0.90, 0.40, 0.80]])
_play_scrib = np.zeros((3, 4), np.uint8)
_play_scrib[1, 1] = 1  # a vessel stroke where SAM says "not vessel"
_play_scrib[2, 1] = 2  # a background stroke where SAM says "vessel"
_play_region = np.ones((3, 4), bool)
_play_region[:, 3] = False  # the last column is outside the evaluation region
pseudo_targets = use(
    my_pseudo_targets,
    reference=_ref_pseudo_targets,
    check=lambda f: [
        np.asarray(f(_play_prob, _play_scrib, _play_region, 0.5, b)).astype(int) for b in (0.0, 0.2)
    ],
    name="pseudo_targets",
)
print("play example · targets with threshold 0.5 and band 0.2 (−1 = ignored):")
print(pseudo_targets(_play_prob, _play_scrib, _play_region, 0.5, 0.2))

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your prediction.</b> The
scribble head of section 4 heard about the strokes only. This head hears about every pixel of the fundus,
most of it in SAM's words. Will it mark more of the held-out fundus than the scribble head, or less? Will
its IoU be higher? Write two guesses.</font></td></tr></table>

This cell turns SAM's maps into targets at the usual cut (threshold 0.5, no band), trains the section-2
head on them, and prints the seconds it took. Not done in a minute, or on a CPU and in a hurry? Set
`TRAIN_YOURSELF = False` and run it again: it loads our "pseudo" checkpoint, trained the same way on all
fit images.

In [ ]:
TRAIN_YOURSELF = True  # False -> load our precomputed "pseudo" head instead of training
if not isinstance(TRAIN_YOURSELF, bool):
    raise SystemExit("TRAIN_YOURSELF must be True or False.")

# the supervision of this path: the strokes, and SAM's guesses from clicks on the strokes (L-005)
sam_targets = [
    pseudo_targets(p, it["scribbles"], region(it), 0.5, 0.0)
    for p, it in zip(pseudo_probs, scrib_train, strict=True)
]
if TRAIN_YOURSELF:
    sam_head, sam_sec = train_head(scrib_feats, sam_targets, HEAD_STEPS)
    src = f"trained here on {len(scrib_train)} images, {HEAD_STEPS} steps, SAM pseudo-labels at 0.5"
else:
    sam_head, entry = load_head(SPECIALISED, "pseudo", side)
    sam_sec = entry["train_sec"]
    src = f"our checkpoint {entry['file']} ({len(entry['train_names'])} images)"
print(f"SAM pseudo-label head @ {side} px · {src} · {sam_sec:.1f} s on {DEVICE}")
if sam_sec > 60:
    print("ℹ️ Slow session? Set TRAIN_YOURSELF = False above to load our checkpoint next time.")

This cell scores the new head on the held-out images with the numbers of section 4 and prints it under
the scribble head, which saw the same strokes but kept every other pixel unknown.

In [ ]:
t0 = time.monotonic()
path2_preds = {
    "scribbles": ladder_preds["scribbles"],
    "sam_pseudo": [
        predict_head(sam_head, features_for(SPECIALISED, it, side), it["mask"].shape) for it in SPEC_EVAL
    ],
}
sam_infer = round(time.monotonic() - t0, 1)
path2_report = {k: rung_report(v) for k, v in path2_preds.items()}
RESULTS["pseudo_label_iou"] = path2_report["sam_pseudo"]["iou"]  # held-out IoU of the head (SPEC §9)
RESULTS["sam_pseudo_head"] = path2_report["sam_pseudo"] | {
    "long_side": side,
    "n_train": len(scrib_train),
    "threshold": 0.5,
    "band": 0.0,
    "pseudo_label_iou_fit": sam_quality["iou"],
    "pseudo_sec": pseudo_sec,
    "sam_run_here": RUN_SAM,
    "trained_here": TRAIN_YOURSELF,
    "train_sec": round(float(sam_sec), 2),
    "infer_sec": sam_infer,
}


def print_path2(reports: dict) -> None:
    """One line per head with the numbers of section 4."""
    print(f"{'approach':<46}{COLUMNS}")
    for k, v in reports.items():
        name = APPROACH_NAMES.get(k, k)
        print(
            f"{name:<46}{v['iou']:>6.3f}{v['cldice']:>8.3f}{v['recall']:>8.3f}{v['precision']:>8.3f}"
            f"{v['marked']:>8.0%}"
        )


print_path2(path2_report)

<table width="100%" bgcolor="#F3E5F5"><tr><td><font color="#4A148C">❓ <b>Question.</b> Run the next cell,
then move the cut.</font></td></tr></table>

**Q23.** 🟡 Tune the pseudo-labels. Raise or lower `PSEUDO_THRESHOLD` and give the doubtful pixels an
`UNSURE_BAND` to ignore; pick your values from the spread printed in 6.1. How many training pixels does
each setting get wrong (the diagnosis lines), and does the head on the held-out images move the same way?
Is there a setting that beats 0.5 on your run?

```python
PSEUDO_THRESHOLD = 0.5   # try 0.55, 0.6, 0.7 (or the median printed in 6.1)
UNSURE_BAND = 0.0        # try 0.03: pixels within 0.03 of the threshold are ignored
```

This cell turns SAM's maps into targets with your setting, measures on the training images how many of
them disagree with the hidden truth (diagnosis only), trains the same head on them (seconds) and prints it
next to the head at 0.5.

In [ ]:
# Q23 — tune the pseudo-label threshold and the unsure band
PSEUDO_THRESHOLD = 0.5  # try 0.55, 0.6, 0.7 (or the median printed in 6.1)
UNSURE_BAND = 0.0  # try 0.03: pixels within 0.03 of the threshold are ignored
ok = all(isinstance(v, (int, float)) for v in (PSEUDO_THRESHOLD, UNSURE_BAND))
if not ok or UNSURE_BAND < 0 or PSEUDO_THRESHOLD - UNSURE_BAND <= 0 or PSEUDO_THRESHOLD + UNSURE_BAND > 1:
    raise SystemExit(
        "PSEUDO_THRESHOLD must lie between 0 and 1, and UNSURE_BAND ≥ 0 must keep both edges inside."
    )


def target_errors(targets: list[np.ndarray]) -> dict:
    """Diagnosis only (uses the hidden truth): share of the fundus ignored, of the true vessel pixels called
    background, and of the true background pixels called vessel, over the training images."""
    pairs = list(zip(targets, scrib_train, strict=True))
    return {
        "ignored": float(np.mean([(t < 0)[region(it)].mean() for t, it in pairs])),
        "vessel_as_bg": float(
            np.mean([((t == 0) & it["mask"]).sum() / max(it["mask"].sum(), 1) for t, it in pairs])
        ),
        "bg_as_vessel": float(
            np.mean(
                [
                    ((t == 1) & ~it["mask"] & region(it)).sum() / max((~it["mask"] & region(it)).sum(), 1)
                    for t, it in pairs
                ]
            )
        ),
    }


tuned_targets = [
    pseudo_targets(p, it["scribbles"], region(it), PSEUDO_THRESHOLD, UNSURE_BAND)
    for p, it in zip(pseudo_probs, scrib_train, strict=True)
]
tuned_head, tuned_sec = train_head(scrib_feats, tuned_targets, HEAD_STEPS)
tuned_preds = [
    predict_head(tuned_head, features_for(SPECIALISED, it, side), it["mask"].shape) for it in SPEC_EVAL
]
tuned_report = rung_report(tuned_preds)
RESULTS["sam_pseudo_tuned"] = tuned_report | {"threshold": PSEUDO_THRESHOLD, "band": UNSURE_BAND}
print(f"{'targets on the training images':<46}{'ignored':>8}{'vessel → bg':>13}{'bg → vessel':>13}")
for name, t in {"threshold 0.5, no band": sam_targets, "your setting": tuned_targets}.items():
    e = target_errors(t)
    print(f"{name:<46}{e['ignored']:>8.0%}{e['vessel_as_bg']:>13.0%}{e['bg_as_vessel']:>13.0%}")
print(f"\nyour head: threshold {PSEUDO_THRESHOLD}, band {UNSURE_BAND} · trained in {tuned_sec:.1f} s")
print_path2({"threshold 0.5, no band": path2_report["sam_pseudo"], "your setting": tuned_report})

<details><summary>What to look for</summary>

The threshold trades one kind of wrong label for the other. A low cut turns more of SAM's region into
"vessel" (bg → vessel rises), and the head learns to mark more; a high cut keeps only SAM's surest
pixels and calls the rest background (vessel → bg rises): the section-3 mistake again, now with SAM as
the annotator who stopped early. When SAM's probabilities crowd around one value, a step of 0.05 can
flip most of the fundus from one side to the other, so there may be no comfortable middle. The band
removes the doubtful pixels instead of guessing, so both error columns can fall at once, paid for with
fewer labelled pixels (the "ignored" column). Whether the held-out numbers follow the training diagnosis
depends on your run. In a real project you would pick the setting on a small fully labelled check set,
never on the test images.

</details>

<table width="100%" bgcolor="#FDECEA"><tr><td><font color="#8B1A10">⚠️ <b>Pitfall: over-trusting SAM's
confident masks.</b> A pseudo-label looks like a real label: crisp edges, full coverage, no "I don't
know". The head cannot tell the difference, and it will learn SAM's habits (here, a region where you
wanted a line) as faithfully as the vessels. Before training on pseudo-labels, score them on a few images
you labelled completely, as the diagnosis line does here.</font></td></tr></table>

### 6.3 Grad-CAM: where a classifier looks

The second colleague does not segment at all. We train a small classifier to answer one yes/no question,
then ask it <i>where</i> in the image its "yes" came from. Grad-CAM turns that answer into a heat map,
and a threshold turns the heat map into a pseudo-mask.

<table width="100%" bgcolor="#FDECEA"><tr><td><font color="#8B1A10">⚠️ <b>Pitfall: an awkward fit, said
plainly.</b> Grad-CAM needs an image-level question with a "no" class, such as "is there a defect in this
image?". Every retina image contains vessels, so there is no "no" at image level. We fall back to a
patch-level question, "does this 64 px tile contain a vessel?", with tile labels read off the strokes.
Most tiles of a fundus crop do contain a vessel (the next cell counts them), and a heat map at tile or
patch resolution cannot outline a line a few pixels wide. Our bake-off judged it a poor fit; we run it
anyway because on a task with real image-level labels (defects, tumours, "contains a cell") it is often
the cheapest supervision there is.</font></td></tr></table>

<table width="100%" bgcolor="#EDE7F6"><tr><td><font color="#311B92">📘 <b>Theory: Grad-CAM.</b> Take the
classifier's last feature maps <i>A<sub>k</sub></i> and the gradient of the class score with respect to
them. Average each gradient map into one weight α<sub>k</sub>, sum α<sub>k</sub>·<i>A<sub>k</sub></i>, keep
the positive part, and rescale to [0, 1]: that is the map (Selvaraju et al., ICCV 2017,
<a href="https://arxiv.org/abs/1610.02391">arXiv:1610.02391</a>; we use the <code>grad-cam</code> package).
Our classifier ends in an average over patches, and for such a classifier Grad-CAM reduces to the older
CAM (Zhou et al., CVPR 2016, <a href="https://arxiv.org/abs/1512.04150">arXiv:1512.04150</a>): the
classifier's own evidence for "vessel", patch by patch. The map is as coarse as the features it is
computed on: one value per 16 px patch.</font></td></tr></table>

This helper cell defines the tile classifier (a 3 × 3 convolution and a 1 × 1 convolution on the frozen
DINOv3 features, averaged over the tile), cuts feature grids into tiles, reads tile labels off the strokes,
trains the classifier and computes its Grad-CAM map on a whole image.

In [ ]:
# @title Grad-CAM helpers: tile classifier, tiles, labels from strokes, the map { display-mode: "form" }
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget


class TileClassifier(torch.nn.Module):
    """'Does this tile contain a vessel?' on frozen DINOv3 features: 3 x 3 conv (384 -> 32), ReLU, one logit
    per patch, averaged. Fully convolutional, so it also runs on a whole image."""

    def __init__(self, dim: int, hidden: int = 32):
        super().__init__()
        self.hidden = torch.nn.Conv2d(dim, hidden, 3, padding=1)
        self.act = torch.nn.ReLU()
        self.out = torch.nn.Conv2d(hidden, 1, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.out(self.act(self.hidden(x))).mean((2, 3))  # [B, 1]


def to_tiles(grid: torch.Tensor, t: int) -> torch.Tensor:
    """[C, h, w] -> [n, C, t, t]: non-overlapping t x t tiles, row by row (a ragged edge is dropped)."""
    c, h, w = grid.shape
    nh, nw = h // t, w // t
    g = grid[:, : nh * t, : nw * t].reshape(c, nh, t, nw, t)
    return g.permute(1, 3, 0, 2, 4).reshape(nh * nw, c, t, t)


def tile_labels(item: dict, grid: tuple[int, int], t: int) -> torch.Tensor:
    """Per tile, from the strokes only: 1 = a vessel stroke crosses it, 0 = only background strokes,
    -1 = no stroke (not used)."""
    has = [to_tiles(patch_share(item["scribbles"] == v, grid)[None], t).amax((1, 2, 3)) > 0 for v in (1, 2)]
    return torch.where(has[0], 1, torch.where(has[1], 0, -1))


def train_tile_classifier(x: torch.Tensor, y: torch.Tensor, steps: int, lr: float = 0.01):
    """Full-batch Adam with the balanced cross-entropy of every head; returns the classifier and seconds."""
    torch.manual_seed(SEED)
    clf = TileClassifier(x.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(clf.parameters(), lr=lr)
    target = (y == 1).float()
    w = torch.where(y == 1, 0.5 / max(int((y == 1).sum()), 1), 0.5 / max(int((y == 0).sum()), 1))
    t0 = time.monotonic()
    for _ in range(steps):
        opt.zero_grad()
        bce = F.binary_cross_entropy_with_logits(clf(x)[:, 0], target, reduction="none")
        (bce * w).sum().backward()
        opt.step()
    return clf.eval(), time.monotonic() - t0


def cam_map(clf: TileClassifier, feat: torch.Tensor) -> np.ndarray:
    """Grad-CAM of the 'contains a vessel' score on a whole (standardised) feature grid: [h, w] in [0, 1]."""
    with GradCAM(model=clf, target_layers=[clf.act]) as cam:
        return cam(input_tensor=feat[None], targets=[ClassifierOutputTarget(0)])[0]

This cell cuts the features of the five training images into tiles of 64 px of the crop, labels each tile
from the strokes (a vessel stroke crosses it → "vessel"; only background strokes → "no vessel"; no stroke
→ unused), trains the classifier (seconds) and prints how many tiles of each kind it saw. The last line
uses the held-out truth to show how rare a vessel-free tile really is.

In [ ]:
TILE_PATCHES = 1 if SMOKE else 4  # 64 px of the 1024 px crop either way (smoke thumbnails are 4× smaller)
CLS_STEPS = 30 if SMOKE else 200
all_feats = torch.cat([f.flatten(1) for f in scrib_feats], 1)
cls_mu, cls_sd = all_feats.mean(1).view(-1, 1, 1), all_feats.std(1).view(-1, 1, 1) + 1e-6
std_feats = [(f - cls_mu) / cls_sd for f in scrib_feats]
tile_x, tile_y = [], []
for it, f in zip(scrib_train, std_feats, strict=True):
    lab = tile_labels(it, tuple(f.shape[1:]), TILE_PATCHES)
    tile_x.append(to_tiles(f, TILE_PATCHES)[lab >= 0])
    tile_y.append(lab[lab >= 0])
tile_x, tile_y = torch.cat(tile_x), torch.cat(tile_y)
clf, cls_sec = train_tile_classifier(tile_x, tile_y, CLS_STEPS)
with torch.no_grad():
    fit_acc = float(((clf(tile_x)[:, 0] > 0).long() == tile_y).float().mean())
# evaluation only: the share of held-out tiles that truly contain a vessel
held_tiles = []
for it in SPEC_EVAL:
    grid = tuple(features_for(SPECIALISED, it, side).shape[1:])
    share = to_tiles(patch_share(it["mask"] & region(it), grid)[None], TILE_PATCHES).amax((1, 2, 3))
    inside = to_tiles(patch_share(region(it), grid)[None], TILE_PATCHES).mean((1, 2, 3)) > 0.5
    held_tiles.append(float((share[inside] > 0).float().mean()))
vessel_tile_share = float(np.mean(held_tiles))
print(
    f"tile classifier · {int((tile_y == 1).sum())} 'vessel' tiles, {int((tile_y == 0).sum())} 'no vessel' "
    f"tiles from the strokes · fit accuracy {fit_acc:.0%} · {cls_sec:.1f} s on {DEVICE}"
)
print(f"held-out truth (evaluation only): {vessel_tile_share:.0%} of the fundus tiles contain a vessel")

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your prediction.</b> The
classifier only ever answered "vessel in this tile, yes or no". Will its heat map look like a vessel
tree, like blobs, or like noise? Write one guess.</font></td></tr></table>

This cell computes the Grad-CAM map of every training image, cuts it at half its peak into a pseudo-mask
(with your `pseudo_targets`: strokes still win), trains the same head on it (seconds; there is no
checkpoint for this one) and scores it on the held-out images. The figure shows one training image.

In [ ]:
CAM_THRESHOLD = 0.5  # share of the map's peak; Grad-CAM maps are rescaled to [0, 1] per image
t0 = time.monotonic()
cam_probs = []
for it, f in zip(scrib_train, std_feats, strict=True):
    cam = torch.from_numpy(cam_map(clf, f))[None, None]
    cam_probs.append(
        F.interpolate(cam, size=it["mask"].shape, mode="bilinear", align_corners=False)[0, 0].numpy()
    )
cam_sec = round(time.monotonic() - t0, 1)
# the supervision: strokes, and the tile classifier's heat map (trained on stroke-derived tile labels)
cam_targets = [
    pseudo_targets(p, it["scribbles"], region(it), CAM_THRESHOLD, 0.0)
    for p, it in zip(cam_probs, scrib_train, strict=True)
]
cam_quality = mean_scores([score(t == 1, it) for t, it in zip(cam_targets, scrib_train, strict=True)])
cam_head, cam_head_sec = train_head(scrib_feats, cam_targets, HEAD_STEPS)
t0 = time.monotonic()
path2_preds["gradcam_pseudo"] = [
    predict_head(cam_head, features_for(SPECIALISED, it, side), it["mask"].shape) for it in SPEC_EVAL
]
cam_infer = round(time.monotonic() - t0, 1)
path2_report["gradcam_pseudo"] = rung_report(path2_preds["gradcam_pseudo"])
RESULTS["gradcam_iou"] = path2_report["gradcam_pseudo"]["iou"]  # held-out IoU of the head (SPEC §9)
RESULTS["gradcam_note"] = (
    f"Patch-level proxy '{SPECIALISED['gradcam']['patch_level_task']}' (tiles labelled from the strokes), "
    f"because every image contains the structure and there is no image-level negative class: an awkward "
    f"fit, {vessel_tile_share:.0%} of the held-out tiles contain foreground"
)
RESULTS["gradcam_head"] = path2_report["gradcam_pseudo"] | {
    "long_side": side,
    "n_train": len(scrib_train),
    "tile_patches": TILE_PATCHES,
    "n_tiles": [int((tile_y == 1).sum()), int((tile_y == 0).sum())],
    "classifier_fit_acc": fit_acc,
    "held_vessel_tile_share": vessel_tile_share,
    "threshold": CAM_THRESHOLD,
    "pseudo_label_iou_fit": cam_quality["iou"],
    "train_sec": round(float(cls_sec + cam_head_sec), 2),
    "pseudo_sec": cam_sec,
    "infer_sec": cam_infer,
}
cam_marked = float(
    np.mean([(t == 1)[region(it)].mean() for t, it in zip(cam_targets, scrib_train, strict=True)])
)
print(
    f"Grad-CAM pseudo-labels · {cam_sec} s · diagnosis (truth used to score only): they mark "
    f"{cam_marked:.0%} of the fundus · IoU {cam_quality['iou']:.3f} · clDice {cam_quality['cldice']:.3f} "
    f"on the training images (SAM at 0.5: {sam_marked:.0%}, IoU {sam_quality['iou']:.3f})"
)
print(f"Grad-CAM head trained in {cam_head_sec:.1f} s on {DEVICE}")
print_path2(path2_report)
show_masks(
    [
        (f"{ex['name']}", ex["image"]),
        ("Grad-CAM: purple = strongest evidence", prob_rgb(cam_probs[0])),
        ("Grad-CAM pseudo-label vs truth", error_map(ex["image"], cam_targets[0] == 1, ex)),
        ("SAM pseudo-label vs truth", error_map(ex["image"], sam_targets[0] == 1, ex)),
    ],
    width=2.8,
)

**Q24.** 🟡 Compare the two pseudo-labels of the same training image. What shape does the Grad-CAM
evidence have, and where does it sit? Which of the two would you rather correct by hand, and why?

<details><summary>What to look for</summary>

The heat map is a set of blobs one or a few patches wide, sitting where the classifier found the most
"vessel-like" patches: usually the thick, dark vessels, and sometimes dark background. It does not
follow a line, because its resolution is the 16 px patch and because the question it learnt ("is there a
vessel somewhere in this tile?") does not need a line. Cut at half its peak, it turns into fat regions
around the vessels the classifier found most convincing: the thick ones, drawn several patches wide,
with the thin ones mostly missing. SAM's guess fails differently, as one big region with a careful edge.
Neither is the vessel map. A coarse but <i>local</i> guess can still teach the head more than a sharp
but global one, so compare the held-out lines above before you pick. On a task where the class really
is a blob (a defect, a nucleus) and some images contain none, the same recipe fits much better.

</details>

<table width="100%" bgcolor="#F3E5F5"><tr><td><font color="#4A148C">❓ <b>Question (optional).</b> Run the
next cell to use the heat map as a SAM prompt instead of a mask.</font></td></tr></table>

**Q25.** 🔴 Instead of cutting the heat map, let it pick SAM clicks: the hottest patches as "vessel", the
coldest as "not vessel" (the trick of section 4.2, with Grad-CAM in place of the prototype). Do SAM's
masks from these clicks look more like the SAM pseudo-labels of 6.1 or like the Grad-CAM blobs?

```python
N_CAM_CLICKS = 5   # try 2, 10
```

This cell picks the clicks on each training image's heat map, asks SAM for a mask (one SAM image encoding
per image) and scores the three kinds of pseudo-label on the training images: diagnosis only, no training.

In [ ]:
# Q25 — Grad-CAM as a SAM prompt: hottest patches = vessel clicks, coldest = background clicks
N_CAM_CLICKS = 5  # try 2, 10
if not isinstance(N_CAM_CLICKS, int) or N_CAM_CLICKS < 1:
    raise SystemExit("N_CAM_CLICKS must be a whole number of at least 1.")
cam_sam_masks, cam_clicks = [], []
for it, f in zip(scrib_train, std_feats, strict=True):
    s = torch.from_numpy(cam_map(clf, f)).to(DEVICE)
    keep = patch_share(region(it), tuple(s.shape))
    pos = top_points(s, keep, N_CAM_CLICKS, it["mask"].shape, largest=True)
    neg = top_points(s, keep, N_CAM_CLICKS, it["mask"].shape, largest=False)
    cam_sam_masks.append(
        sam.probability(it["image"], points=pos + neg, labels=[1] * len(pos) + [0] * len(neg)) >= 0.5
    )
    cam_clicks.append(pos)
cam_sam_quality = mean_scores([score(m, it) for m, it in zip(cam_sam_masks, scrib_train, strict=True)])
RESULTS["gradcam_sam_prompt_iou_fit"] = cam_sam_quality["iou"]
print("pseudo-labels on the training images (diagnosis only: scored against the hidden truth)")
for name, q in {
    "SAM from stroke clicks (6.1, at 0.5)": sam_quality,
    f"Grad-CAM cut at {CAM_THRESHOLD}": cam_quality,
    f"SAM from {N_CAM_CLICKS} + {N_CAM_CLICKS} Grad-CAM clicks": cam_sam_quality,
}.items():
    print(f"{name:<44} IoU {q['iou']:.3f} · clDice {q['cldice']:.3f}")
show_clicks(
    [
        ("Grad-CAM clicks: vessel", ex["image"], cam_clicks[0]),
        ("SAM from Grad-CAM clicks vs truth", error_map(ex["image"], cam_sam_masks[0], ex), cam_clicks[0]),
    ],
    width=3.4,
)

<details><summary>What to look for</summary>

The clicks land where the heat map is hottest, usually on or next to a thick vessel, so SAM's mask looks
like SAM's masks of 6.1: sharp edges around whatever object contains the clicks, often a region rather
than a line. The prompt route swaps Grad-CAM's coarse blobs for SAM's sharp edges, but it also brings
back SAM's idea of an object, and a click that lands on dark background drags the mask there. Two weak
signals combined are not automatically a strong one; their mistakes can add up as easily as cancel out.

</details>

### 6.4 Path 2, side by side

This cell puts the three heads that start from the same strokes next to each other: the scribble head of
section 4 (unlabelled = unknown), the head on SAM's guesses, the head on Grad-CAM's guesses. Then a bar
chart in the approach colours and the error maps on the held-out demo image (violet found, orange
invented, green missed). It also writes the annotation effort for the comparison of section 8: the same
strokes for all three.

In [ ]:
RESULTS["path2_effort_min"] = {
    k: round(effort_minutes(n_strokes=n_strokes), 2) for k in ("sam_pseudo", "gradcam_pseudo")
}
print_path2(path2_report)
print(
    f"{'everything is foreground':<46}{RESULTS['specialised_everything_iou']:>6.3f}\n"
    f"annotation for all three: {n_strokes} strokes on {len(scrib_train)} images "
    f"(≈ {effort_minutes(n_strokes=n_strokes):.1f} min); the pseudo-labels add compute, not annotation"
)
metrics = ["iou", "cldice", "recall", "precision"]
x, width = np.arange(len(metrics)), 0.8 / len(path2_report)
fig, ax = plt.subplots(figsize=(7.2, 3.2))
for n, (k, v) in enumerate(path2_report.items()):
    shift = (n - (len(path2_report) - 1) / 2) * width
    ax.bar(x + shift, [v[m] for m in metrics], width, color=APPROACH_COLOURS[k], label=APPROACH_NAMES[k])
ax.axhline(
    RESULTS["specialised_everything_iou"], color=THEME["background"], ls="--", label='IoU of "everything"'
)
ax.set_xticks(x, ["IoU", "clDice", "recall", "precision"])
ax.set_ylim(0, 1)
ax.set_title("Path 2: same strokes, three ways to fill in the rest")
ax.legend(fontsize=7, frameon=False, loc="upper left")
plt.tight_layout()
plt.show()
show_masks(
    [
        (
            f"{APPROACH_NAMES[k].split(':')[-1].strip()} · IoU {score(p[k_demo], demo)['iou']:.2f}",
            error_map(demo["image"], p[k_demo], demo),
        )
        for k, p in path2_preds.items()
    ]
    + [("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"]))],
    width=2.8,
)

**Q26.** 🟡 All three heads cost the same strokes. Which one would you train if you could add one more
afternoon of work: more strokes, or a better second model (say, a SAM checkpoint fine-tuned on retinas)?
What does the table suggest, and what would you want to measure first?

<details><summary>A possible answer</summary>

Pseudo-labels pass on the second model's habits, so a head trained on them is capped by how well that
model understands <i>your</i> structures. Here SAM thinks in objects and Grad-CAM in tiles, and neither
thinks in thin lines, so better pseudo-labels would need a better teacher for this domain, not a cleverer
threshold. More strokes add true information but no extra coverage between them. Which pays more depends
on the gap you see: if the pseudo-labels are close to the truth on a few fully labelled images, a better
teacher (or a cleaned-up pseudo-label) is the lever; if they are far, spend the afternoon on strokes. In
both cases the first measurement is the diagnosis line of 6.1 on a handful of images you labelled
completely.

</details>

<table width="100%" bgcolor="#E8EAF6"><tr><td><font color="#1A237E">🔀 <b>Across approaches.</b> Path 1
changed what the loss is told about unlabelled pixels; Path 2 asks another model to label them. The
scribble head leaves the gaps unknown; the SAM head fills them with SAM's idea of an object; the
Grad-CAM head fills them with a classifier's coarse evidence. Each adds information <i>and</i> the second
model's bias, and the head cannot tell which is which. Section 7 drops the head altogether and asks SAM
directly, with the same clicks.</font></td></tr></table>

## 7. Path 3 · Skip specialist training

**Objective:** prompt the foundation model directly with scribble points and boxes, no training.

Paths 1 and 2 trained a head: the strokes on five images taught a small model, and that model then
segmented every new image on its own. Path 3 trains nothing. The strokes a person draws on an image
become SAM's prompts on <i>that same image</i>, and SAM's answer is the result. It is the difference
between teaching an intern once and pointing at every image yourself: no training time, no training
data, but the pointing never stops, because every new image needs its own strokes. Section 2 already
gave SAM clicks on the vessels and one big box; here we add the two things it lacked: "not here" clicks
from the background strokes, and many small boxes, one per stroke, instead of one box around everything.

<table width="100%" bgcolor="#EDE7F6"><tr><td><font color="#311B92">📘 <b>Theory: what a SAM prompt
says.</b> SAM (Kirillov et al., ICCV 2023, <a href="https://arxiv.org/abs/2304.02643">arXiv:2304.02643</a>;
SAM 2, Ravi et al. 2024, <a href="https://arxiv.org/abs/2408.00714">arXiv:2408.00714</a>) turns a prompt
into <i>one object mask</i>. A positive click says "the object contains this pixel", a negative click
"it does not contain this one", and a box "the object fills this rectangle". Every prompt set describes
a single object, so forty clicks scattered over a vessel tree ask for one object that contains them all.
A prompt can also be split into many small ones, each its own object, and the masks combined: SAM then
answers many local questions instead of one global one. Nothing is trained; the image encoder runs once
per image and every extra prompt costs only a fast decoder pass.</font></td></tr></table>

This helper cell adds a colour for each direct approach and defines the prompts we take from the
strokes: the separate strokes of an image, a click at the centre of a stroke, SAM on many (box, click)
objects at once, the two direct recipes on a list of images, and a figure that draws boxes.

In [ ]:
# @title Path 3 helpers: colours, strokes, SAM on many prompts, the two recipes { display-mode: "form" }
from matplotlib.patches import Rectangle

APPROACH_COLOURS |= {
    "sam_clicks": "#546E7A",
    "sam_boxes": "#880E4F",
}
APPROACH_NAMES |= {
    "sam_clicks": "direct: SAM from stroke clicks",
    "sam_boxes": "direct: SAM, one box + click per stroke",
}


def stroke_masks(item: dict, value: int = 1) -> list[np.ndarray]:
    """The separate strokes of one kind (1 vessel, 2 background) as boolean masks: the 8-connected pieces
    of the scribble map."""
    lab, n = ndi.label(item["scribbles"] == value, structure=np.ones((3, 3)))
    return [lab == k for k in range(1, n + 1)]


def stroke_click(stroke: np.ndarray) -> list[float]:
    """(x, y) of the stroke pixel nearest the stroke's centre: a click that is always on the stroke."""
    ys, xs = np.nonzero(stroke)
    k = int(np.argmin((xs - xs.mean()) ** 2 + (ys - ys.mean()) ** 2))
    return [float(xs[k]), float(ys[k])]


@torch.no_grad()
def sam_many(image: np.ndarray, boxes: list[list[float]], clicks: list[list[float]] | None = None, chunk=32):
    """SAM with one object per box (plus one positive click inside it, if given), decoded in chunks after a
    single image encoding. Returns the highest probability over the objects: a pixel is foreground as soon
    as one object covers it."""
    best = np.zeros(image.shape[:2], np.float32)
    for s in range(0, len(boxes), chunk):
        kw = {"input_boxes": [boxes[s : s + chunk]]}
        if clicks is not None:
            c = clicks[s : s + chunk]
            kw |= {"input_points": [[[p] for p in c]], "input_labels": [[[1]] * len(c)]}
        inputs = sam.proc(images=image, return_tensors="pt", **kw)
        if sam._cache is None or sam._cache[0] is not image:  # the same image cache as Sam.probability
            sam._cache = (image, sam.model.get_image_embeddings(inputs["pixel_values"].to(sam.device)))
        out = sam.model(
            image_embeddings=sam._cache[1],
            input_points=inputs["input_points"].to(sam.device) if clicks is not None else None,
            input_labels=inputs["input_labels"].to(sam.device) if clicks is not None else None,
            input_boxes=inputs["input_boxes"].to(sam.device),
            multimask_output=False,
        )
        logits = sam.proc.post_process_masks(out.pred_masks.cpu(), inputs["original_sizes"], binarize=False)[
            0
        ]
        best = np.maximum(best, torch.sigmoid(logits[:, 0]).amax(0).numpy())
    return best


def direct_clicks(items: list[dict], n: int) -> tuple[list[np.ndarray], list[tuple]]:
    """Recipe 1: n vessel clicks and n background clicks spread along the strokes of each image, one SAM
    prompt per image. Returns the masks and each image's (vessel, background) clicks."""
    preds, clicks = [], []
    for it in items:
        pos, neg = stroke_points(it, 1, n), stroke_points(it, 2, n)
        preds.append(
            sam.probability(it["image"], points=pos + neg, labels=[1] * len(pos) + [0] * len(neg)) >= 0.5
        )
        clicks.append((pos, neg))
    return preds, clicks


def direct_boxes(items: list[dict], pad: int, with_click: bool) -> tuple[list[np.ndarray], list[tuple]]:
    """Recipe 2: one box around every vessel stroke (grown by `pad` px; `stroke_box` is set in 7.2), plus a
    click at the stroke's centre if `with_click`; the masks of all strokes are combined. Returns the masks
    and each image's (boxes, clicks)."""
    preds, prompts = [], []
    for it in items:
        strokes = stroke_masks(it, 1)
        boxes = [[float(v) for v in stroke_box(s, pad)] for s in strokes]
        clicks = [stroke_click(s) for s in strokes] if with_click else None
        preds.append(
            sam_many(it["image"], boxes, clicks) >= 0.5 if boxes else np.zeros(it["mask"].shape, bool)
        )
        prompts.append((boxes, clicks))
    return preds, prompts


def show_boxes(panels: list[tuple[str, np.ndarray, list | None]], width: float = 3.2):
    """One row of panels (title, RGB image, boxes or None); boxes are drawn as teal rectangles."""
    fig, axes = plt.subplots(1, len(panels), figsize=(width * len(panels), width))
    for ax, (title, img, boxes) in zip(np.atleast_1d(axes), panels, strict=True):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
        for x0, y0, x1, y1 in boxes or []:
            ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, lw=1, color=THEME["scribble"]))
    plt.tight_layout()
    plt.show()

### 7.1 Clicks on the vessels, and "not here"

The first recipe is section 6.1 without the head: twenty clicks spread along the vessel strokes and
twenty along the background strokes of each <i>held-out</i> image, and SAM's mask is the answer. The
strokes on these images are prompts a person would draw at use time, not training labels; the ground
truth only scores the result (L-005). The heads of sections 4 to 6 never saw a stroke on these images,
so keep in mind that Path 3 gets more help per image.

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your prediction.</b> In section
2, twenty vessel clicks alone gave the IoU printed below as "section 2". Adding twenty "not here" clicks
on the background: will SAM's mask shrink onto the vessels, stay a region, or do something else?
Write one guess.</font></td></tr></table>

This cell prompts SAM with the stroke clicks on every evaluation image (one SAM image encoding per image:
seconds on a GPU, about a second per image on a CPU), scores the masks against the hidden truth and shows
the demo image.

In [ ]:
N_DIRECT_CLICKS = 20  # clicks per kind (vessel / background) and image, taken from the strokes
t0 = time.monotonic()
path3_preds = {"zero_shot": ladder_preds["zero_shot"], "scribbles": ladder_preds["scribbles"]}
path3_preds["sam_clicks"], click_prompts = direct_clicks(SPEC_EVAL, N_DIRECT_CLICKS)
clicks_sec = round(time.monotonic() - t0, 1)
path3_report = {k: rung_report(v) for k, v in path3_preds.items()}
print(
    f"SAM from {N_DIRECT_CLICKS} + {N_DIRECT_CLICKS} stroke clicks on {len(SPEC_EVAL)} images · "
    f"{clicks_sec} s on {DEVICE}"
)
print_path2(path3_report)
print(f"{'section 2: SAM, 20 vessel clicks only':<46}{RESULTS['sam_prompt_ladder']['20 points']:>6.3f}")
pos, neg = click_prompts[k_demo]
show_clicks(
    [
        (f"{demo['name']} · vessel clicks", demo["image"], pos),
        ("background clicks", demo["image"], neg),
        ("SAM from both vs truth", error_map(demo["image"], path3_preds["sam_clicks"][k_demo], demo), None),
    ],
    width=3.0,
)

In the last panel, violet = vessel found, orange = SAM marked background as vessel, green = a vessel
SAM missed.

**Q27.** 🟡 Did the "not here" clicks pull SAM's mask onto the vessels? Change the number of clicks per
kind and run the next cell: does SAM do better with fewer clicks or with more? Why would that be, given
that every prompt set describes one object?

```python
N_CLICKS_TRY = 5   # try 1, 5, 50
```

This cell runs the same recipe with your number of clicks on every evaluation image and prints it under
the default of 20.

In [ ]:
# Q27 — fewer or more stroke clicks for SAM
N_CLICKS_TRY = 5  # try 1, 5, 50
if not isinstance(N_CLICKS_TRY, int) or N_CLICKS_TRY < 1:
    raise SystemExit("N_CLICKS_TRY must be a whole number of at least 1.")
try_preds, _ = direct_clicks(SPEC_EVAL, N_CLICKS_TRY)
RESULTS["sam_direct_clicks_try"] = rung_report(try_preds) | {"n_clicks": N_CLICKS_TRY}
print_path2(
    {
        f"{N_DIRECT_CLICKS} + {N_DIRECT_CLICKS} clicks (default)": path3_report["sam_clicks"],
        f"{N_CLICKS_TRY} + {N_CLICKS_TRY} clicks (yours)": RESULTS["sam_direct_clicks_try"],
    }
)

<details><summary>What to look for</summary>

SAM looks for one object that contains every positive click and none of the negative ones. A vessel
tree is one connected structure, but it is a few pixels wide and spread over the whole crop, so the
simplest object that contains clicks all over the tree is a large region around it, and the negative
clicks, which also sit all over the crop, mostly cut holes in that region rather than turning it into a
line. A few clicks can leave SAM a more local object to settle on; many clicks push it towards "the
whole fundus minus a few holes". The marked column tells you which of the two happened on your run. More
prompts in <i>one</i> set are not more information to SAM if they describe something it does not see as
one object.

</details>

### 7.2 One small box per stroke

The second recipe splits the question. Each vessel stroke gets its own box, a few pixels larger than the
stroke, and a click at its centre; SAM answers each box as a separate object, and the vessel map is the
union of all the answers. One stroke is a short piece of one vessel, so each box asks a local question
SAM can answer: "the thing along this line, inside this rectangle".

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your turn (optional, 2 min).</b>
Fill <code>my_stroke_box</code>. In: one stroke as a boolean image (<code>True</code> on the stroke) and a
margin <code>pad</code> in pixels. Out: the box <code>[x0, y0, x1, y1]</code> around the stroke's pixels,
grown by <code>pad</code> on every side and clipped to the image (x from 0 to width − 1, y from 0 to
height − 1). Remember that <code>np.nonzero</code> returns rows (y) before columns (x). The cell checks
your version on a 6 × 8 play stroke, without and with a margin, and continues with ours if you skip it.
</font></td></tr></table>

In [ ]:
# ✍️ YOUR CODE
def my_stroke_box(stroke: np.ndarray, pad: int) -> list[float]:
    """The box [x0, y0, x1, y1] around the True pixels of `stroke`, grown by `pad` pixels on every side and
    clipped to the image: 0 <= x <= width - 1, 0 <= y <= height - 1."""
    ...


def _ref_stroke_box(stroke: np.ndarray, pad: int) -> list[float]:
    """Reference: min/max of the stroke's columns and rows, padded, then clipped."""
    ys, xs = np.nonzero(stroke)
    h, w = stroke.shape
    return [
        float(max(xs.min() - pad, 0)),
        float(max(ys.min() - pad, 0)),
        float(min(xs.max() + pad, w - 1)),
        float(min(ys.max() + pad, h - 1)),
    ]


_play_stroke = np.zeros((6, 8), bool)
_play_stroke[1, 1:4] = True  # a short diagonal stroke near the top-left corner
_play_stroke[2, 4] = _play_stroke[3, 5] = True
stroke_box = use(
    my_stroke_box,
    reference=_ref_stroke_box,
    check=lambda f: [[float(v) for v in f(_play_stroke, p)] for p in (0, 2)],
    name="stroke_box",
)
print(
    "play stroke · box with pad 0:", stroke_box(_play_stroke, 0), "· with pad 2:", stroke_box(_play_stroke, 2)
)

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Your prediction.</b> The strokes
cover only some of the vessels (section 4 dropped 70 % of the segments), and each box sits on a stroke.
Will this recipe mark more of the fundus than the clicks of 7.1, or less? Will it beat simply thickening
the strokes themselves by a few pixels, with no model at all? Write two guesses.</font></td></tr></table>

This cell draws a box around every vessel stroke of every evaluation image (4 px of margin on the 1024 px
crop), asks SAM for one object per box with a click at the stroke's centre (all boxes of an image in a
few decoder passes), and prints the result next to the strokes thickened by 3 px, the "no model"
reference.

In [ ]:
BOX_PAD = 1 if SMOKE else 4  # px around each stroke (smoke thumbnails are 4× smaller)
THICKEN = 1 if SMOKE else 3  # px: the "no model" reference, the vessel strokes thickened
t0 = time.monotonic()
path3_preds["sam_boxes"], box_prompts = direct_boxes(SPEC_EVAL, BOX_PAD, with_click=True)
boxes_sec = round(time.monotonic() - t0, 1)
path3_report["sam_boxes"] = rung_report(path3_preds["sam_boxes"])
strokes_alone = [ndi.binary_dilation(it["scribbles"] == 1, iterations=THICKEN) for it in SPEC_EVAL]
strokes_report = rung_report(strokes_alone)
n_boxes = [len(b) for b, _ in box_prompts]
print(
    f"SAM, one box + click per vessel stroke · {int(np.mean(n_boxes))} boxes per image on average · "
    f"{boxes_sec} s on {DEVICE}"
)
print_path2(path3_report | {f"no model: the strokes thickened by {THICKEN} px": strokes_report})
boxes, _ = box_prompts[k_demo]
show_boxes(
    [
        (f"{demo['name']} · {len(boxes)} stroke boxes", demo["image"], boxes),
        (
            "SAM, one object per box vs truth",
            error_map(demo["image"], path3_preds["sam_boxes"][k_demo], demo),
            None,
        ),
        ("strokes thickened vs truth", error_map(demo["image"], strokes_alone[k_demo], demo), None),
    ],
    width=3.0,
)

**Q28.** 🟡 Compare the box recipe with the clicks of 7.1 and with the thickened strokes: which column
moved, recall or precision? Then grow the boxes and drop the centre click in the next cell. What does a
larger box let SAM find, and what does it cost?

```python
BOX_PAD_TRY = 16      # try 0, 16, 32 (px of the 1024 px crop)
WITH_CLICK_TRY = True  # try False: the box alone
```

This cell runs the box recipe with your margin and click setting on every evaluation image and prints it
under the default.

In [ ]:
# Q28 — larger or smaller boxes, with or without the centre click
BOX_PAD_TRY = 16  # try 0, 16, 32 (px of the 1024 px crop)
WITH_CLICK_TRY = True  # try False: the box alone
if not isinstance(BOX_PAD_TRY, int) or BOX_PAD_TRY < 0 or not isinstance(WITH_CLICK_TRY, bool):
    raise SystemExit("BOX_PAD_TRY must be a whole number ≥ 0 and WITH_CLICK_TRY True or False.")
pad_try = max(BOX_PAD_TRY // 4, 0) if SMOKE else BOX_PAD_TRY  # smoke thumbnails are 4× smaller
try_preds, _ = direct_boxes(SPEC_EVAL, pad_try, WITH_CLICK_TRY)
RESULTS["sam_direct_boxes_try"] = rung_report(try_preds) | {"pad": BOX_PAD_TRY, "with_click": WITH_CLICK_TRY}
print_path2(
    {
        f"pad {BOX_PAD} px, with click (default)": path3_report["sam_boxes"],
        f"pad {BOX_PAD_TRY} px, {'with' if WITH_CLICK_TRY else 'no'} click (yours)": RESULTS[
            "sam_direct_boxes_try"
        ],
    }
)

<details><summary>What to look for</summary>

Each box can only find what lies inside it, so recall is capped by how much of the tree the strokes
touch: about the vessels a person stroked, plus whatever branches cross their boxes. That is why the
box recipe marks a small share of the fundus, unlike the clicks of 7.1. A larger box reaches more
branches (recall can rise) but also more background that SAM may take for the object (precision
falls), and at some size it turns back into section 2's big box. The centre click tells SAM which thing
inside the box is meant, which tends to help most when the box holds more than one candidate. If the
thickened strokes come close to or beat SAM here, that is a real finding, not a bug: on thin structures
a cheap baseline built from the hints alone is the first thing to compare against.

</details>

### 7.3 Path 3, side by side

This cell puts the two direct recipes next to the section-2 box, the scribble head of section 4 and the
SAM pseudo-label head of section 6, with the "no model" and "everything" references, then a bar chart and
the error maps on the demo image. It also writes the numbers section 8 compares: no training time, but
annotation on <i>every</i> image you segment, counted as 2 s per stroke (a click or a box is taken from
a stroke, so the strokes are the cost).

In [ ]:
path3_preds["sam_pseudo"] = path2_preds["sam_pseudo"]
path3_report["sam_pseudo"] = path2_report["sam_pseudo"]
eval_meta = SPECIALISED["eval"]
strokes_per_image = {
    "sam_clicks": float(np.mean([m.get("n_fg_strokes", 0) + m.get("n_bg_strokes", 0) for m in eval_meta])),
    "sam_boxes": float(np.mean([m.get("n_fg_strokes", 0) for m in eval_meta])),  # vessel strokes only
}
effort = {k: effort_minutes(n_strokes=round(v * len(eval_meta))) for k, v in strokes_per_image.items()}
sec = {"sam_clicks": clicks_sec, "sam_boxes": boxes_sec}
result_key = {"sam_clicks": "sam_direct_clicks", "sam_boxes": "sam_direct_boxes"}
for k in ("sam_clicks", "sam_boxes"):
    per_image = [score(p, it)["iou"] for p, it in zip(path3_preds[k], SPEC_EVAL, strict=True)]
    RESULTS[result_key[k]] = path3_report[k] | {
        "iou_std": float(np.std(per_image)),
        "train_sec": 0.0,
        "infer_sec": sec[k],
        "n_images": len(SPEC_EVAL),
        "strokes_per_image": strokes_per_image[k],
        "effort_min_per_image": round(effort_minutes(n_strokes=1) * strokes_per_image[k], 2),
        "effort_min": round(effort[k], 2),
    }
RESULTS["sam_direct_boxes"] |= {"pad": BOX_PAD, "with_click": True}
RESULTS["sam_direct_clicks"] |= {"n_clicks": N_DIRECT_CLICKS}
RESULTS["sam_direct_iou"] = path3_report["sam_boxes"]["iou"]  # headline: points + boxes (SPEC §6, 07)
RESULTS["sam_direct_clicks_iou"] = path3_report["sam_clicks"]["iou"]
RESULTS["sam_direct_strokes_alone"] = strokes_report | {"thicken_px": THICKEN}
RESULTS["path3_effort_min"] = {k: round(v, 2) for k, v in effort.items()}

order = ["zero_shot", "scribbles", "sam_pseudo", "sam_clicks", "sam_boxes"]
print_path2({k: path3_report[k] for k in order})
print(
    f"{'no model: strokes thickened':<46}{strokes_report['iou']:>6.3f}\n"
    f"{'everything is foreground':<46}{RESULTS['specialised_everything_iou']:>6.3f}"
)
print(
    f"\nannotation · the heads: {n_strokes} strokes on {len(scrib_train)} training images, once "
    f"(≈ {effort_minutes(n_strokes=n_strokes):.1f} min)"
)
for k in ("sam_clicks", "sam_boxes"):
    print(
        f"annotation · {APPROACH_NAMES[k]}: ≈ {RESULTS[result_key[k]]['effort_min_per_image']:.1f} min on "
        f"every image you segment ({effort[k]:.0f} min for these {len(SPEC_EVAL)})"
    )
metrics = ["iou", "cldice", "recall", "precision"]
x, width = np.arange(len(metrics)), 0.8 / len(order)
fig, ax = plt.subplots(figsize=(7.2, 3.2))
for n, k in enumerate(order):
    shift = (n - (len(order) - 1) / 2) * width
    ax.bar(
        x + shift,
        [path3_report[k][m] for m in metrics],
        width,
        color=APPROACH_COLOURS[k],
        label=APPROACH_NAMES[k],
    )
ax.axhline(
    RESULTS["specialised_everything_iou"], color=THEME["background"], ls="--", label='IoU of "everything"'
)
ax.set_xticks(x, ["IoU", "clDice", "recall", "precision"])
ax.set_ylim(0, 1)
ax.set_title("Path 3: SAM prompted directly, next to the trained heads")
ax.legend(fontsize=7, frameon=False, loc="upper left")
plt.tight_layout()
plt.show()
show_masks(
    [
        (
            f"{APPROACH_NAMES[k].split(':')[-1].strip()} · "
            f"IoU {score(path3_preds[k][k_demo], demo)['iou']:.2f}",
            error_map(demo["image"], path3_preds[k][k_demo], demo),
        )
        for k in order
    ]
    + [("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"]))],
    width=2.6,
)

**Q29.** 🟡 Path 3 trains nothing, but its annotation never stops. Suppose you must segment 1,000 new
fundus images next month. Using the effort line above and the table, when would you prompt SAM directly,
and when would you rather train a head once? Is there a way to combine the two?

<details><summary>A possible answer</summary>

The heads pay their strokes once, on a few training images, and then segment any number of new images
for free; Path 3 pays per image, so at 1,000 images it costs a thousand times the per-image line, hours
or days of clicking. Direct prompting wins when there are few images, when each one matters (a case to
review, an image nobody will see again), or when the task changes too often to retrain. A head wins when
the images keep coming. The two combine naturally: prompt SAM on a handful of images, correct its masks
by hand, and train the head on those (Path 2, with a person in the loop), or let the head propose and a
person fix only the worst images. Whatever you pick, compare it with the cheapest baseline built from the
same hints, such as the thickened strokes here.

</details>

<table width="100%" bgcolor="#FDECEA"><tr><td><font color="#8B1A10">⚠️ <b>Pitfall: over-trusting SAM's
confident masks.</b> A mask from your own clicks feels like your own work: it has crisp edges and sits
where you pointed. SAM still decides what "the object" is, and on structures it was not trained on that
decision can be a region where you wanted a line, or a piece where you wanted the whole. Score a few
directly prompted images against a careful manual mask before you trust the recipe on the rest.
</font></td></tr></table>

### 7.4 🔴 Optional: SAM 3 from a word

SAM 3 (Meta, 2025) adds <i>concept</i> prompts: type a short noun phrase and it segments every instance
of it, with no click at all. It is gated on Hugging Face (Meta approves access per account), so the
workshop never requires it: <code>setup_check.ipynb</code> caches it only if your account has access. If
it is not in your cache, the next cell says so and stops; nothing later depends on it.

<table width="100%" bgcolor="#F3E5F5"><tr><td><font color="#4A148C">❓ <b>Question (optional).</b> Run the
next cell if you have SAM 3, then try other words.</font></td></tr></table>

**Q30.** 🔴 Does a word find the vessels better than your strokes did? Try a few phrasings. Which words
work, and what does that tell you about the images SAM 3 learnt its concepts from?

```python
SAM3_TEXT = "blood vessel"   # try "vessel", "artery", "dark line", "retina"
```

This cell loads SAM 3 from your cache only (never from the internet), in fp16 on a GPU, runs it with
your phrase on every evaluation image and keeps the union of the instances it finds.

In [ ]:
# Q30 — 🔴 SAM 3 from a text prompt (optional, only if SAM 3 is in your cache)
SAM3_TEXT = "blood vessel"  # try "vessel", "artery", "dark line", "retina"
SAM3_ID, SAM3_LICENCE = "facebook/sam3", "SAM License (Meta)"
if not isinstance(SAM3_TEXT, str) or not SAM3_TEXT.strip():
    raise SystemExit('SAM3_TEXT must be a short phrase in quotes, e.g. "blood vessel".')
RESULTS["sam3_text_iou"] = None
sam3 = None
try:
    from transformers import Sam3Model, Sam3Processor

    sam3_dtype = torch.float16 if DEVICE == "cuda" else torch.float32  # fp16 on a T4, never bf16 (SPEC §5)
    sam3_proc = Sam3Processor.from_pretrained(SAM3_ID, local_files_only=True)
    sam3 = Sam3Model.from_pretrained(SAM3_ID, local_files_only=True, dtype=sam3_dtype).eval().to(DEVICE)
    print(f"SAM 3 · source {SAM3_ID} · licence {SAM3_LICENCE} · {str(sam3_dtype).split('.')[-1]} on {DEVICE}")
except Exception as err:  # noqa: BLE001 - not cached, no access, an older transformers: all mean "skip"
    RESULTS["sam3_note"] = f"SAM 3 not available in this session ({type(err).__name__}); it is optional"
    print(
        "ℹ️ SAM 3 is not in your cache, so this optional cell stops here. To try it later: request access on "
        f"huggingface.co/{SAM3_ID}, then run setup_check.ipynb with your HF token. Nothing else needs it."
    )
if sam3 is not None:
    try:
        t0, sam3_preds = time.monotonic(), []
        for it in SPEC_EVAL:
            inputs = sam3_proc(images=it["image"], text=SAM3_TEXT.strip(), return_tensors="pt").to(DEVICE)
            inputs["pixel_values"] = inputs["pixel_values"].to(sam3_dtype)
            with torch.no_grad():
                out = sam3(**inputs)
            found = sam3_proc.post_process_instance_segmentation(
                out, threshold=0.3, mask_threshold=0.5, target_sizes=[it["mask"].shape]
            )[0]
            masks = found["masks"].bool().cpu().numpy()
            sam3_preds.append(masks.any(0) if len(masks) else np.zeros(it["mask"].shape, bool))
        sam3_report = rung_report(sam3_preds)
        RESULTS["sam3_text_iou"] = sam3_report["iou"]
        RESULTS["sam3_text"] = sam3_report | {"text": SAM3_TEXT, "infer_sec": round(time.monotonic() - t0, 1)}
        print_path2(
            {
                f'SAM 3, "{SAM3_TEXT}"': sam3_report,
                "SAM 2.1, one box + click per stroke": path3_report["sam_boxes"],
            }
        )
        show_masks(
            [
                (f'SAM 3, "{SAM3_TEXT}" vs truth', error_map(demo["image"], sam3_preds[k_demo], demo)),
                (
                    "SAM 2.1 from stroke boxes vs truth",
                    error_map(demo["image"], path3_preds["sam_boxes"][k_demo], demo),
                ),
            ],
            width=3.2,
        )
    except Exception as err:  # noqa: BLE001 - an optional extension must never stop the notebook
        RESULTS["sam3_note"] = f"SAM 3 loaded but failed: {type(err).__name__}: {err}"
        print(f"⚠️ SAM 3 loaded but did not run here ({type(err).__name__}: {err}). It is optional; carry on.")

<details><summary>What to look for</summary>

A concept prompt only works for concepts SAM 3 met in training, named the way its captions named them.
"Blood vessel" in a fundus photograph is a rare, specialist phrase next to "dog" or "car", so expect it
to find thick vessels, a blob, or nothing, and to change a lot with the wording. Generic words like
"dark line" sometimes do better than the correct medical term, which is a hint that the model matches
appearance more than meaning. Whatever you see, a word costs no annotation at all, so even a weak result
is a free starting point for the strokes of 7.2.

</details>

<table width="100%" bgcolor="#FFE8D6"><tr><td><font color="#7A2E00">✏️ <b>Try it on your image (optional,
2 min).</b> Direct prompting needs no training, so it is the quickest thing to try on a photo of your
own. Upload one with the folder icon on the left, put its path in <code>MY_IMAGE</code>, and add
<code>(x, y)</code> clicks on the thing (<code>MY_FG_CLICKS</code>), clicks on what you do not want
(<code>MY_BG_CLICKS</code>) and/or small boxes <code>(x0, y0, x1, y1)</code>, one per piece of the thing
(<code>MY_BOXES</code>). Left empty, it runs on the demo image with prompts from its strokes. Section 9
lets you draw the strokes yourself.</font></td></tr></table>

This cell asks SAM for one mask from all your clicks together and one mask from your boxes, each box its
own object, and shows both: one SAM image encoding, no training.

In [ ]:
MY_IMAGE = ""  # e.g. "/content/my_photo.jpg"; empty = the demo image
MY_FG_CLICKS = []  # e.g. [(120, 80), (300, 210)] on the thing you want
MY_BG_CLICKS = []  # e.g. [(20, 20)] on what you do not want
MY_BOXES = []  # e.g. [(100, 60, 160, 120)], one small box per piece of the thing
lists = (MY_FG_CLICKS, MY_BG_CLICKS, MY_BOXES)
if not isinstance(MY_IMAGE, str) or not all(isinstance(v, list) for v in lists):
    raise SystemExit('MY_IMAGE must be a path in quotes (or "") and the clicks and boxes lists.')
if MY_IMAGE and not Path(MY_IMAGE).is_file():
    raise SystemExit(f"{MY_IMAGE} not found. Upload it with the folder icon on the left and copy its path.")
if MY_IMAGE and not (MY_FG_CLICKS or MY_BOXES):
    raise SystemExit("Add at least one (x, y) click to MY_FG_CLICKS or one (x0, y0, x1, y1) box to MY_BOXES.")
my_img = np.array(Image.open(MY_IMAGE).convert("RGB")) if MY_IMAGE else demo["image"]
my_h, my_w = my_img.shape[:2]
if MY_IMAGE:
    my_fg = [[float(x), float(y)] for x, y in MY_FG_CLICKS]
    my_bg = [[float(x), float(y)] for x, y in MY_BG_CLICKS]
    my_boxes = [[float(v) for v in b] for b in MY_BOXES]
else:
    my_fg, my_bg = stroke_points(demo, 1, 5), stroke_points(demo, 2, 5)
    my_boxes = box_prompts[k_demo][0]
if not all(0 <= x < my_w and 0 <= y < my_h for x, y in my_fg + my_bg):
    raise SystemExit(f"every click must lie inside the image: 0 ≤ x < {my_w}, 0 ≤ y < {my_h}.")
if not all(len(b) == 4 and 0 <= b[0] < b[2] < my_w and 0 <= b[1] < b[3] < my_h for b in my_boxes):
    raise SystemExit(
        f"every box must be (x0, y0, x1, y1) with 0 ≤ x0 < x1 < {my_w} and 0 ≤ y0 < y1 < {my_h}."
    )
panels = []
if my_fg:
    clicks_mask = (
        sam.probability(my_img, points=my_fg + my_bg, labels=[1] * len(my_fg) + [0] * len(my_bg)) >= 0.5
    )
    panels.append(
        (f"clicks: {len(my_fg)} on, {len(my_bg)} off", overlay(my_img, clicks_mask, THEME["fails"]), None)
    )
if my_boxes:
    boxes_mask = sam_many(my_img, my_boxes) >= 0.5
    panels.append(
        (f"{len(my_boxes)} boxes, one object each", overlay(my_img, boxes_mask, THEME["fails"]), my_boxes)
    )
show_boxes([(MY_IMAGE or f"demo · {demo['name']}", my_img, None)] + panels, width=3.0)

<table width="100%" bgcolor="#E8EAF6"><tr><td><font color="#1A237E">🔀 <b>Across approaches.</b> Path 1
changed the loss, Path 2 filled in the labels with a second model, Path 3 skipped training and asked
the foundation model on every image. The first two pay annotation once and reuse a head; the third pays
on every image and needs no training at all. On thin structures, how you split the prompt (one global
object or many local ones) mattered as much as how many hints you gave, and a model-free baseline built
from the same hints is the fair yardstick. Section 8 puts every approach of the notebook in one table:
accuracy, annotation, reliability and compute side by side.</font></td></tr></table>